## Cek dataset

In [ ]:

# ============================================================
# CIC-IDS2017
# CHECK CLASS COMPOSITION BY DAY
# Google Drive Connection + CSV Loading + Class Distribution
# ============================================================


# ============================================================
# 1. CONNECT GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount('/content/drive')


# ============================================================
# 2. IMPORTS
# ============================================================

import os
import glob
import gc
import pandas as pd
import numpy as np


# ============================================================
# 3. CONFIGURATION
# ============================================================

DATA_DIR = (
    "/content/drive/MyDrive/"
    "My Research/USULAN PENELITIAN DIKTI/2026/"
    "IDS-KN/luaran/MachineLearningCSV"
)

OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "My Research/USULAN PENELITIAN DIKTI/2026/"
    "IDS-KN/luaran/outputs"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

EXPECTED_DAYS = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday"
]


# ============================================================
# 4. FIND CSV FILES
# ============================================================

csv_files = sorted(
    glob.glob(os.path.join(DATA_DIR, "*.csv"))
)

print("=" * 70)
print("CIC-IDS2017: CLASS COMPOSITION BY DAY")
print("=" * 70)

print("Data directory :", DATA_DIR)
print("CSV files found:", len(csv_files))

if len(csv_files) == 0:
    raise FileNotFoundError(
        "No CSV files found. Please check DATA_DIR."
    )

for file in csv_files:
    print(" -", os.path.basename(file))


# ============================================================
# 5. LOAD LABELS AND IDENTIFY SOURCE DAY
# ============================================================

frames = []

for file in csv_files:

    filename = os.path.basename(file)
    filename_lower = filename.lower()

    # Identify day from the original filename
    day = None

    for candidate_day in EXPECTED_DAYS:
        if candidate_day.lower() in filename_lower:
            day = candidate_day
            break

    if day is None:
        raise ValueError(
            "Could not identify the day from filename: "
            + filename
        )

    print("\nReading:", filename)
    print("Detected day:", day)

    # Read header first to handle possible spaces in column names
    header = pd.read_csv(file, nrows=0)

    label_columns = [
        col for col in header.columns
        if str(col).strip().lower() == "label"
    ]

    if len(label_columns) != 1:
        raise ValueError(
            "Expected exactly one Label column in: "
            + filename
        )

    actual_label_column = label_columns[0]

    # Read only the Label column to reduce memory usage
    temp = pd.read_csv(
        file,
        usecols=[actual_label_column],
        low_memory=False
    )

    temp.columns = ["Label"]

    # Normalize label strings
    temp["Label"] = (
        temp["Label"]
        .astype("string")
        .str.strip()
    )

    # Remove missing/empty labels
    temp = temp.dropna(subset=["Label"])
    temp = temp[temp["Label"] != ""]

    # Preserve the original source file and day
    temp["Day"] = day
    temp["SourceFile"] = filename

    print("Valid samples:", f"{len(temp):,}")
    print("Classes found:", temp["Label"].nunique())

    frames.append(temp)

    del temp, header
    gc.collect()


# ============================================================
# 6. COMBINE LABEL METADATA
# ============================================================

labels_df = pd.concat(
    frames,
    ignore_index=True
)

del frames
gc.collect()

print("\n" + "=" * 70)
print("DATA SUMMARY")
print("=" * 70)

print("Total valid samples:", f"{len(labels_df):,}")
print("Total classes       :", labels_df["Label"].nunique())
print("Total source files  :", labels_df["SourceFile"].nunique())


# ============================================================
# 7. CLASS COUNTS PER SOURCE FILE
# ============================================================

counts_by_file = pd.crosstab(
    labels_df["Label"],
    labels_df["SourceFile"],
    margins=False
)

# Include total samples for each source file
file_totals = counts_by_file.sum(axis=0)

print("\n" + "=" * 70)
print("CLASS COUNTS PER SOURCE FILE")
print("=" * 70)

print(counts_by_file.to_string())

print("\nTotal samples per source file:")
print(file_totals.to_string())

counts_by_file.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "CICIDS2017_class_counts_by_source_file.csv"
    )
)


# ============================================================
# 8. CLASS COUNTS PER DAY
# ============================================================

counts_by_day = pd.crosstab(
    labels_df["Label"],
    labels_df["Day"]
)

# Keep days in Monday-Friday order
available_days = [
    day for day in EXPECTED_DAYS
    if day in counts_by_day.columns
]

counts_by_day = counts_by_day.reindex(
    columns=available_days,
    fill_value=0
)

# Add row/column totals
counts_by_day_with_total = counts_by_day.copy()
counts_by_day_with_total["TOTAL"] = (
    counts_by_day_with_total.sum(axis=1)
)

counts_by_day_with_total.loc["TOTAL"] = (
    counts_by_day_with_total.sum(axis=0)
)

print("\n" + "=" * 70)
print("CLASS COUNTS PER DAY")
print("=" * 70)

print(counts_by_day_with_total.to_string())

counts_by_day_with_total.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "CICIDS2017_class_counts_by_day.csv"
    )
)


# ============================================================
# 9. CLASS PERCENTAGES WITHIN EACH DAY
# ============================================================

daily_totals = counts_by_day.sum(axis=0)

percent_by_day = (
    counts_by_day
    .div(daily_totals.replace(0, np.nan), axis=1)
    * 100
)

print("\n" + "=" * 70)
print("CLASS PERCENTAGES WITHIN EACH DAY (%)")
print("=" * 70)

print(percent_by_day.round(4).to_string())

percent_by_day.round(6).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "CICIDS2017_class_percentages_by_day.csv"
    )
)


# ============================================================
# 10. DAY-LEVEL SUMMARY
# ============================================================

summary_rows = []

for day in EXPECTED_DAYS:

    day_data = labels_df[labels_df["Day"] == day]

    if day_data.empty:
        summary_rows.append({
            "Day": day,
            "Source_Files": 0,
            "Total_Samples": 0,
            "Classes_Present": 0,
            "Classes_Missing": None,
            "Missing_Class_Names": "No data"
        })
        continue

    present_classes = sorted(
        day_data["Label"].unique().tolist()
    )

    all_classes = sorted(
        labels_df["Label"].unique().tolist()
    )

    missing_classes = sorted(
        set(all_classes) - set(present_classes)
    )

    summary_rows.append({
        "Day": day,
        "Source_Files": day_data["SourceFile"].nunique(),
        "Total_Samples": len(day_data),
        "Classes_Present": len(present_classes),
        "Classes_Missing": len(missing_classes),
        "Missing_Class_Names": "; ".join(missing_classes)
    })

day_summary = pd.DataFrame(summary_rows)

print("\n" + "=" * 70)
print("DAY-LEVEL SUMMARY")
print("=" * 70)

print(day_summary.to_string(index=False))

day_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "CICIDS2017_day_summary.csv"
    ),
    index=False
)


# ============================================================
# 11. VERIFY CLASS COVERAGE
# ============================================================

all_classes = sorted(
    labels_df["Label"].unique().tolist()
)

print("\n" + "=" * 70)
print("CLASS COVERAGE CHECK")
print("=" * 70)

print("Classes found in the complete dataset:")
for i, label in enumerate(all_classes, start=1):
    print(f"{i:02d}. {label}")

print("\nNumber of classes in the complete dataset:",
      len(all_classes))

print("\nClasses present in ALL observed days:")

if len(available_days) > 0:
    classes_in_all_days = [
        label for label in all_classes
        if all(counts_by_day.loc[label, day] > 0
               for day in available_days)
    ]

    print("Count:", len(classes_in_all_days))

    for label in classes_in_all_days:
        print(" -", label)

    print("\nClasses NOT present in every observed day:")

    classes_not_in_all_days = [
        label for label in all_classes
        if label not in classes_in_all_days
    ]

    for label in classes_not_in_all_days:
        print(" -", label)


# ============================================================
# 12. SAVE FINAL METADATA (OPTIONAL)
# ============================================================

labels_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "CICIDS2017_label_metadata_by_day.csv"
    ),
    index=False
)

print("\n" + "=" * 70)
print("ANALYSIS COMPLETED")
print("=" * 70)

print("Output directory:")
print(OUTPUT_DIR)

print("\nGenerated files:")
print("1. CICIDS2017_class_counts_by_source_file.csv")
print("2. CICIDS2017_class_counts_by_day.csv")
print("3. CICIDS2017_class_percentages_by_day.csv")
print("4. CICIDS2017_day_summary.csv")
print("5. CICIDS2017_label_metadata_by_day.csv")


## part 1

In [ ]:
# ============================================================
# PART 1
# CIC-IDS2017
# DATA LOADING + PREPROCESSING + 80/10/10 SPLIT
# + MUTUAL INFORMATION + TOP-64 FEATURES
# + ARTIFACT SAVING
# ============================================================

# ============================================================
# 0. GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount('/content/drive')


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import glob
import gc
import random
import warnings
import pickle

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder


warnings.filterwarnings("ignore")


# ============================================================
# 2. CONFIGURATION
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)


DATA_DIR = (
    "/content/drive/MyDrive/"
    "My Research/USULAN PENELITIAN DIKTI/2026/"
    "IDS-KN/luaran/MachineLearningCSV"
)


OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "My Research/USULAN PENELITIAN DIKTI/2026/"
    "IDS-KN/luaran/outputs"
)


os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# 3. EXPERIMENT CONFIGURATION
# ============================================================

TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
TEST_RATIO  = 0.10

MI_SAMPLE_SIZE = 100_000

N_SELECTED_FEATURES = 64

IMAGE_SIZE = 8


assert (
    TRAIN_RATIO +
    VAL_RATIO +
    TEST_RATIO
) == 1.0


# ============================================================
# 4. OUTPUT FILES
# ============================================================

TRAIN_PATH = os.path.join(
    OUTPUT_DIR,
    "train_final.pkl"
)

VAL_PATH = os.path.join(
    OUTPUT_DIR,
    "val_final.pkl"
)

TEST_PATH = os.path.join(
    OUTPUT_DIR,
    "test_final.pkl"
)

FEATURE_INFO_PATH = os.path.join(
    OUTPUT_DIR,
    "feature_information.pkl"
)

MI_RANKING_PATH = os.path.join(
    OUTPUT_DIR,
    "MI_feature_ranking.csv"
)

SELECTED_FEATURES_PATH = os.path.join(
    OUTPUT_DIR,
    "selected_64_features.csv"
)

LABEL_ENCODER_PATH = os.path.join(
    OUTPUT_DIR,
    "label_encoder.pkl"
)

IMAGE_MINMAX_PATH = os.path.join(
    OUTPUT_DIR,
    "image_normalization.pkl"
)


# ============================================================
# 5. CHECK EXISTING ARTIFACTS
# ============================================================

required_files = [
    TRAIN_PATH,
    VAL_PATH,
    TEST_PATH,
    FEATURE_INFO_PATH,
    MI_RANKING_PATH,
    SELECTED_FEATURES_PATH,
    LABEL_ENCODER_PATH,
    IMAGE_MINMAX_PATH
]


artifacts_exist = all(
    os.path.exists(path)
    for path in required_files
)


if artifacts_exist:

    print("============================================")
    print("PREPROCESSING ARTIFACTS FOUND")
    print("============================================")

    print(
        "Saved preprocessing results already exist."
    )

    print(
        "Part 1 can be skipped if they are unchanged."
    )

else:

    print("============================================")
    print("RUNNING PART 1")
    print("============================================")


    # ========================================================
    # 6. FIND CSV FILES
    # ========================================================

    csv_files = sorted(
        glob.glob(
            os.path.join(
                DATA_DIR,
                "*.csv"
            )
        )
    )


    print("\n============================================")
    print("CIC-IDS2017 CSV FILES")
    print("============================================")

    print(
        "CSV files found:",
        len(csv_files)
    )


    for file in csv_files:

        print(
            " -",
            os.path.basename(file)
        )


    if len(csv_files) == 0:

        raise FileNotFoundError(
            "No CSV files found in:\n" +
            DATA_DIR
        )


    # ========================================================
    # 7. LOAD ALL CSV FILES
    # ========================================================

    frames = []


    for file in csv_files:

        print(
            "\nLoading:",
            os.path.basename(file)
        )


        temp = pd.read_csv(
            file,
            low_memory=False
        )


        temp.columns = (
            temp.columns
            .astype(str)
            .str.strip()
        )


        frames.append(
            temp
        )


    df = pd.concat(
        frames,
        ignore_index=True
    )


    del frames
    gc.collect()


    print("\n============================================")
    print("RAW DATASET")
    print("============================================")

    print(
        "Shape:",
        df.shape
    )


    # ========================================================
    # 8. CHECK LABEL
    # ========================================================

    if "Label" not in df.columns:

        raise ValueError(
            "Column 'Label' not found."
        )


    df["Label"] = (
        df["Label"]
        .astype(str)
        .str.strip()
    )


    # ========================================================
    # 9. PRESERVE TIMESTAMP
    #
    # Timestamp is NOT used as a model feature.
    # It is retained for temporal ordering in Part 3.
    # ========================================================

    if "Timestamp" in df.columns:

        df["Timestamp"] = pd.to_datetime(
            df["Timestamp"],
            errors="coerce"
        )


    # ========================================================
    # 10. REPLACE INFINITY
    # ========================================================

    df.replace(
        [np.inf, -np.inf],
        np.nan,
        inplace=True
    )


    # ========================================================
    # 11. REMOVE INVALID LABELS
    # ========================================================

    df = df.dropna(
        subset=["Label"]
    ).reset_index(
        drop=True
    )


    print(
        "Shape after basic cleaning:",
        df.shape
    )


    # ========================================================
    # 12. EXCLUDED COLUMNS
    #
    # These are not used as ML features.
    # ========================================================

    EXCLUDE_COLUMNS = [

        "Label",

        "Flow ID",
        "FlowID",

        "Source IP",
        "Destination IP",

        "Source Port",
        "Destination Port",

        "Protocol",

        "Timestamp"
    ]


    # ========================================================
    # 13. IDENTIFY NUMERIC FEATURES
    # ========================================================

    candidate_features = [

        col

        for col in df.columns

        if col not in EXCLUDE_COLUMNS
    ]


    numeric_features = (

        df[
            candidate_features
        ]

        .select_dtypes(
            include=[np.number]
        )

        .columns

        .tolist()
    )


    print("\n============================================")
    print("FEATURE INFORMATION")
    print("============================================")

    print(
        "Numeric features:",
        len(numeric_features)
    )


    for i, feature in enumerate(
        numeric_features,
        start=1
    ):

        print(
            "{:02d}: {}".format(
                i,
                feature
            )
        )


    # ========================================================
    # 14. STRATIFIED 80/10/10 SPLIT
    # ========================================================

    print("\n============================================")
    print("STRATIFIED 80/10/10 SPLIT")
    print("============================================")


    train_df, temp_df = train_test_split(

        df,

        test_size=(
            VAL_RATIO +
            TEST_RATIO
        ),

        stratify=df["Label"],

        random_state=SEED,

        shuffle=True
    )


    val_fraction_of_temp = (
        VAL_RATIO /
        (
            VAL_RATIO +
            TEST_RATIO
        )
    )


    val_df, test_df = train_test_split(

        temp_df,

        test_size=(
            1.0 -
            val_fraction_of_temp
        ),

        stratify=temp_df["Label"],

        random_state=SEED,

        shuffle=True
    )


    del temp_df
    gc.collect()


    train_df = train_df.reset_index(
        drop=True
    )

    val_df = val_df.reset_index(
        drop=True
    )

    test_df = test_df.reset_index(
        drop=True
    )


    print(
        "Train:",
        train_df.shape
    )

    print(
        "Validation:",
        val_df.shape
    )

    print(
        "Test:",
        test_df.shape
    )


    # ========================================================
    # 15. CREATE LABEL ENCODER
    #
    # Fit ONLY on training labels.
    # ========================================================

    label_encoder = LabelEncoder()


    y_train = label_encoder.fit_transform(
        train_df["Label"]
    )


    y_val = label_encoder.transform(
        val_df["Label"]
    )


    y_test = label_encoder.transform(
        test_df["Label"]
    )


    NUM_CLASSES = len(
        label_encoder.classes_
    )


    print("\n============================================")
    print("CLASSES")
    print("============================================")

    print(
        "Number of classes:",
        NUM_CLASSES
    )


    for i, class_name in enumerate(
        label_encoder.classes_
    ):

        print(
            i,
            "->",
            class_name
        )


    # ========================================================
    # 16. CREATE FEATURE MATRICES
    # ========================================================

    X_train = train_df[
        numeric_features
    ].copy()


    X_val = val_df[
        numeric_features
    ].copy()


    X_test = test_df[
        numeric_features
    ].copy()


    # ========================================================
    # 17. TRAIN-ONLY MEDIAN IMPUTATION
    #
    # Important:
    # Median is calculated ONLY from training data.
    # ========================================================

    train_medians = (
        X_train
        .median()
    )


    X_train = X_train.fillna(
        train_medians
    )


    X_val = X_val.fillna(
        train_medians
    )


    X_test = X_test.fillna(
        train_medians
    )


    # ========================================================
    # 18. MUTUAL INFORMATION SAMPLE
    #
    # MI is calculated ONLY from training data.
    # Stratified sample preserves class proportions.
    # ========================================================

    print("\n============================================")
    print("MUTUAL INFORMATION")
    print("============================================")


    if len(X_train) > MI_SAMPLE_SIZE:

        X_mi, _, y_mi, _ = train_test_split(

            X_train,

            y_train,

            train_size=MI_SAMPLE_SIZE,

            stratify=y_train,

            random_state=SEED
        )

    else:

        X_mi = X_train.copy()

        y_mi = y_train.copy()


    print(
        "MI sample:",
        X_mi.shape
    )


    # ========================================================
    # 19. MUTUAL INFORMATION
    # ========================================================

    mi_scores = mutual_info_classif(

        X_mi,

        y_mi,

        random_state=SEED,

        n_neighbors=3,

        n_jobs=-1
    )


    mi_table = pd.DataFrame({

        "feature":
            X_train.columns,

        "MI":
            mi_scores

    })


    mi_table = (

        mi_table

        .sort_values(
            "MI",
            ascending=False
        )

        .reset_index(
            drop=True
        )
    )


    mi_table.insert(

        0,

        "rank",

        np.arange(
            1,
            len(mi_table) + 1
        )
    )


    # ========================================================
    # 20. SELECT TOP 64 FEATURES
    # ========================================================

    selected_64 = (

        mi_table

        .head(
            N_SELECTED_FEATURES
        )

        ["feature"]

        .tolist()
    )


    print("\n============================================")
    print("TOP 64 FEATURES")
    print("============================================")


    for _, row in mi_table.head(
        N_SELECTED_FEATURES
    ).iterrows():

        print(

            "{:02d}. {:45s} MI = {:.6f}".format(

                int(row["rank"]),

                row["feature"],

                row["MI"]
            )
        )


    # ========================================================
    # 21. IMAGE NORMALIZATION PARAMETERS
    #
    # Train min/max ONLY.
    # ========================================================

    image_train_min = (

        X_train[
            selected_64
        ]

        .min()

        .values

        .astype(
            np.float32
        )
    )


    image_train_max = (

        X_train[
            selected_64
        ]

        .max()

        .values

        .astype(
            np.float32
        )
    )


    # ========================================================
    # 22. SAVE SPLITS
    #
    # Keep:
    # - original dataframe
    # - numeric features
    # - labels
    # - timestamp for temporal ordering
    # ========================================================

    train_artifact = {

        "df": train_df,

        "X": X_train,

        "y": y_train,

        "selected_64": selected_64,

        "timestamp": (

            train_df["Timestamp"].copy()

            if "Timestamp" in train_df.columns

            else None
        )
    }


    val_artifact = {

        "df": val_df,

        "X": X_val,

        "y": y_val,

        "selected_64": selected_64,

        "timestamp": (

            val_df["Timestamp"].copy()

            if "Timestamp" in val_df.columns

            else None
        )
    }


    test_artifact = {

        "df": test_df,

        "X": X_test,

        "y": y_test,

        "selected_64": selected_64,

        "timestamp": (

            test_df["Timestamp"].copy()

            if "Timestamp" in test_df.columns

            else None
        )
    }


    with open(
        TRAIN_PATH,
        "wb"
    ) as f:

        pickle.dump(
            train_artifact,
            f,
            protocol=pickle.HIGHEST_PROTOCOL
        )


    with open(
        VAL_PATH,
        "wb"
    ) as f:

        pickle.dump(
            val_artifact,
            f,
            protocol=pickle.HIGHEST_PROTOCOL
        )


    with open(
        TEST_PATH,
        "wb"
    ) as f:

        pickle.dump(
            test_artifact,
            f,
            protocol=pickle.HIGHEST_PROTOCOL
        )


    # ========================================================
    # 23. SAVE MI RESULTS
    # ========================================================

    mi_table.to_csv(
        MI_RANKING_PATH,
        index=False
    )


    mi_table.head(
        N_SELECTED_FEATURES
    ).to_csv(
        SELECTED_FEATURES_PATH,
        index=False
    )


    # ========================================================
    # 24. SAVE LABEL ENCODER
    # ========================================================

    with open(
        LABEL_ENCODER_PATH,
        "wb"
    ) as f:

        pickle.dump(
            label_encoder,
            f
        )


    # ========================================================
    # 25. SAVE NORMALIZATION PARAMETERS
    # ========================================================

    image_normalization = {

        "features":
            selected_64,

        "train_min":
            image_train_min,

        "train_max":
            image_train_max,

        "image_size":
            IMAGE_SIZE
    }


    with open(
        IMAGE_MINMAX_PATH,
        "wb"
    ) as f:

        pickle.dump(
            image_normalization,
            f
        )


    # ========================================================
    # 26. SAVE FEATURE INFORMATION
    # ========================================================

    feature_information = {

        "numeric_features":
            numeric_features,

        "selected_64":
            selected_64,

        "train_medians":
            train_medians,

        "num_classes":
            NUM_CLASSES,

        "class_names":
            label_encoder.classes_.tolist(),

        "seed":
            SEED,

        "train_ratio":
            TRAIN_RATIO,

        "val_ratio":
            VAL_RATIO,

        "test_ratio":
            TEST_RATIO
    }


    with open(
        FEATURE_INFO_PATH,
        "wb"
    ) as f:

        pickle.dump(
            feature_information,
            f
        )


    # ========================================================
    # 27. CLEAN MEMORY
    # ========================================================

    del df

    gc.collect()


    print("\n============================================")
    print("PART 1 COMPLETED")
    print("============================================")

    print(
        "Train:",
        X_train.shape
    )

    print(
        "Validation:",
        X_val.shape
    )

    print(
        "Test:",
        X_test.shape
    )

    print(
        "Selected features:",
        len(selected_64)
    )

    print(
        "Classes:",
        NUM_CLASSES
    )

    print(
        "\nArtifacts saved to:"
    )

    print(
        OUTPUT_DIR
    )

## MASTER LOAD jika DISCONNET

In [ ]:
# ============================================================
# MASTER LOAD — RESUME SAFE
# ============================================================
# This cell:
#   1. Mounts Google Drive
#   2. Checks which artifacts already exist
#   3. Loads ONLY available artifacts
#   4. Does NOT train anything
#   5. Does NOT fail if later stages are missing
#
# Example:
#
# Part 1 ✓
# Part 2 ✓
# Part 3 ✗
# Part 4 ✗
#
# The cell will load Part 1 + Part 2 and skip Part 3 + Part 4.
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import gc
import pickle

import numpy as np
import pandas as pd


# ============================================================
# 2. GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount(
    "/content/drive"
)


# ============================================================
# 3. PATHS
# ============================================================

DATA_DIR = (
    "/content/drive/MyDrive/"
    "My Research/USULAN PENELITIAN DIKTI/2026/"
    "IDS-KN/luaran/MachineLearningCSV"
)


OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "My Research/USULAN PENELITIAN DIKTI/2026/"
    "IDS-KN/luaran/outputs"
)


# ============================================================
# 4. CONFIGURATION
# ============================================================

SEED = 42

IMAGE_SIZE = 8

N_IMAGE_FEATURES = 64

NUM_CLASSES = 15

DEVIT_EMBED_DIM = 96


# ============================================================
# 5. HELPER
# ============================================================

def load_pickle_if_exists(path):

    if os.path.exists(path):

        with open(
            path,
            "rb"
        ) as f:

            return pickle.load(f)

    return None


def load_numpy_if_exists(path):

    if os.path.exists(path):

        return np.load(
            path
        )

    return None


def status(path):

    if os.path.exists(path):

        return "✓ EXISTS"

    return "✗ NOT FOUND"


# ============================================================
# 6. ARTIFACT PATHS
# ============================================================

PATHS = {

    # --------------------------------------------------------
    # Part 1
    # --------------------------------------------------------

    "label_encoder":
        os.path.join(
            OUTPUT_DIR,
            "label_encoder.pkl"
        ),

    "feature_information":
        os.path.join(
            OUTPUT_DIR,
            "feature_information.pkl"
        ),

    # --------------------------------------------------------
    # Part 2 — XGBoost
    # --------------------------------------------------------

    "xgb_model":
        os.path.join(
            OUTPUT_DIR,
            "xgb_model.pkl"
        ),

    "xgb_train_prob":
        os.path.join(
            OUTPUT_DIR,
            "xgb_train_prob.npy"
        ),

    "xgb_val_prob":
        os.path.join(
            OUTPUT_DIR,
            "xgb_val_prob.npy"
        ),

    "xgb_test_prob":
        os.path.join(
            OUTPUT_DIR,
            "xgb_test_prob.npy"
        ),

    # --------------------------------------------------------
    # Part 3 — DeViT
    # --------------------------------------------------------

    "devit_checkpoint":
        os.path.join(
            OUTPUT_DIR,
            "devit_checkpoints",
            "best_temporal_deformable_devit.pt"
        ),

    "devit_train_features":
        os.path.join(
            OUTPUT_DIR,
            "devit_train_features.npy"
        ),

    "devit_val_features":
        os.path.join(
            OUTPUT_DIR,
            "devit_val_features.npy"
        ),

    "devit_test_features":
        os.path.join(
            OUTPUT_DIR,
            "devit_test_features.npy"
        ),

    "devit_train_labels":
        os.path.join(
            OUTPUT_DIR,
            "devit_train_labels.npy"
        ),

    "devit_val_labels":
        os.path.join(
            OUTPUT_DIR,
            "devit_val_labels.npy"
        ),

    "devit_test_labels":
        os.path.join(
            OUTPUT_DIR,
            "devit_test_labels.npy"
        ),

    # --------------------------------------------------------
    # Part 4 — Fusion
    # --------------------------------------------------------

    "fusion_train":
        os.path.join(
            OUTPUT_DIR,
            "fusion_train.npy"
        ),

    "fusion_val":
        os.path.join(
            OUTPUT_DIR,
            "fusion_val.npy"
        ),

    "fusion_test":
        os.path.join(
            OUTPUT_DIR,
            "fusion_test.npy"
        ),

    # --------------------------------------------------------
    # Part 4 — Random Forest
    # --------------------------------------------------------

    "rf_model":
        os.path.join(
            OUTPUT_DIR,
            "random_forest_fusion.pkl"
        ),

    "rf_results":
        os.path.join(
            OUTPUT_DIR,
            "random_forest_fusion_results.csv"
        ),

    "rf_report":
        os.path.join(
            OUTPUT_DIR,
            "random_forest_fusion_classification_report.csv"
        ),

    "rf_cm":
        os.path.join(
            OUTPUT_DIR,
            "random_forest_fusion_confusion_matrix.csv"
        )
}


# ============================================================
# 7. SHOW STATUS
# ============================================================

print(
    "\n============================================"
)

print(
    "PIPELINE ARTIFACT STATUS"
)

print(
    "============================================"
)


for name, path in PATHS.items():

    print(
        "{:<25} {}".format(
            name,
            status(path)
        )
    )


# ============================================================
# 8. LOAD PART 1
# ============================================================

print(
    "\n============================================"
)

print(
    "LOADING PART 1"
)

print(
    "============================================"
)


label_encoder = load_pickle_if_exists(
    PATHS["label_encoder"]
)


feature_information = load_pickle_if_exists(
    PATHS["feature_information"]
)


if label_encoder is not None:

    class_names = (
        label_encoder.classes_.tolist()
    )

    NUM_CLASSES = len(
        class_names
    )

    print(
        "Label encoder loaded."
    )

    print(
        "Number of classes:",
        NUM_CLASSES
    )

else:

    class_names = None

    print(
        "Label encoder not available."
    )


# ============================================================
# 9. LOAD PART 2 — XGBOOST
# ============================================================

print(
    "\n============================================"
)

print(
    "LOADING PART 2 — XGBOOST"
)

print(
    "============================================"
)


xgb_model = None

xgb_train_prob = None

xgb_val_prob = None

xgb_test_prob = None


if os.path.exists(
    PATHS["xgb_model"]
):

    xgb_model = load_pickle_if_exists(
        PATHS["xgb_model"]
    )

    print(
        "✓ XGBoost model loaded."
    )

else:

    print(
        "✗ XGBoost model not available."
    )


if os.path.exists(
    PATHS["xgb_train_prob"]
):

    xgb_train_prob = np.load(
        PATHS["xgb_train_prob"]
    )

    print(
        "✓ XGBoost train probability:",
        xgb_train_prob.shape
    )

else:

    print(
        "✗ XGBoost train probability "
        "not available."
    )


if os.path.exists(
    PATHS["xgb_val_prob"]
):

    xgb_val_prob = np.load(
        PATHS["xgb_val_prob"]
    )

    print(
        "✓ XGBoost validation probability:",
        xgb_val_prob.shape
    )

else:

    print(
        "✗ XGBoost validation probability "
        "not available."
    )


if os.path.exists(
    PATHS["xgb_test_prob"]
):

    xgb_test_prob = np.load(
        PATHS["xgb_test_prob"]
    )

    print(
        "✓ XGBoost test probability:",
        xgb_test_prob.shape
    )

else:

    print(
        "✗ XGBoost test probability "
        "not available."
    )


# ============================================================
# 10. LOAD PART 3 — DEVIT
# ============================================================

print(
    "\n============================================"
)

print(
    "LOADING PART 3 — DeViT"
)

print(
    "============================================"
)


devit_checkpoint = None

devit_train_features = None

devit_val_features = None

devit_test_features = None

devit_train_labels = None

devit_val_labels = None

devit_test_labels = None


# ------------------------------------------------------------
# Checkpoint
# ------------------------------------------------------------

if os.path.exists(
    PATHS["devit_checkpoint"]
):

    try:

        import torch

        devit_checkpoint = torch.load(

            PATHS["devit_checkpoint"],

            map_location="cpu"
        )

        print(
            "✓ DeViT checkpoint loaded."
        )

    except Exception as e:

        print(
            "⚠ DeViT checkpoint exists "
            "but could not be loaded yet."
        )

        print(
            "Reason:",
            e
        )

else:

    print(
        "✗ DeViT checkpoint not available."
    )


# ------------------------------------------------------------
# DeViT features
# ------------------------------------------------------------

if os.path.exists(
    PATHS["devit_train_features"]
):

    devit_train_features = np.load(
        PATHS["devit_train_features"]
    )

    print(
        "✓ DeViT train features:",
        devit_train_features.shape
    )

else:

    print(
        "✗ DeViT train features not available."
    )


if os.path.exists(
    PATHS["devit_val_features"]
):

    devit_val_features = np.load(
        PATHS["devit_val_features"]
    )

    print(
        "✓ DeViT validation features:",
        devit_val_features.shape
    )

else:

    print(
        "✗ DeViT validation features "
        "not available."
    )


if os.path.exists(
    PATHS["devit_test_features"]
):

    devit_test_features = np.load(
        PATHS["devit_test_features"]
    )

    print(
        "✓ DeViT test features:",
        devit_test_features.shape
    )

else:

    print(
        "✗ DeViT test features not available."
    )


# ------------------------------------------------------------
# DeViT labels
# ------------------------------------------------------------

if os.path.exists(
    PATHS["devit_train_labels"]
):

    devit_train_labels = np.load(
        PATHS["devit_train_labels"]
    )

    print(
        "✓ DeViT train labels:",
        devit_train_labels.shape
    )


if os.path.exists(
    PATHS["devit_val_labels"]
):

    devit_val_labels = np.load(
        PATHS["devit_val_labels"]
    )

    print(
        "✓ DeViT validation labels:",
        devit_val_labels.shape
    )


if os.path.exists(
    PATHS["devit_test_labels"]
):

    devit_test_labels = np.load(
        PATHS["devit_test_labels"]
    )

    print(
        "✓ DeViT test labels:",
        devit_test_labels.shape
    )


# ============================================================
# 11. LOAD PART 4 — FUSION
# ============================================================

print(
    "\n============================================"
)

print(
    "LOADING PART 4 — FUSION"
)

print(
    "============================================"
)


fusion_train = None

fusion_val = None

fusion_test = None


if os.path.exists(
    PATHS["fusion_train"]
):

    fusion_train = np.load(
        PATHS["fusion_train"]
    )

    print(
        "✓ Fusion train:",
        fusion_train.shape
    )

else:

    print(
        "✗ Fusion train not available."
    )


if os.path.exists(
    PATHS["fusion_val"]
):

    fusion_val = np.load(
        PATHS["fusion_val"]
    )

    print(
        "✓ Fusion validation:",
        fusion_val.shape
    )

else:

    print(
        "✗ Fusion validation not available."
    )


if os.path.exists(
    PATHS["fusion_test"]
):

    fusion_test = np.load(
        PATHS["fusion_test"]
    )

    print(
        "✓ Fusion test:",
        fusion_test.shape
    )

else:

    print(
        "✗ Fusion test not available."
    )


# ============================================================
# 12. LOAD RANDOM FOREST
# ============================================================

print(
    "\n============================================"
)

print(
    "LOADING RANDOM FOREST"
)

print(
    "============================================"
)


rf_model = None


if os.path.exists(
    PATHS["rf_model"]
):

    rf_model = load_pickle_if_exists(
        PATHS["rf_model"]
    )

    print(
        "✓ Random Forest loaded."
    )

else:

    print(
        "✗ Random Forest not available."
    )


# ============================================================
# 13. LOAD RESULTS
# ============================================================

rf_results_df = None

rf_report_df = None

rf_cm_df = None


if os.path.exists(
    PATHS["rf_results"]
):

    rf_results_df = pd.read_csv(
        PATHS["rf_results"]
    )


if os.path.exists(
    PATHS["rf_report"]
):

    rf_report_df = pd.read_csv(
        PATHS["rf_report"],
        index_col=0
    )


if os.path.exists(
    PATHS["rf_cm"]
):

    rf_cm_df = pd.read_csv(
        PATHS["rf_cm"],
        index_col=0
    )


# ============================================================
# 14. CREATE COMMON LABEL VARIABLES
# ============================================================

if devit_train_labels is not None:

    y_train = devit_train_labels

    y_val = devit_val_labels

    y_test = devit_test_labels

else:

    y_train = None

    y_val = None

    y_test = None


# ============================================================
# 15. FINAL PIPELINE STATUS
# ============================================================

print(
    "\n============================================"
)

print(
    "CURRENT PIPELINE STATUS"
)

print(
    "============================================"
)


part1_ready = (

    label_encoder is not None
)


part2_ready = (

    xgb_model is not None
    and
    xgb_test_prob is not None
)


part3_ready = (

    devit_train_features is not None
    and
    devit_val_features is not None
    and
    devit_test_features is not None
)


part4_ready = (

    fusion_train is not None
    and
    fusion_val is not None
    and
    fusion_test is not None
    and
    rf_model is not None
)


print(
    "Part 1:",
    "READY" if part1_ready else "NOT READY"
)


print(
    "Part 2 XGBoost:",
    "READY" if part2_ready else "NOT READY"
)


print(
    "Part 3 DeViT:",
    "READY" if part3_ready else "NOT READY"
)


print(
    "Part 4 Fusion + RF:",
    "READY" if part4_ready else "NOT READY"
)


# ============================================================
# 16. WHAT TO DO NEXT
# ============================================================

print(
    "\n============================================"
)

print(
    "NEXT ACTION"
)

print(
    "============================================"
)


if not part1_ready:

    print(
        "→ Run Part 1."
    )

elif not part2_ready:

    print(
        "→ Run Part 2 — XGBoost."
    )

elif not part3_ready:

    print(
        "→ Run Part 3 — DeViT."
    )

elif not part4_ready:

    print(
        "→ Run Part 4 — Fusion + Random Forest."
    )

else:

    print(
        "→ Core pipeline is COMPLETE."
    )

    print(
        "→ You can run ablation and robustness validation."
    )


print(
    "\n============================================"
)

print(
    "MASTER LOAD FINISHED"
)

print(
    "============================================"
)

## PART 2 XGBoost

In [ ]:
# ============================================================
# PART 2
# XGBOOST — FULL TABULAR FEATURES
#
# Input  :
#   train_final.pkl
#   val_final.pkl
#   test_final.pkl
#
# XGBoost menggunakan:
#   SEMUA numerical flow features
#
# Output:
#   1. Trained XGBoost model
#   2. Validation predictions
#   3. Test predictions
#   4. Train/Val/Test probabilities
#   5. Evaluation metrics
#   6. Classification reports
#   7. Confusion matrix
#
# Probability output akan digunakan pada PART 4
# untuk fusion dengan DeViT 96-D features.
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import gc
import pickle

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier


# ============================================================
# 2. CONFIGURATION
# ============================================================

SEED = 42

OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "My Research/USULAN PENELITIAN DIKTI/2026/"
    "IDS-KN/luaran/outputs"
)


# ============================================================
# 3. INPUT FILES
# ============================================================

TRAIN_PATH = os.path.join(
    OUTPUT_DIR,
    "train_final.pkl"
)

VAL_PATH = os.path.join(
    OUTPUT_DIR,
    "val_final.pkl"
)

TEST_PATH = os.path.join(
    OUTPUT_DIR,
    "test_final.pkl"
)

FEATURE_INFO_PATH = os.path.join(
    OUTPUT_DIR,
    "feature_information.pkl"
)

LABEL_ENCODER_PATH = os.path.join(
    OUTPUT_DIR,
    "label_encoder.pkl"
)


# ============================================================
# 4. OUTPUT FILES
# ============================================================

XGB_MODEL_PATH = os.path.join(
    OUTPUT_DIR,
    "xgboost_full_features.pkl"
)

XGB_TRAIN_PROB_PATH = os.path.join(
    OUTPUT_DIR,
    "xgb_train_prob.npy"
)

XGB_VAL_PROB_PATH = os.path.join(
    OUTPUT_DIR,
    "xgb_val_prob.npy"
)

XGB_TEST_PROB_PATH = os.path.join(
    OUTPUT_DIR,
    "xgb_test_prob.npy"
)

XGB_TRAIN_PRED_PATH = os.path.join(
    OUTPUT_DIR,
    "xgb_train_pred.npy"
)

XGB_VAL_PRED_PATH = os.path.join(
    OUTPUT_DIR,
    "xgb_val_pred.npy"
)

XGB_TEST_PRED_PATH = os.path.join(
    OUTPUT_DIR,
    "xgb_test_pred.npy"
)

XGB_RESULTS_PATH = os.path.join(
    OUTPUT_DIR,
    "xgboost_results.csv"
)

XGB_TRAIN_REPORT_PATH = os.path.join(
    OUTPUT_DIR,
    "xgboost_train_report.csv"
)

XGB_VAL_REPORT_PATH = os.path.join(
    OUTPUT_DIR,
    "xgboost_validation_report.csv"
)

XGB_TEST_REPORT_PATH = os.path.join(
    OUTPUT_DIR,
    "xgboost_test_report.csv"
)

XGB_CM_PATH = os.path.join(
    OUTPUT_DIR,
    "xgboost_test_confusion_matrix.csv"
)


# ============================================================
# 5. CHECK PART 1 ARTIFACTS
# ============================================================

required_files = [

    TRAIN_PATH,
    VAL_PATH,
    TEST_PATH,
    FEATURE_INFO_PATH,
    LABEL_ENCODER_PATH

]


print("============================================")
print("CHECKING PART 1 ARTIFACTS")
print("============================================")


for path in required_files:

    if not os.path.exists(path):

        raise FileNotFoundError(
            "\nRequired file not found:\n" +
            path
        )


print(
    "All required Part 1 files are available."
)


# ============================================================
# 6. LOAD PART 1 ARTIFACTS
# ============================================================

print("\n============================================")
print("LOADING DATA")
print("============================================")


with open(
    TRAIN_PATH,
    "rb"
) as f:

    train_artifact = pickle.load(f)


with open(
    VAL_PATH,
    "rb"
) as f:

    val_artifact = pickle.load(f)


with open(
    TEST_PATH,
    "rb"
) as f:

    test_artifact = pickle.load(f)


with open(
    FEATURE_INFO_PATH,
    "rb"
) as f:

    feature_information = pickle.load(f)


with open(
    LABEL_ENCODER_PATH,
    "rb"
) as f:

    label_encoder = pickle.load(f)


# ============================================================
# 7. EXTRACT FEATURES AND LABELS
# ============================================================

X_train = train_artifact["X"]

y_train = train_artifact["y"]


X_val = val_artifact["X"]

y_val = val_artifact["y"]


X_test = test_artifact["X"]

y_test = test_artifact["y"]


# ============================================================
# 8. MODEL INFORMATION
# ============================================================

numeric_features = (
    feature_information[
        "numeric_features"
    ]
)


NUM_CLASSES = (
    feature_information[
        "num_classes"
    ]
)


class_names = (
    feature_information[
        "class_names"
    ]
)


print("\n============================================")
print("DATA INFORMATION")
print("============================================")


print(
    "Training samples:",
    len(X_train)
)

print(
    "Validation samples:",
    len(X_val)
)

print(
    "Test samples:",
    len(X_test)
)

print(
    "Full numerical features:",
    len(numeric_features)
)

print(
    "Number of classes:",
    NUM_CLASSES
)


# ============================================================
# 9. ENSURE CONSISTENT FEATURE ORDER
# ============================================================

X_train = X_train[
    numeric_features
]

X_val = X_val[
    numeric_features
]

X_test = X_test[
    numeric_features
]


print(
    "\nX_train shape:",
    X_train.shape
)

print(
    "X_val shape:",
    X_val.shape
)

print(
    "X_test shape:",
    X_test.shape
)


# ============================================================
# 10. CREATE XGBOOST MODEL
# ============================================================

print("\n============================================")
print("CREATING XGBOOST MODEL")
print("============================================")


xgb_model = XGBClassifier(

    n_estimators=300,

    max_depth=8,

    learning_rate=0.05,

    subsample=0.8,

    colsample_bytree=0.8,

    objective="multi:softprob",

    num_class=NUM_CLASSES,

    eval_metric="mlogloss",

    tree_method="hist",

    random_state=SEED,

    n_jobs=-1,

    verbosity=1
)


# ============================================================
# 11. TRAIN XGBOOST
# ============================================================

print("\n============================================")
print("TRAINING XGBOOST")
print("============================================")

print(
    "Using FULL numerical flow features:"
)

print(
    "Number of features:",
    len(numeric_features)
)


xgb_model.fit(

    X_train,

    y_train,

    eval_set=[
        (
            X_val,
            y_val
        )
    ],

    verbose=True
)


print("\n============================================")
print("XGBOOST TRAINING COMPLETED")
print("============================================")


# ============================================================
# 12. GENERATE PROBABILITIES
#
# IMPORTANT:
# These probabilities will be used by PART 4.
# ============================================================

print("\n============================================")
print("GENERATING PROBABILITIES")
print("============================================")


xgb_train_prob = (
    xgb_model.predict_proba(
        X_train
    )
)


xgb_val_prob = (
    xgb_model.predict_proba(
        X_val
    )
)


xgb_test_prob = (
    xgb_model.predict_proba(
        X_test
    )
)


print(
    "Train probability shape:",
    xgb_train_prob.shape
)

print(
    "Validation probability shape:",
    xgb_val_prob.shape
)

print(
    "Test probability shape:",
    xgb_test_prob.shape
)


# Expected:
#
# Train: (N_train, 15)
# Val  : (N_val, 15)
# Test : (N_test, 15)


# ============================================================
# 13. GENERATE PREDICTIONS
# ============================================================

xgb_train_pred = np.argmax(
    xgb_train_prob,
    axis=1
)


xgb_val_pred = np.argmax(
    xgb_val_prob,
    axis=1
)


xgb_test_pred = np.argmax(
    xgb_test_prob,
    axis=1
)


# ============================================================
# 14. EVALUATION FUNCTION
# ============================================================

def evaluate_xgb(
    y_true,
    y_pred,
    dataset_name,
    class_names
):

    accuracy = accuracy_score(
        y_true,
        y_pred
    )


    macro_precision = precision_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )


    macro_recall = recall_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )


    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )


    weighted_f1 = f1_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )


    print(
        "\n============================================"
    )

    print(
        "XGBOOST —",
        dataset_name
    )

    print(
        "============================================"
    )


    print(
        "Accuracy:",
        accuracy
    )

    print(
        "Macro Precision:",
        macro_precision
    )

    print(
        "Macro Recall:",
        macro_recall
    )

    print(
        "Macro F1:",
        macro_f1
    )

    print(
        "Weighted F1:",
        weighted_f1
    )


    report = classification_report(

        y_true,

        y_pred,

        labels=np.arange(
            len(class_names)
        ),

        target_names=class_names,

        output_dict=True,

        zero_division=0
    )


    report_df = (
        pd.DataFrame(
            report
        )
        .transpose()
    )


    return {

        "dataset":
            dataset_name,

        "accuracy":
            accuracy,

        "macro_precision":
            macro_precision,

        "macro_recall":
            macro_recall,

        "macro_f1":
            macro_f1,

        "weighted_f1":
            weighted_f1

    }, report_df


# ============================================================
# 15. TRAIN EVALUATION
# ============================================================

train_results, train_report = evaluate_xgb(

    y_train,

    xgb_train_pred,

    "TRAIN",

    class_names
)


# ============================================================
# 16. VALIDATION EVALUATION
# ============================================================

val_results, val_report = evaluate_xgb(

    y_val,

    xgb_val_pred,

    "VALIDATION",

    class_names
)


# ============================================================
# 17. TEST EVALUATION
# ============================================================

test_results, test_report = evaluate_xgb(

    y_test,

    xgb_test_pred,

    "TEST",

    class_names
)


# ============================================================
# 18. PRINT FINAL TEST CLASSIFICATION REPORT
# ============================================================

print("\n============================================")
print("XGBOOST TEST CLASSIFICATION REPORT")
print("============================================")


print(

    classification_report(

        y_test,

        xgb_test_pred,

        labels=np.arange(
            NUM_CLASSES
        ),

        target_names=class_names,

        digits=4,

        zero_division=0
    )
)


# ============================================================
# 19. CONFUSION MATRIX
# ============================================================

xgb_cm = confusion_matrix(

    y_test,

    xgb_test_pred,

    labels=np.arange(
        NUM_CLASSES
    )
)


cm_df = pd.DataFrame(

    xgb_cm,

    index=class_names,

    columns=class_names
)


# ============================================================
# 20. SAVE XGBOOST MODEL
# ============================================================

print("\n============================================")
print("SAVING XGBOOST MODEL")
print("============================================")


with open(

    XGB_MODEL_PATH,

    "wb"

) as f:

    pickle.dump(

        xgb_model,

        f,

        protocol=pickle.HIGHEST_PROTOCOL
    )


# ============================================================
# 21. SAVE PROBABILITIES
#
# These files are IMPORTANT for PART 4.
# ============================================================

print(
    "Saving XGBoost probabilities..."
)


np.save(

    XGB_TRAIN_PROB_PATH,

    xgb_train_prob
)


np.save(

    XGB_VAL_PROB_PATH,

    xgb_val_prob
)


np.save(

    XGB_TEST_PROB_PATH,

    xgb_test_prob
)


# ============================================================
# 22. SAVE PREDICTIONS
# ============================================================

np.save(

    XGB_TRAIN_PRED_PATH,

    xgb_train_pred
)


np.save(

    XGB_VAL_PRED_PATH,

    xgb_val_pred
)


np.save(

    XGB_TEST_PRED_PATH,

    xgb_test_pred
)


# ============================================================
# 23. SAVE METRICS
# ============================================================

results_df = pd.DataFrame([

    train_results,

    val_results,

    test_results

])


results_df.to_csv(

    XGB_RESULTS_PATH,

    index=False
)


# ============================================================
# 24. SAVE CLASSIFICATION REPORTS
# ============================================================

train_report.to_csv(

    XGB_TRAIN_REPORT_PATH
)


val_report.to_csv(

    XGB_VAL_REPORT_PATH
)


test_report.to_csv(

    XGB_TEST_REPORT_PATH
)


# ============================================================
# 25. SAVE CONFUSION MATRIX
# ============================================================

cm_df.to_csv(

    XGB_CM_PATH
)


# ============================================================
# 26. FINAL SUMMARY
# ============================================================

print("\n============================================")
print("PART 2 COMPLETED")
print("============================================")


print("\nXGBoost uses:")
print(
    "FULL numerical flow features =",
    len(numeric_features)
)


print("\n---------------- VALIDATION ----------------")

print(
    "Accuracy:",
    val_results["accuracy"]
)

print(
    "Macro Precision:",
    val_results["macro_precision"]
)

print(
    "Macro Recall:",
    val_results["macro_recall"]
)

print(
    "Macro F1:",
    val_results["macro_f1"]
)


print("\n---------------- TEST ----------------")

print(
    "Accuracy:",
    test_results["accuracy"]
)

print(
    "Macro Precision:",
    test_results["macro_precision"]
)

print(
    "Macro Recall:",
    test_results["macro_recall"]
)

print(
    "Macro F1:",
    test_results["macro_f1"]
)


print("\n============================================")
print("SAVED ARTIFACTS")
print("============================================")


print(
    "Model:",
    XGB_MODEL_PATH
)

print(
    "Train probability:",
    XGB_TRAIN_PROB_PATH
)

print(
    "Validation probability:",
    XGB_VAL_PROB_PATH
)

print(
    "Test probability:",
    XGB_TEST_PROB_PATH
)

print(
    "\nPart 2 is ready for Part 4 fusion."
)


# ============================================================
# 27. MEMORY CLEANUP
# ============================================================

gc.collect()

## Part 3 DeViT v2

In [ ]:
# ============================================================
# PART 3.1
# LOAD AND VALIDATE MI-SELECTED FEATURES
#
# DeViT V2
#
# Purpose:
#   1. Load existing Part 1 artifacts
#   2. Retrieve the 64 MI-selected features
#   3. Validate train/validation/test consistency
#   4. Prepare DeViT V2 output directory
#
# IMPORTANT:
#   - No DeViT training
#   - No feature ordering yet
#   - No 8x8 transformation yet
#   - No large data copy
#   - Existing XGBoost artifacts are NOT modified
# ============================================================

import os
import gc
import json
import pickle

import numpy as np
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42

NUM_FEATURES = 64

NUM_CLASSES = 15


# ============================================================
# BASE OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "My Research/USULAN PENELITIAN DIKTI/2026/"
    "IDS-KN/luaran/outputs"
)


# ============================================================
# DEVIT V2 DIRECTORY
# ============================================================

DEVIT_V2_DIR = os.path.join(
    OUTPUT_DIR,
    "devit_v2"
)

os.makedirs(
    DEVIT_V2_DIR,
    exist_ok=True
)


# ============================================================
# SUBDIRECTORIES
# ============================================================

DEVIT_V2_ARTIFACT_DIR = os.path.join(
    DEVIT_V2_DIR,
    "artifacts"
)

DEVIT_V2_CHECKPOINT_DIR = os.path.join(
    DEVIT_V2_DIR,
    "checkpoints"
)

DEVIT_V2_RESULTS_DIR = os.path.join(
    DEVIT_V2_DIR,
    "results"
)

DEVIT_V2_ORDERING_DIR = os.path.join(
    DEVIT_V2_DIR,
    "ordering"
)


for directory in [

    DEVIT_V2_ARTIFACT_DIR,

    DEVIT_V2_CHECKPOINT_DIR,

    DEVIT_V2_RESULTS_DIR,

    DEVIT_V2_ORDERING_DIR

]:

    os.makedirs(
        directory,
        exist_ok=True
    )


# ============================================================
# PART 1 ARTIFACT PATHS
# ============================================================

TRAIN_FINAL_PATH = os.path.join(
    OUTPUT_DIR,
    "train_final.pkl"
)

VAL_FINAL_PATH = os.path.join(
    OUTPUT_DIR,
    "val_final.pkl"
)

TEST_FINAL_PATH = os.path.join(
    OUTPUT_DIR,
    "test_final.pkl"
)

LABEL_ENCODER_PATH = os.path.join(
    OUTPUT_DIR,
    "label_encoder.pkl"
)


# ============================================================
# HEADER
# ============================================================

print(
    "\n============================================"
)

print(
    "PART 3.1"
)

print(
    "DEVIT V2 — LOAD MI-SELECTED FEATURES"
)

print(
    "============================================"
)

print(
    "Selected features:",
    NUM_FEATURES
)

print(
    "Number of classes:",
    NUM_CLASSES
)

print(
    "Output directory:"
)

print(
    DEVIT_V2_DIR
)


# ============================================================
# CHECK FILES
# ============================================================

print(
    "\n============================================"
)

print(
    "CHECKING PART 1 ARTIFACTS"
)

print(
    "============================================"
)


required_paths = {

    "train_final":
        TRAIN_FINAL_PATH,

    "val_final":
        VAL_FINAL_PATH,

    "test_final":
        TEST_FINAL_PATH,

    "label_encoder":
        LABEL_ENCODER_PATH
}


missing = []


for name, path in required_paths.items():

    if os.path.exists(path):

        print(
            "FOUND:",
            name
        )

    else:

        print(
            "MISSING:",
            name
        )

        missing.append(
            path
        )


if len(missing) > 0:

    raise FileNotFoundError(
        "\nRequired Part 1 artifacts are missing."
    )


# ============================================================
# LOAD LABEL ENCODER
# ============================================================

print(
    "\n============================================"
)

print(
    "LOADING LABEL ENCODER"
)

print(
    "============================================"
)


with open(
    LABEL_ENCODER_PATH,
    "rb"
) as f:

    label_encoder = pickle.load(
        f
    )


class_names = (
    label_encoder.classes_.tolist()
)


print(
    "Classes:",
    len(class_names)
)


for i, class_name in enumerate(
    class_names
):

    print(
        f"{i:02d}: {class_name}"
    )


if len(class_names) != NUM_CLASSES:

    raise ValueError(
        "Number of classes does not match "
        f"expected {NUM_CLASSES}."
    )


# ============================================================
# LOAD TRAIN ARTIFACT
# ============================================================

print(
    "\n============================================"
)

print(
    "LOADING TRAIN ARTIFACT"
)

print(
    "============================================"
)


train_artifact = pd.read_pickle(
    TRAIN_FINAL_PATH
)


print(
    "Type:",
    type(train_artifact)
)


if not isinstance(
    train_artifact,
    dict
):

    raise TypeError(
        "train_final must be a dictionary."
    )


print(
    "Keys:",
    train_artifact.keys()
)


# ============================================================
# LOAD VALIDATION ARTIFACT
# ============================================================

print(
    "\n============================================"
)

print(
    "LOADING VALIDATION ARTIFACT"
)

print(
    "============================================"
)


val_artifact = pd.read_pickle(
    VAL_FINAL_PATH
)


print(
    "Type:",
    type(val_artifact)
)


if not isinstance(
    val_artifact,
    dict
):

    raise TypeError(
        "val_final must be a dictionary."
    )


print(
    "Keys:",
    val_artifact.keys()
)


# ============================================================
# LOAD TEST ARTIFACT
# ============================================================

print(
    "\n============================================"
)

print(
    "LOADING TEST ARTIFACT"
)

print(
    "============================================"
)


test_artifact = pd.read_pickle(
    TEST_FINAL_PATH
)


print(
    "Type:",
    type(test_artifact)
)


if not isinstance(
    test_artifact,
    dict
):

    raise TypeError(
        "test_final must be a dictionary."
    )


print(
    "Keys:",
    test_artifact.keys()
)


# ============================================================
# CHECK REQUIRED KEYS
# ============================================================

required_keys = [

    "X",

    "y",

    "selected_64"
]


for split_name, artifact in [

    ("TRAIN", train_artifact),

    ("VALIDATION", val_artifact),

    ("TEST", test_artifact)

]:

    for key in required_keys:

        if key not in artifact:

            raise KeyError(
                f"{key} missing from "
                f"{split_name} artifact."
            )


# ============================================================
# RETRIEVE SELECTED FEATURE NAMES
# ============================================================

selected_64 = train_artifact[
    "selected_64"
]


print(
    "\n============================================"
)

print(
    "MI-SELECTED FEATURE INFORMATION"
)

print(
    "============================================"
)


print(
    "Type:",
    type(selected_64)
)


if isinstance(
    selected_64,
    pd.Index
):

    selected_64 = (
        selected_64.tolist()
    )


elif isinstance(
    selected_64,
    np.ndarray
):

    selected_64 = (
        selected_64.tolist()
    )


elif isinstance(
    selected_64,
    tuple
):

    selected_64 = list(
        selected_64
    )


elif isinstance(
    selected_64,
    list
):

    selected_64 = list(
        selected_64
    )


else:

    raise TypeError(
        "Unsupported selected_64 type: "
        + str(type(selected_64))
    )


print(
    "Number of selected features:",
    len(selected_64)
)


if len(selected_64) != NUM_FEATURES:

    raise ValueError(
        "Expected exactly "
        f"{NUM_FEATURES} MI-selected features, "
        f"but found {len(selected_64)}."
    )


print(
    "\nSelected features:"
)


for i, feature_name in enumerate(
    selected_64
):

    print(
        f"{i+1:02d}: {feature_name}"
    )


# ============================================================
# VERIFY FEATURE CONSISTENCY
# ============================================================

print(
    "\n============================================"
)

print(
    "VERIFYING FEATURE CONSISTENCY"
)

print(
    "============================================"
)


for split_name, artifact in [

    ("TRAIN", train_artifact),

    ("VALIDATION", val_artifact),

    ("TEST", test_artifact)

]:

    X = artifact[
        "X"
    ]


    print(
        split_name,
        "X shape:",
        X.shape
    )


    print(
        split_name,
        "y shape:",
        artifact["y"].shape
    )


    if X.shape[1] != 77:

        raise ValueError(
            f"{split_name} X expected "
            "77 numerical flow features."
        )


    if X.shape[0] != artifact[
        "y"
    ].shape[0]:

        raise ValueError(
            f"{split_name}: X and y "
            "sample counts differ."
        )


# ============================================================
# IMPORTANT:
# SELECTED_64 IS A COLUMN-NAME LIST
#
# The actual X matrix contains 77 numerical features.
#
# We therefore need to determine the corresponding
# column indices from the artifact DataFrame.
# ============================================================

print(
    "\n============================================"
)

print(
    "MAPPING MI FEATURES TO COLUMN INDICES"
)

print(
    "============================================"
)


train_df = train_artifact[
    "df"
]


if not isinstance(
    train_df,
    pd.DataFrame
):

    raise TypeError(
        "train_artifact['df'] must be "
        "a pandas DataFrame."
    )


missing_selected = [

    feature

    for feature in selected_64

    if feature not in train_df.columns
]


if len(missing_selected) > 0:

    raise KeyError(
        "Some MI-selected features are "
        "not present in train DataFrame:\n"
        + str(missing_selected)
    )


selected_indices = [

    train_df.columns.get_loc(
        feature
    )

    for feature in selected_64

]


print(
    "Selected feature indices:"
)


for i, (feature, index) in enumerate(

    zip(
        selected_64,
        selected_indices
    )

):

    print(
        f"{i+1:02d}: "
        f"{feature} "
        f"-> column {index}"
    )


# ============================================================
# VERIFY SELECTED INDICES
# ============================================================

if len(
    selected_indices
) != NUM_FEATURES:

    raise ValueError(
        "Selected feature index count "
        "does not equal 64."
    )


if len(
    set(selected_indices)
) != NUM_FEATURES:

    raise ValueError(
        "Duplicate selected feature "
        "indices detected."
    )


# ============================================================
# SAVE FEATURE ORDER INFORMATION
#
# This is NOT the new topology ordering yet.
#
# This file stores the original MI ranking.
# Part 3.2 will generate the proposed
# topology-aware ordering.
# ============================================================

MI_ORDERING_PATH = os.path.join(

    DEVIT_V2_ORDERING_DIR,

    "mi_selected_64_original_order.json"

)


mi_ordering_info = {

    "method":
        "MI-selected features from Part 1",

    "number_of_features":
        NUM_FEATURES,

    "feature_names":
        selected_64,

    "column_indices":
        selected_indices

}


with open(
    MI_ORDERING_PATH,
    "w"
) as f:

    json.dump(
        mi_ordering_info,
        f,
        indent=4
    )


print(
    "\nSaved original MI ordering:"
)

print(
    MI_ORDERING_PATH
)


# ============================================================
# SAVE BASIC CONFIGURATION
# ============================================================

DEVIT_V2_CONFIG_PATH = os.path.join(

    DEVIT_V2_DIR,

    "devit_v2_config.json"

)


config = {

    "seed":
        SEED,

    "num_features":
        NUM_FEATURES,

    "num_classes":
        NUM_CLASSES,

    "image_height":
        8,

    "image_width":
        8,

    "feature_ordering":
        "TO_BE_DEFINED_IN_PART_3.2",

    "loss":
        "TO_BE_DEFINED_IN_PART_3.5",

    "model_selection":
        "Macro F1 primary, Macro Recall secondary",

    "calibration":
        "TO_BE_DEFINED_IN_PART_3.9",

    "class_names":
        class_names

}


with open(
    DEVIT_V2_CONFIG_PATH,
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=4
    )


# ============================================================
# SUMMARY
# ============================================================

print(
    "\n============================================"
)

print(
    "PART 3.1 COMPLETED"
)

print(
    "============================================"
)

print(
    "Train samples:",
    f"{train_artifact['X'].shape[0]:,}"
)

print(
    "Validation samples:",
    f"{val_artifact['X'].shape[0]:,}"
)

print(
    "Test samples:",
    f"{test_artifact['X'].shape[0]:,}"
)

print(
    "Original numerical features:",
    train_artifact["X"].shape[1]
)

print(
    "MI-selected features:",
    len(selected_64)
)

print(
    "Classes:",
    NUM_CLASSES
)

print(
    "\nNext:"
)

print(
    "PART 3.2 — Feature Ordering / Topology"
)

print(
    "============================================"
)


# ============================================================
# CLEANUP
# ============================================================

gc.collect()

In [ ]:
# ============================================================
# PART 3.2
# CORRELATION-AWARE FEATURE TOPOLOGY
#
# DeViT V2
#
# Purpose:
#   1. Extract the 64 MI-selected flow features
#   2. Compute correlation using TRAINING DATA ONLY
#   3. Build correlation-based feature distance
#   4. Perform hierarchical clustering
#   5. Apply optimal leaf ordering
#   6. Generate correlation-aware 64-feature ordering
#   7. Generate 8x8 feature topology
#   8. Save all topology artifacts
#
# IMPORTANT:
#   - Training data ONLY is used for topology construction.
#   - Validation and test data are never used here.
#   - No DeViT training is performed.
#   - XGBoost artifacts are not modified.
#   - Feature selection itself is NOT repeated.
# ============================================================

import os
import gc
import json

import numpy as np
import pandas as pd

from scipy.cluster.hierarchy import (
    linkage,
    leaves_list,
    optimal_leaf_ordering
)

from scipy.spatial.distance import squareform


# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42

NUM_FEATURES = 64

np.random.seed(SEED)


# ============================================================
# CHECK PART 3.1 VARIABLES
# ============================================================

print(
    "\n============================================"
)

print(
    "PART 3.2"
)

print(
    "CORRELATION-AWARE FEATURE TOPOLOGY"
)

print(
    "============================================"
)


required_variables = [

    "train_artifact",
    "selected_64",
    "DEVIT_V2_ORDERING_DIR",
    "NUM_FEATURES"

]


missing_variables = [

    variable

    for variable in required_variables

    if variable not in globals()

]


if len(missing_variables) > 0:

    raise RuntimeError(

        "\nRequired variables from Part 3.1 "
        "are missing:\n"
        + str(missing_variables)
        + "\n\nPlease run Part 3.1 first."

    )


# ============================================================
# PREPARE OUTPUT DIRECTORY
# ============================================================

os.makedirs(

    DEVIT_V2_ORDERING_DIR,

    exist_ok=True

)


# ============================================================
# GET TRAINING X
# ============================================================

print(
    "\n============================================"
)

print(
    "EXTRACTING TRAINING FEATURES"
)

print(
    "============================================"
)


train_X = train_artifact[
    "X"
]


print(
    "Training matrix shape:",
    train_X.shape
)


# ============================================================
# VALIDATE X TYPE
# ============================================================

if not isinstance(

    train_X,

    pd.DataFrame

):

    raise TypeError(

        "train_artifact['X'] must be "
        "a pandas DataFrame for this "
        "feature-ordering implementation."

    )


# ============================================================
# VALIDATE ORIGINAL FEATURE COUNT
# ============================================================

if train_X.shape[1] != 77:

    raise ValueError(

        "Expected 77 numerical flow features "
        f"in X, but found {train_X.shape[1]}."

    )


# ============================================================
# VALIDATE SELECTED FEATURE LIST
# ============================================================

if not isinstance(

    selected_64,

    list

):

    selected_64 = list(
        selected_64
    )


if len(selected_64) != NUM_FEATURES:

    raise ValueError(

        "Expected exactly "
        f"{NUM_FEATURES} MI-selected features, "
        f"but found {len(selected_64)}."

    )


# ============================================================
# CHECK FEATURE NAMES
#
# IMPORTANT:
#   We use feature names directly.
#   We do NOT use indices derived from df.
# ============================================================

print(
    "\n============================================"
)

print(
    "VALIDATING MI-SELECTED FEATURE NAMES"
)

print(
    "============================================"
)


missing_selected_features = [

    feature

    for feature in selected_64

    if feature not in train_X.columns

]


if len(missing_selected_features) > 0:

    print(
        "\nMissing features:"
    )

    for feature in missing_selected_features:

        print(
            "-",
            feature
        )


    raise KeyError(

        "Some MI-selected features "
        "are not present in train_X."

    )


print(
    "All 64 MI-selected features "
    "are present in X."
)


# ============================================================
# EXTRACT 64 FEATURES BY NAME
#
# This fixes the previous indexing problem.
# ============================================================

print(
    "\n============================================"
)

print(
    "EXTRACTING 64 MI-SELECTED FEATURES"
)

print(
    "============================================"
)


X_selected = train_X[
    selected_64
].copy()


print(
    "Selected matrix shape:",
    X_selected.shape
)


if X_selected.shape[1] != NUM_FEATURES:

    raise ValueError(

        "Selected matrix contains "
        f"{X_selected.shape[1]} features; "
        f"expected {NUM_FEATURES}."

    )


# ============================================================
# GET INDICES RELATIVE TO X
#
# These indices are safe for train_X because they are
# obtained directly from train_X.columns.
# ============================================================

selected_indices_in_X = [

    train_X.columns.get_loc(
        feature
    )

    for feature in selected_64

]


# ============================================================
# VALIDATE INDICES
# ============================================================

if len(

    selected_indices_in_X

) != NUM_FEATURES:

    raise ValueError(

        "Number of selected X indices "
        "does not equal 64."

    )


if len(

    set(selected_indices_in_X)

) != NUM_FEATURES:

    raise ValueError(

        "Duplicate feature indices detected."

    )


if min(

    selected_indices_in_X

) < 0:

    raise ValueError(

        "Invalid negative feature index detected."

    )


if max(

    selected_indices_in_X

) >= train_X.shape[1]:

    raise ValueError(

        "Selected feature index exceeds "
        "the number of columns in X."

    )


print(
    "\nMI-selected features and X indices:"
)


for i, (

    feature,
    index

) in enumerate(

    zip(

        selected_64,
        selected_indices_in_X

    )

):

    print(

        f"{i+1:02d}: "
        f"{feature} "
        f"-> X column {index}"

    )


# ============================================================
# CHECK NUMERICAL DATA
# ============================================================

print(
    "\n============================================"
)

print(
    "CHECKING NUMERICAL VALUES"
)

print(
    "============================================"
)


# Convert to numeric safely
for feature in selected_64:

    if not pd.api.types.is_numeric_dtype(

        X_selected[feature]

    ):

        X_selected[feature] = pd.to_numeric(

            X_selected[feature],

            errors="coerce"

        )


# Replace infinities
X_selected = X_selected.replace(

    [np.inf, -np.inf],

    np.nan

)


missing_before = int(

    X_selected.isna().sum().sum()

)


print(
    "Missing/invalid values:",
    missing_before
)


# ============================================================
# HANDLE MISSING VALUES
#
# Training medians are calculated ONLY from training data.
# ============================================================

if missing_before > 0:

    print(
        "Replacing missing values "
        "with training medians..."
    )


    feature_medians = X_selected.median(

        numeric_only=True

    )


    X_selected = X_selected.fillna(

        feature_medians

    )


missing_after = int(

    X_selected.isna().sum().sum()

)


if missing_after > 0:

    raise ValueError(

        "Missing values remain after "
        "median imputation."

    )


# ============================================================
# COMPUTE PEARSON CORRELATION
#
# TRAINING DATA ONLY
# ============================================================

print(
    "\n============================================"
)

print(
    "COMPUTING PEARSON CORRELATION"
)

print(
    "============================================"
)

print(
    "Data source: TRAINING SET ONLY"
)

print(
    "Number of features:",
    NUM_FEATURES
)


correlation_df = X_selected.corr(

    method="pearson"

)


correlation_matrix = (

    correlation_df.values

    .astype(
        np.float64
    )

)


# ============================================================
# VALIDATE CORRELATION MATRIX
# ============================================================

expected_shape = (

    NUM_FEATURES,
    NUM_FEATURES

)


if correlation_matrix.shape != expected_shape:

    raise ValueError(

        "Unexpected correlation matrix shape: "

        + str(correlation_matrix.shape)

    )


if not np.all(

    np.isfinite(
        correlation_matrix
    )

):

    raise ValueError(

        "Correlation matrix contains "
        "non-finite values."

    )


# Symmetrize
correlation_matrix = (

    correlation_matrix

    + correlation_matrix.T

) / 2.0


# Exact diagonal
np.fill_diagonal(

    correlation_matrix,

    1.0

)


# ============================================================
# ABSOLUTE CORRELATION
#
# Strong positive and negative relationships are both
# considered strong feature relationships.
# ============================================================

abs_correlation_matrix = np.abs(

    correlation_matrix

)


# ============================================================
# DISTANCE MATRIX
#
# d(i,j) = 1 - |corr(i,j)|
#
# Strong correlation:
#   |r| -> 1
#   distance -> 0
#
# Weak correlation:
#   |r| -> 0
#   distance -> 1
# ============================================================

feature_distance_matrix = (

    1.0

    - abs_correlation_matrix

)


# Numerical safety
feature_distance_matrix = np.clip(

    feature_distance_matrix,

    0.0,

    1.0

)


# Exact diagonal
np.fill_diagonal(

    feature_distance_matrix,

    0.0

)


# Symmetrize
feature_distance_matrix = (

    feature_distance_matrix

    + feature_distance_matrix.T

) / 2.0


# ============================================================
# VALIDATE DISTANCE MATRIX
# ============================================================

if not np.all(

    np.isfinite(
        feature_distance_matrix
    )

):

    raise ValueError(

        "Feature distance matrix "
        "contains non-finite values."

    )


if not np.allclose(

    feature_distance_matrix,

    feature_distance_matrix.T,

    atol=1e-8

):

    raise ValueError(

        "Feature distance matrix "
        "is not symmetric."

    )


# ============================================================
# SAVE CORRELATION MATRIX
# ============================================================

CORRELATION_MATRIX_PATH = os.path.join(

    DEVIT_V2_ORDERING_DIR,

    "correlation_matrix_64.npy"

)


np.save(

    CORRELATION_MATRIX_PATH,

    correlation_matrix

)


# ============================================================
# SAVE CORRELATION CSV
# ============================================================

CORRELATION_CSV_PATH = os.path.join(

    DEVIT_V2_ORDERING_DIR,

    "correlation_matrix_64.csv"

)


correlation_df.to_csv(

    CORRELATION_CSV_PATH

)


# ============================================================
# SAVE ABSOLUTE CORRELATION
# ============================================================

ABS_CORRELATION_PATH = os.path.join(

    DEVIT_V2_ORDERING_DIR,

    "absolute_correlation_matrix_64.npy"

)


np.save(

    ABS_CORRELATION_PATH,

    abs_correlation_matrix

)


# ============================================================
# SAVE DISTANCE MATRIX
# ============================================================

DISTANCE_MATRIX_PATH = os.path.join(

    DEVIT_V2_ORDERING_DIR,

    "feature_distance_matrix_64.npy"

)


np.save(

    DISTANCE_MATRIX_PATH,

    feature_distance_matrix

)


# ============================================================
# CONVERT DISTANCE MATRIX TO CONDENSED FORM
# ============================================================

print(
    "\n============================================"
)

print(
    "PREPARING HIERARCHICAL CLUSTERING"
)

print(
    "============================================"
)


condensed_distance = squareform(

    feature_distance_matrix,

    checks=True

)


# ============================================================
# HIERARCHICAL CLUSTERING
#
# Average linkage is used.
# ============================================================

print(
    "Linkage method: average"
)


Z = linkage(

    condensed_distance,

    method="average"

)


# ============================================================
# INITIAL LEAF ORDER
# ============================================================

initial_order = leaves_list(

    Z

)


# ============================================================
# OPTIMAL LEAF ORDERING
# ============================================================

print(
    "\n============================================"
)

print(
    "APPLYING OPTIMAL LEAF ORDERING"
)

print(
    "============================================"
)


Z_optimal = optimal_leaf_ordering(

    Z,

    condensed_distance

)


ordered_indices = leaves_list(

    Z_optimal

)


# ============================================================
# VALIDATE ORDER
# ============================================================

if len(

    ordered_indices

) != NUM_FEATURES:

    raise ValueError(

        "Optimal ordering does not contain "
        "64 features."

    )


if len(

    np.unique(
        ordered_indices
    )

) != NUM_FEATURES:

    raise ValueError(

        "Duplicate feature detected "
        "in optimal ordering."

    )


if set(

    ordered_indices.tolist()

) != set(

    range(NUM_FEATURES)

):

    raise ValueError(

        "Optimal ordering does not contain "
        "all 64 selected features."

    )


# ============================================================
# MAP ORDERED INDICES TO FEATURE NAMES
# ============================================================

ordered_feature_names = [

    selected_64[
        int(index)
    ]

    for index in ordered_indices

]


# ============================================================
# MAP ORDERED FEATURES TO ORIGINAL X COLUMN INDICES
# ============================================================

ordered_original_indices = [

    selected_indices_in_X[
        int(index)
    ]

    for index in ordered_indices

]


# ============================================================
# PRINT FINAL ORDERING
# ============================================================

print(
    "\n============================================"
)

print(
    "CORRELATION-AWARE FEATURE ORDER"
)

print(
    "============================================"
)


for position in range(

    NUM_FEATURES

):

    selected_index = int(

        ordered_indices[
            position
        ]

    )

    feature_name = (

        ordered_feature_names[
            position
        ]

    )

    original_index = int(

        ordered_original_indices[
            position
        ]

    )

    print(

        f"{position+1:02d}: "
        f"{feature_name} "
        f"(selected index={selected_index}, "
        f"X column={original_index})"

    )


# ============================================================
# BUILD 8 × 8 FEATURE TOPOLOGY
# ============================================================

print(
    "\n============================================"
)

print(
    "BUILDING 8 × 8 FEATURE TOPOLOGY"
)

print(
    "============================================"
)


topology_8x8 = np.array(

    ordered_feature_names,

    dtype=object

).reshape(

    8,
    8

)


topology_indices_8x8 = np.array(

    ordered_indices,

    dtype=np.int64

).reshape(

    8,
    8

)


topology_original_indices_8x8 = np.array(

    ordered_original_indices,

    dtype=np.int64

).reshape(

    8,
    8

)


# ============================================================
# PRINT 8 × 8 TOPOLOGY
# ============================================================

print(
    "\n8 × 8 FEATURE TOPOLOGY:"
)

print(
    "--------------------------------------------"
)


for row in range(8):

    row_items = []

    for col in range(8):

        feature = str(

            topology_8x8[
                row,
                col
            ]

        )

        row_items.append(

            feature

        )

    print(

        " | ".join(
            row_items
        )

    )


# ============================================================
# SAVE ORDERED FEATURE INDICES
# ============================================================

ORDERED_INDICES_PATH = os.path.join(

    DEVIT_V2_ORDERING_DIR,

    "correlation_ordered_indices.npy"

)


np.save(

    ORDERED_INDICES_PATH,

    np.asarray(

        ordered_indices,

        dtype=np.int64

    )

)


# ============================================================
# SAVE ORDERED X COLUMN INDICES
# ============================================================

ORDERED_X_INDICES_PATH = os.path.join(

    DEVIT_V2_ORDERING_DIR,

    "correlation_ordered_X_column_indices.npy"

)


np.save(

    ORDERED_X_INDICES_PATH,

    np.asarray(

        ordered_original_indices,

        dtype=np.int64

    )

)


# ============================================================
# SAVE 8 × 8 FEATURE NAMES
# ============================================================

TOPOLOGY_8X8_PATH = os.path.join(

    DEVIT_V2_ORDERING_DIR,

    "feature_topology_8x8.npy"

)


np.save(

    TOPOLOGY_8X8_PATH,

    topology_8x8

)


# ============================================================
# SAVE 8 × 8 SELECTED-FEATURE INDICES
# ============================================================

TOPOLOGY_INDICES_8X8_PATH = os.path.join(

    DEVIT_V2_ORDERING_DIR,

    "feature_topology_indices_8x8.npy"

)


np.save(

    TOPOLOGY_INDICES_8X8_PATH,

    topology_indices_8x8

)


# ============================================================
# SAVE 8 × 8 ORIGINAL X COLUMN INDICES
# ============================================================

TOPOLOGY_X_INDICES_8X8_PATH = os.path.join(

    DEVIT_V2_ORDERING_DIR,

    "feature_topology_X_column_indices_8x8.npy"

)


np.save(

    TOPOLOGY_X_INDICES_8X8_PATH,

    topology_original_indices_8x8

)


# ============================================================
# SAVE ORDERING JSON
# ============================================================

ORDERING_JSON_PATH = os.path.join(

    DEVIT_V2_ORDERING_DIR,

    "correlation_aware_ordering.json"

)


ordering_info = {

    "method":
        "Correlation-aware feature topology",

    "feature_selection":
        "MI-selected 64 features",

    "correlation_method":
        "absolute Pearson correlation",

    "distance_method":
        "1 - abs(Pearson correlation)",

    "clustering_method":
        "hierarchical average linkage",

    "leaf_ordering":
        "optimal leaf ordering",

    "data_source":
        "training set only",

    "num_features":
        NUM_FEATURES,

    "grid_shape":
        [8, 8],

    "ordered_feature_names":
        ordered_feature_names,

    "ordered_indices_within_selected_64":
        [
            int(x)
            for x in ordered_indices
        ],

    "ordered_X_column_indices":
        [
            int(x)
            for x in ordered_original_indices
        ],

    "topology_8x8":
        topology_8x8.tolist()

}


with open(

    ORDERING_JSON_PATH,

    "w"

) as f:

    json.dump(

        ordering_info,

        f,

        indent=4

    )


# ============================================================
# SAVE HUMAN-READABLE TOPOLOGY CSV
# ============================================================

TOPOLOGY_CSV_PATH = os.path.join(

    DEVIT_V2_ORDERING_DIR,

    "feature_topology_8x8.csv"

)


topology_rows = []


for row in range(8):

    for col in range(8):

        position = (

            row * 8
            + col

        )

        selected_index = int(

            ordered_indices[
                position
            ]

        )

        original_x_index = int(

            ordered_original_indices[
                position
            ]

        )

        feature_name = (

            ordered_feature_names[
                position
            ]

        )

        topology_rows.append({

            "row":
                row,

            "column":
                col,

            "position":
                position,

            "feature":
                feature_name,

            "selected_64_index":
                selected_index,

            "X_column_index":
                original_x_index

        })


topology_df = pd.DataFrame(

    topology_rows

)


topology_df.to_csv(

    TOPOLOGY_CSV_PATH,

    index=False

)


# ============================================================
# SAVE ORIGINAL MI ORDER + TOPOLOGY COMPARISON
# ============================================================

ORDER_COMPARISON_PATH = os.path.join(

    DEVIT_V2_ORDERING_DIR,

    "ordering_comparison.csv"

)


comparison_rows = []


for position in range(

    NUM_FEATURES

):

    original_feature = (

        selected_64[
            position
        ]

    )

    ordered_feature = (

        ordered_feature_names[
            position
        ]

    )

    comparison_rows.append({

        "position":
            position,

        "original_MI_order":
            original_feature,

        "correlation_aware_order":
            ordered_feature

    })


comparison_df = pd.DataFrame(

    comparison_rows

)


comparison_df.to_csv(

    ORDER_COMPARISON_PATH,

    index=False

)


# ============================================================
# UPDATE DEVIT V2 CONFIGURATION
# ============================================================

DEVIT_V2_DIR = os.path.dirname(

    DEVIT_V2_ORDERING_DIR

)


DEVIT_V2_CONFIG_PATH = os.path.join(

    DEVIT_V2_DIR,

    "devit_v2_config.json"

)


config = {}


if os.path.exists(

    DEVIT_V2_CONFIG_PATH

):

    try:

        with open(

            DEVIT_V2_CONFIG_PATH,

            "r"

        ) as f:

            config = json.load(
                f
            )

    except Exception:

        config = {}


config.update({

    "seed":
        SEED,

    "num_features":
        NUM_FEATURES,

    "grid_shape":
        [8, 8],

    "feature_ordering":
        "correlation-aware",

    "correlation_method":
        "absolute Pearson",

    "distance_method":
        "1 - abs(Pearson correlation)",

    "clustering_method":
        "average linkage",

    "leaf_ordering":
        "optimal leaf ordering",

    "topology_data_source":
        "training set only"

})


with open(

    DEVIT_V2_CONFIG_PATH,

    "w"

) as f:

    json.dump(

        config,

        f,

        indent=4

    )


# ============================================================
# FINAL SUMMARY
# ============================================================

print(
    "\n============================================"
)

print(
    "PART 3.2 COMPLETED"
)

print(
    "============================================"
)

print(
    "Training samples:",
    f"{train_X.shape[0]:,}"
)

print(
    "Original X features:",
    train_X.shape[1]
)

print(
    "MI-selected features:",
    NUM_FEATURES
)

print(
    "Correlation matrix:",
    correlation_matrix.shape
)

print(
    "Distance matrix:",
    feature_distance_matrix.shape
)

print(
    "Final topology:",
    topology_8x8.shape
)

print(
    "\nOrdering:"
)

print(
    "MI-selected 64"
)

print(
    "      ↓"
)

print(
    "Absolute Pearson correlation"
)

print(
    "      ↓"
)

print(
    "Distance = 1 - |correlation|"
)

print(
    "      ↓"
)

print(
    "Average-linkage clustering"
)

print(
    "      ↓"
)

print(
    "Optimal leaf ordering"
)

print(
    "      ↓"
)

print(
    "8 × 8 feature topology"
)

print(
    "\nSaved artifacts:"
)

print(
    DEVIT_V2_ORDERING_DIR
)

print(
    "\nNext:"
)

print(
    "PART 3.3 — Build ordered 8 × 8 representations"
)

print(
    "============================================"
)


# ============================================================
# CLEANUP
# ============================================================

del X_selected

del correlation_df

del correlation_matrix

del abs_correlation_matrix

del feature_distance_matrix

del condensed_distance

del Z

del Z_optimal

gc.collect()

In [ ]:
# ============================================================
# PART 3.3
# BUILD CORRELATION-AWARE 8×8 FEATURE REPRESENTATIONS
#
# DeViT V2
#
# Pipeline:
#
# 77 flow features
#        ↓
# 64 MI-selected features
#        ↓
# Correlation-aware ordering
#        ↓
# 8 × 8 representation
#
# IMPORTANT:
#   - No model training
#   - No correlation recalculation
#   - No clustering recalculation
#   - Uses ordering artifact from Part 3.2
#   - Processes data in chunks
#   - Saves train/validation/test representations
# ============================================================

import os
import gc
import json

import numpy as np
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42

NUM_FEATURES = 64

GRID_SIZE = 8

CHUNK_SIZE = 32768

np.random.seed(SEED)


# ============================================================
# HEADER
# ============================================================

print(
    "\n============================================"
)

print(
    "PART 3.3"
)

print(
    "BUILD CORRELATION-AWARE 8×8 REPRESENTATIONS"
)

print(
    "============================================"
)


# ============================================================
# CHECK REQUIRED VARIABLES
# ============================================================

required_variables = [

    "train_artifact",

    "val_artifact",

    "test_artifact",

    "DEVIT_V2_ORDERING_DIR",

    "NUM_FEATURES"

]


missing_variables = [

    variable

    for variable in required_variables

    if variable not in globals()

]


if len(missing_variables) > 0:

    raise RuntimeError(

        "\nMissing required variables:\n"
        + str(missing_variables)
        + "\n\n"
        "Please make sure Part 3.1 and Part 3.2 "
        "have been completed."

    )


# ============================================================
# CHECK ORDERING ARTIFACTS
# ============================================================

ORDERED_X_INDICES_PATH = os.path.join(

    DEVIT_V2_ORDERING_DIR,

    "correlation_ordered_X_column_indices.npy"

)


TOPOLOGY_8X8_PATH = os.path.join(

    DEVIT_V2_ORDERING_DIR,

    "feature_topology_8x8.npy"

)


ORDERING_JSON_PATH = os.path.join(

    DEVIT_V2_ORDERING_DIR,

    "correlation_aware_ordering.json"

)


print(
    "\n============================================"
)

print(
    "CHECKING PART 3.2 ARTIFACTS"
)

print(
    "============================================"
)


required_artifacts = [

    ORDERED_X_INDICES_PATH,

    TOPOLOGY_8X8_PATH,

    ORDERING_JSON_PATH

]


missing_artifacts = [

    path

    for path in required_artifacts

    if not os.path.exists(path)

]


if len(missing_artifacts) > 0:

    for path in missing_artifacts:

        print(
            "MISSING:",
            path
        )

    raise FileNotFoundError(

        "\nPart 3.2 artifacts are incomplete. "
        "Please run Part 3.2 first."

    )


for path in required_artifacts:

    print(
        "FOUND:",
        os.path.basename(path)
    )


# ============================================================
# LOAD ORDERING
# ============================================================

ordered_X_indices = np.load(

    ORDERED_X_INDICES_PATH

)


topology_8x8 = np.load(

    TOPOLOGY_8X8_PATH,

    allow_pickle=True

)


with open(

    ORDERING_JSON_PATH,

    "r"

) as f:

    ordering_info = json.load(
        f
    )


# ============================================================
# VALIDATE ORDERING
# ============================================================

print(
    "\n============================================"
)

print(
    "VALIDATING FEATURE ORDER"
)

print(
    "============================================"
)


ordered_X_indices = np.asarray(

    ordered_X_indices,

    dtype=np.int64

)


if ordered_X_indices.shape != (

    NUM_FEATURES,

):

    raise ValueError(

        "Expected 64 ordered feature indices, "
        f"got {ordered_X_indices.shape}."

    )


if len(

    np.unique(
        ordered_X_indices
    )

) != NUM_FEATURES:

    raise ValueError(

        "Duplicate X feature indices "
        "detected in ordering."

    )


if np.min(

    ordered_X_indices

) < 0:

    raise ValueError(

        "Negative X feature index detected."

    )


if np.max(

    ordered_X_indices

) >= 77:

    raise ValueError(

        "Ordering contains an X feature index "
        "outside the 77-feature range."

    )


if topology_8x8.shape != (

    GRID_SIZE,
    GRID_SIZE

):

    raise ValueError(

        "8×8 topology artifact has "
        f"unexpected shape: {topology_8x8.shape}"

    )


print(
    "Ordered features:",
    len(ordered_X_indices)
)

print(
    "Grid:",
    topology_8x8.shape
)

print(
    "Maximum X column index:",
    int(np.max(ordered_X_indices))
)


# ============================================================
# DISPLAY TOPOLOGY
# ============================================================

print(
    "\n============================================"
)

print(
    "CORRELATION-AWARE 8×8 TOPOLOGY"
)

print(
    "============================================"
)


for row in range(GRID_SIZE):

    row_features = [

        str(
            topology_8x8[
                row,
                col
            ]
        )

        for col in range(GRID_SIZE)

    ]

    print(
        " | ".join(
            row_features
        )
    )


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

DEVIT_V2_REPRESENTATION_DIR = os.path.join(

    os.path.dirname(
        DEVIT_V2_ORDERING_DIR
    ),

    "representations"

)


os.makedirs(

    DEVIT_V2_REPRESENTATION_DIR,

    exist_ok=True

)


# ============================================================
# FUNCTION:
# BUILD 8×8 REPRESENTATION
# ============================================================

def build_8x8_representation(

    artifact,

    dataset_name,

    output_dir,

    ordered_indices,

    chunk_size=32768

):

    """
    Convert the 64 correlation-aware ordered features
    into an 8×8 representation.

    Input:
        artifact["X"]

    Output:
        N × 8 × 8 float32 array
    """

    print(
        "\n============================================"
    )

    print(
        f"BUILDING 8×8 REPRESENTATION — "
        f"{dataset_name.upper()}"
    )

    print(
        "============================================"
    )


    X = artifact["X"]


    # --------------------------------------------------------
    # Validate type
    # --------------------------------------------------------

    if not isinstance(

        X,

        pd.DataFrame

    ):

        raise TypeError(

            f"{dataset_name}: artifact['X'] "
            "must be a pandas DataFrame."

        )


    # --------------------------------------------------------
    # Validate feature count
    # --------------------------------------------------------

    if X.shape[1] != 77:

        raise ValueError(

            f"{dataset_name}: expected 77 "
            f"features, found {X.shape[1]}."

        )


    n_samples = X.shape[0]


    print(
        "Samples:",
        f"{n_samples:,}"
    )

    print(
        "Input features:",
        X.shape[1]
    )

    print(
        "Selected features:",
        len(ordered_indices)
    )

    print(
        "Chunk size:",
        f"{chunk_size:,}"
    )


    # --------------------------------------------------------
    # Allocate final representation
    #
    # float32 is used to substantially reduce memory.
    #
    # N × 8 × 8
    # --------------------------------------------------------

    output = np.empty(

        (

            n_samples,

            GRID_SIZE,

            GRID_SIZE

        ),

        dtype=np.float32

    )


    # --------------------------------------------------------
    # Process chunks
    # --------------------------------------------------------

    total_chunks = (

        n_samples
        + chunk_size
        - 1

    ) // chunk_size


    for chunk_number, start in enumerate(

        range(

            0,

            n_samples,

            chunk_size

        ),

        start=1

    ):

        end = min(

            start + chunk_size,

            n_samples

        )


        # ----------------------------------------------------
        # Extract only required 64 columns
        #
        # Using iloc because ordered_indices are
        # positional indices within X.
        # ----------------------------------------------------

        chunk_df = X.iloc[

            start:end,

            ordered_indices

        ]


        # ----------------------------------------------------
        # Convert to float32
        # ----------------------------------------------------

        chunk_array = chunk_df.to_numpy(

            dtype=np.float32,

            copy=True

        )


        # ----------------------------------------------------
        # Handle invalid numerical values
        # ----------------------------------------------------

        if not np.all(

            np.isfinite(
                chunk_array
            )

        ):

            chunk_array = np.nan_to_num(

                chunk_array,

                nan=0.0,

                posinf=0.0,

                neginf=0.0

            )


        # ----------------------------------------------------
        # Reshape:
        #
        # N × 64
        #       ↓
        # N × 8 × 8
        # ----------------------------------------------------

        chunk_8x8 = chunk_array.reshape(

            -1,

            GRID_SIZE,

            GRID_SIZE

        )


        output[

            start:end

        ] = chunk_8x8


        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if (

            chunk_number == 1

            or

            chunk_number % 10 == 0

            or

            end == n_samples

        ):

            print(

                f"Processed: "
                f"{end:,} / {n_samples:,} "
                f"({100.0 * end / n_samples:.2f}%)"

            )


        # ----------------------------------------------------
        # Free chunk memory
        # ----------------------------------------------------

        del chunk_df

        del chunk_array

        del chunk_8x8


    # ========================================================
    # SAVE REPRESENTATION
    # ========================================================

    output_path = os.path.join(

        output_dir,

        f"devit_v2_{dataset_name}_8x8.npy"

    )


    np.save(

        output_path,

        output

    )


    print(
        "\nSaved:",
        output_path
    )

    print(
        "Shape:",
        output.shape
    )

    print(
        "Dtype:",
        output.dtype
    )


    # ========================================================
    # VALIDATE SAVED FILE
    # ========================================================

    if not os.path.exists(

        output_path

    ):

        raise IOError(

            f"Failed to save {dataset_name} "
            "representation."

        )


    file_size_mb = (

        os.path.getsize(
            output_path
        )
        / (
            1024 ** 2
        )

    )


    print(
        "File size:",
        f"{file_size_mb:.2f} MB"
    )


    # ========================================================
    # CLEANUP
    # ========================================================

    del output

    gc.collect()


    return output_path


# ============================================================
# BUILD TRAIN REPRESENTATION
# ============================================================

TRAIN_8X8_PATH = build_8x8_representation(

    train_artifact,

    "train",

    DEVIT_V2_REPRESENTATION_DIR,

    ordered_X_indices,

    CHUNK_SIZE

)


# ============================================================
# BUILD VALIDATION REPRESENTATION
# ============================================================

VAL_8X8_PATH = build_8x8_representation(

    val_artifact,

    "val",

    DEVIT_V2_REPRESENTATION_DIR,

    ordered_X_indices,

    CHUNK_SIZE

)


# ============================================================
# BUILD TEST REPRESENTATION
# ============================================================

TEST_8X8_PATH = build_8x8_representation(

    test_artifact,

    "test",

    DEVIT_V2_REPRESENTATION_DIR,

    ordered_X_indices,

    CHUNK_SIZE

)


# ============================================================
# SAVE REPRESENTATION CONFIGURATION
# ============================================================

REPRESENTATION_CONFIG_PATH = os.path.join(

    DEVIT_V2_REPRESENTATION_DIR,

    "representation_config.json"

)


representation_config = {

    "method":
        "Correlation-aware 8x8 feature representation",

    "input_features":
        77,

    "selected_features":
        64,

    "grid_rows":
        8,

    "grid_columns":
        8,

    "ordering_source":
        "Part 3.2",

    "ordering_method":
        "absolute Pearson correlation + "
        "average linkage + optimal leaf ordering",

    "correlation_data_source":
        "training set only",

    "dtype":
        "float32",

    "chunk_size":
        CHUNK_SIZE,

    "train_path":
        TRAIN_8X8_PATH,

    "validation_path":
        VAL_8X8_PATH,

    "test_path":
        TEST_8X8_PATH

}


with open(

    REPRESENTATION_CONFIG_PATH,

    "w"

) as f:

    json.dump(

        representation_config,

        f,

        indent=4

    )


# ============================================================
# SAVE TOPOLOGY COPY WITH REPRESENTATIONS
# ============================================================

TOPOLOGY_COPY_PATH = os.path.join(

    DEVIT_V2_REPRESENTATION_DIR,

    "feature_topology_8x8.npy"

)


np.save(

    TOPOLOGY_COPY_PATH,

    topology_8x8

)


# ============================================================
# FINAL VALIDATION
# ============================================================

print(
    "\n============================================"
)

print(
    "VALIDATING SAVED REPRESENTATIONS"
)

print(
    "============================================"
)


for name, path in [

    ("TRAIN", TRAIN_8X8_PATH),

    ("VALIDATION", VAL_8X8_PATH),

    ("TEST", TEST_8X8_PATH)

]:

    if not os.path.exists(path):

        raise FileNotFoundError(

            f"{name} representation not found: "
            f"{path}"

        )


    # Memory-map only for validation
    arr = np.load(

        path,

        mmap_mode="r"

    )


    print(

        f"{name}: "
        f"shape={arr.shape}, "
        f"dtype={arr.dtype}"

    )


    if arr.ndim != 3:

        raise ValueError(

            f"{name} representation "
            "must be 3-dimensional."

        )


    if arr.shape[1:] != (

        GRID_SIZE,

        GRID_SIZE

    ):

        raise ValueError(

            f"{name} representation has "
            f"invalid spatial shape: "
            f"{arr.shape[1:]}"

        )


    if arr.dtype != np.float32:

        raise ValueError(

            f"{name} representation "
            f"must be float32."

        )


    del arr


# ============================================================
# SAVE LABELS
#
# These are copied from the corresponding artifacts.
# ============================================================

print(
    "\n============================================"
)

print(
    "SAVING LABELS"
)

print(
    "============================================"
)


label_paths = {}


for dataset_name, artifact in [

    ("train", train_artifact),

    ("val", val_artifact),

    ("test", test_artifact)

]:

    if "y" not in artifact:

        print(

            f"WARNING: "
            f"{dataset_name} artifact does not "
            "contain 'y'. Labels will not be saved."

        )

        continue


    labels = np.asarray(

        artifact["y"],

        dtype=np.int64

    )


    label_path = os.path.join(

        DEVIT_V2_REPRESENTATION_DIR,

        f"devit_v2_{dataset_name}_labels.npy"

    )


    np.save(

        label_path,

        labels

    )


    label_paths[
        dataset_name
    ] = label_path


    print(

        f"{dataset_name}: "
        f"{labels.shape} → "
        f"{label_path}"

    )


# ============================================================
# SAVE FEATURE ORDER
# ============================================================

ORDERED_FEATURE_NAMES_PATH = os.path.join(

    DEVIT_V2_REPRESENTATION_DIR,

    "ordered_feature_names.json"

)


with open(

    ORDERED_FEATURE_NAMES_PATH,

    "w"

) as f:

    json.dump(

        ordering_info[
            "ordered_feature_names"
        ],

        f,

        indent=4

    )


# ============================================================
# FINAL SUMMARY
# ============================================================

print(
    "\n============================================"
)

print(
    "PART 3.3 COMPLETED"
)

print(
    "============================================"
)

print(
    "Representation:"
)

print(
    "77 flow features"
)

print(
    "      ↓"
)

print(
    "64 MI-selected features"
)

print(
    "      ↓"
)

print(
    "Correlation-aware ordering"
)

print(
    "      ↓"
)

print(
    "8 × 8 representation"
)

print(
    "      ↓"
)

print(
    "Ready for Patch Embedding"
)


print(
    "\nTrain representation:"
)

print(
    TRAIN_8X8_PATH
)


print(
    "\nValidation representation:"
)

print(
    VAL_8X8_PATH
)


print(
    "\nTest representation:"
)

print(
    TEST_8X8_PATH
)


print(
    "\nRepresentation configuration:"
)

print(
    REPRESENTATION_CONFIG_PATH
)


print(
    "\nOutput directory:"
)

print(
    DEVIT_V2_REPRESENTATION_DIR
)


print(
    "\n============================================"
)

In [ ]:
# ============================================================
# PART 3.4
# TEMPORAL-AWARE PATCH EMBEDDING
#
# DeViT V2
#
# Input:
#   Part 3.3
#       N × 8 × 8
#
# Processing:
#       8 × 8 feature topology
#              ↓
#       Patch Embedding
#              ↓
#       Temporal-aware embedding
#
# Output:
#       Saved patch embeddings for
#       train / validation / test
#
# IMPORTANT:
#   - No Transformer training here
#   - No DeViT training here
#   - Uses Part 3.3 artifacts
#   - Processes data in chunks
#   - Saves intermediate artifacts
# ============================================================

import os
import gc
import json
import math

import numpy as np
import torch
import torch.nn as nn


# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42

PATCH_SIZE = 2

IMAGE_SIZE = 8

NUM_PATCHES = (
    IMAGE_SIZE // PATCH_SIZE
) ** 2

EMBED_DIM = 64

CHUNK_SIZE = 8192

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        SEED
    )


# ============================================================
# HEADER
# ============================================================

print(
    "\n============================================"
)

print(
    "PART 3.4"
)

print(
    "TEMPORAL-AWARE PATCH EMBEDDING"
)

print(
    "============================================"
)


print(
    "Device:",
    DEVICE
)

print(
    "Image size:",
    f"{IMAGE_SIZE} × {IMAGE_SIZE}"
)

print(
    "Patch size:",
    f"{PATCH_SIZE} × {PATCH_SIZE}"
)

print(
    "Number of patches:",
    NUM_PATCHES
)

print(
    "Embedding dimension:",
    EMBED_DIM
)


# ============================================================
# CHECK PART 3.3 VARIABLES
# ============================================================

required_variables = [

    "DEVIT_V2_REPRESENTATION_DIR",

    "TRAIN_8X8_PATH",

    "VAL_8X8_PATH",

    "TEST_8X8_PATH"

]


missing_variables = [

    variable

    for variable in required_variables

    if variable not in globals()

]


if len(missing_variables) > 0:

    raise RuntimeError(

        "\nMissing variables from Part 3.3:\n"
        + str(missing_variables)
        + "\n\n"
        "Please run Part 3.3 first."

    )


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

DEVIT_V2_EMBEDDING_DIR = os.path.join(

    os.path.dirname(
        DEVIT_V2_REPRESENTATION_DIR
    ),

    "embeddings"

)


os.makedirs(

    DEVIT_V2_EMBEDDING_DIR,

    exist_ok=True

)


# ============================================================
# PATCH EMBEDDING MODULE
# ============================================================

class TemporalAwarePatchEmbedding(
    nn.Module
):

    """
    Patch embedding for the 8×8 correlation-aware
    feature topology.

    Each 2×2 patch is flattened and projected
    into EMBED_DIM dimensions.

    A learnable temporal-aware positional embedding
    is then added to preserve the ordered patch
    structure.
    """

    def __init__(

        self,

        image_size=8,

        patch_size=2,

        embed_dim=64

    ):

        super().__init__()


        self.image_size = (
            image_size
        )

        self.patch_size = (
            patch_size
        )

        self.embed_dim = (
            embed_dim
        )


        self.num_patches = (

            image_size
            // patch_size

        ) ** 2


        patch_dim = (

            patch_size
            * patch_size

        )


        # ----------------------------------------------------
        # Linear patch projection
        # ----------------------------------------------------

        self.patch_projection = nn.Linear(

            patch_dim,

            embed_dim

        )


        # ----------------------------------------------------
        # Learnable positional-temporal embedding
        #
        # Each spatial patch receives a distinct embedding.
        # This provides ordering information to the Transformer.
        # ----------------------------------------------------

        self.temporal_position_embedding = (

            nn.Parameter(

                torch.zeros(

                    1,

                    self.num_patches,

                    embed_dim

                )

            )

        )


        # ----------------------------------------------------
        # Normalization
        # ----------------------------------------------------

        self.norm = nn.LayerNorm(

            embed_dim

        )


        # ----------------------------------------------------
        # Initialization
        # ----------------------------------------------------

        nn.init.trunc_normal_(

            self.temporal_position_embedding,

            std=0.02

        )


    def forward(

        self,

        x

    ):

        """
        Input:
            x = [B, 8, 8]

        Output:
            embeddings = [B, 16, EMBED_DIM]
        """

        B, H, W = x.shape


        if H != self.image_size:

            raise ValueError(

                f"Expected H={self.image_size}, "
                f"got {H}"

            )


        if W != self.image_size:

            raise ValueError(

                f"Expected W={self.image_size}, "
                f"got {W}"

            )


        # ----------------------------------------------------
        # Split into 2×2 patches
        # ----------------------------------------------------

        patches = x.unfold(

            1,

            self.patch_size,

            self.patch_size

        ).unfold(

            2,

            self.patch_size,

            self.patch_size

        )


        # Current:
        # [B, 4, 4, 2, 2]

        # ----------------------------------------------------
        # Flatten spatial patch positions
        # ----------------------------------------------------

        patches = patches.contiguous().view(

            B,

            self.num_patches,

            -1

        )


        # ----------------------------------------------------
        # Linear projection
        # ----------------------------------------------------

        embeddings = self.patch_projection(

            patches

        )


        # ----------------------------------------------------
        # Add temporal-aware positional information
        # ----------------------------------------------------

        embeddings = (

            embeddings
            + self.temporal_position_embedding

        )


        # ----------------------------------------------------
        # Normalize
        # ----------------------------------------------------

        embeddings = self.norm(

            embeddings

        )


        return embeddings


# ============================================================
# CREATE MODULE
# ============================================================

patch_embedding_model = (

    TemporalAwarePatchEmbedding(

        image_size=IMAGE_SIZE,

        patch_size=PATCH_SIZE,

        embed_dim=EMBED_DIM

    )

).to(

    DEVICE

)


patch_embedding_model.eval()


# ============================================================
# PRINT MODEL
# ============================================================

print(
    "\n============================================"
)

print(
    "PATCH EMBEDDING MODEL"
)

print(
    "============================================"
)

print(
    patch_embedding_model
)


# ============================================================
# FUNCTION
# PROCESS ONE DATASET
# ============================================================

def generate_patch_embeddings(

    input_path,

    dataset_name,

    output_dir,

    model,

    chunk_size=8192

):

    print(
        "\n============================================"
    )

    print(
        f"GENERATING EMBEDDINGS — "
        f"{dataset_name.upper()}"
    )

    print(
        "============================================"
    )


    # --------------------------------------------------------
    # Memory-map input
    # --------------------------------------------------------

    X = np.load(

        input_path,

        mmap_mode="r"

    )


    print(
        "Input shape:",
        X.shape
    )

    print(
        "Input dtype:",
        X.dtype
    )


    # --------------------------------------------------------
    # Validate input
    # --------------------------------------------------------

    if X.ndim != 3:

        raise ValueError(

            f"Expected 3D input, got {X.ndim}D."

        )


    if X.shape[1:] != (

        IMAGE_SIZE,

        IMAGE_SIZE

    ):

        raise ValueError(

            "Expected input shape "
            f"(N, {IMAGE_SIZE}, {IMAGE_SIZE}), "
            f"got {X.shape}"

        )


    n_samples = X.shape[0]


    # --------------------------------------------------------
    # Output path
    # --------------------------------------------------------

    output_path = os.path.join(

        output_dir,

        f"devit_v2_{dataset_name}_patch_embeddings.npy"

    )


    # --------------------------------------------------------
    # Preallocate output using open_memmap
    #
    # Shape:
    #   N × 16 × 64
    # --------------------------------------------------------

    embeddings = np.lib.format.open_memmap(

        output_path,

        mode="w+",

        dtype=np.float32,

        shape=(

            n_samples,

            NUM_PATCHES,

            EMBED_DIM

        )

    )


    # --------------------------------------------------------
    # Processing
    # --------------------------------------------------------

    n_chunks = math.ceil(

        n_samples
        / chunk_size

    )


    with torch.no_grad():

        for chunk_id, start in enumerate(

            range(

                0,

                n_samples,

                chunk_size

            ),

            start=1

        ):

            end = min(

                start + chunk_size,

                n_samples

            )


            # ------------------------------------------------
            # Load chunk
            # ------------------------------------------------

            chunk = np.asarray(

                X[start:end],

                dtype=np.float32

            )


            # ------------------------------------------------
            # Handle invalid values
            # ------------------------------------------------

            if not np.all(

                np.isfinite(
                    chunk
                )

            ):

                chunk = np.nan_to_num(

                    chunk,

                    nan=0.0,

                    posinf=0.0,

                    neginf=0.0

                )


            # ------------------------------------------------
            # Torch conversion
            # ------------------------------------------------

            chunk_tensor = torch.from_numpy(

                chunk

            ).to(

                DEVICE,

                non_blocking=True

            )


            # ------------------------------------------------
            # Patch embedding
            # ------------------------------------------------

            chunk_embedding = (

                model(

                    chunk_tensor

                )

            )


            # ------------------------------------------------
            # Move to CPU
            # ------------------------------------------------

            chunk_embedding = (

                chunk_embedding

                .detach()

                .cpu()

                .numpy()

                .astype(

                    np.float32,

                    copy=False

                )

            )


            # ------------------------------------------------
            # Save chunk
            # ------------------------------------------------

            embeddings[

                start:end

            ] = chunk_embedding


            # ------------------------------------------------
            # Progress
            # ------------------------------------------------

            if (

                chunk_id == 1

                or

                chunk_id % 10 == 0

                or

                end == n_samples

            ):

                print(

                    f"Processed: "
                    f"{end:,} / {n_samples:,} "
                    f"({100.0 * end / n_samples:.2f}%)"

                )


            # ------------------------------------------------
            # Cleanup
            # ------------------------------------------------

            del chunk

            del chunk_tensor

            del chunk_embedding


            if DEVICE.type == "cuda":

                torch.cuda.empty_cache()


    # --------------------------------------------------------
    # Flush memory map
    # --------------------------------------------------------

    embeddings.flush()

    del embeddings

    del X

    gc.collect()


    # --------------------------------------------------------
    # Validate output
    # --------------------------------------------------------

    saved = np.load(

        output_path,

        mmap_mode="r"

    )


    print(
        "\nSaved:",
        output_path
    )

    print(
        "Output shape:",
        saved.shape
    )

    print(
        "Output dtype:",
        saved.dtype
    )


    expected_shape = (

        n_samples,

        NUM_PATCHES,

        EMBED_DIM

    )


    if saved.shape != expected_shape:

        raise ValueError(

            f"Unexpected output shape: "
            f"{saved.shape}; "
            f"expected {expected_shape}"

        )


    del saved

    gc.collect()


    return output_path


# ============================================================
# TRAIN
# ============================================================

TRAIN_PATCH_EMBEDDING_PATH = (

    generate_patch_embeddings(

        TRAIN_8X8_PATH,

        "train",

        DEVIT_V2_EMBEDDING_DIR,

        patch_embedding_model,

        CHUNK_SIZE

    )

)


# ============================================================
# VALIDATION
# ============================================================

VAL_PATCH_EMBEDDING_PATH = (

    generate_patch_embeddings(

        VAL_8X8_PATH,

        "val",

        DEVIT_V2_EMBEDDING_DIR,

        patch_embedding_model,

        CHUNK_SIZE

    )

)


# ============================================================
# TEST
# ============================================================

TEST_PATCH_EMBEDDING_PATH = (

    generate_patch_embeddings(

        TEST_8X8_PATH,

        "test",

        DEVIT_V2_EMBEDDING_DIR,

        patch_embedding_model,

        CHUNK_SIZE

    )

)


# ============================================================
# SAVE PATCH EMBEDDING CONFIG
# ============================================================

PATCH_CONFIG_PATH = os.path.join(

    DEVIT_V2_EMBEDDING_DIR,

    "patch_embedding_config.json"

)


patch_config = {

    "method":
        "Temporal-aware patch embedding",

    "input_shape":
        [8, 8],

    "patch_size":
        [2, 2],

    "num_patches":
        NUM_PATCHES,

    "embedding_dimension":
        EMBED_DIM,

    "output_shape":
        [
            "N",
            NUM_PATCHES,
            EMBED_DIM
        ],

    "dtype":
        "float32",

    "chunk_size":
        CHUNK_SIZE,

    "train_output":
        TRAIN_PATCH_EMBEDDING_PATH,

    "validation_output":
        VAL_PATCH_EMBEDDING_PATH,

    "test_output":
        TEST_PATCH_EMBEDDING_PATH

}


with open(

    PATCH_CONFIG_PATH,

    "w"

) as f:

    json.dump(

        patch_config,

        f,

        indent=4

    )


# ============================================================
# SAVE PATCH EMBEDDING WEIGHTS
#
# IMPORTANT:
#   Part 3.4 embedding layer itself is learned.
#   Save its state so that later stages use exactly
#   the same transformation.
# ============================================================

PATCH_MODEL_PATH = os.path.join(

    DEVIT_V2_EMBEDDING_DIR,

    "temporal_aware_patch_embedding.pt"

)


torch.save(

    {

        "model_state_dict":
            patch_embedding_model.state_dict(),

        "image_size":
            IMAGE_SIZE,

        "patch_size":
            PATCH_SIZE,

        "num_patches":
            NUM_PATCHES,

        "embed_dim":
            EMBED_DIM

    },

    PATCH_MODEL_PATH

)


# ============================================================
# FINAL SUMMARY
# ============================================================

print(
    "\n============================================"
)

print(
    "PART 3.4 COMPLETED"
)

print(
    "============================================"
)

print(
    "Input:"
)

print(
    "N × 8 × 8"
)

print(
    "      ↓"
)

print(
    "2 × 2 Patch Embedding"
)

print(
    "      ↓"
)

print(
    f"{NUM_PATCHES} patches × {EMBED_DIM}-D"
)

print(
    "      ↓"
)

print(
    "Temporal-aware positional embedding"
)

print(
    "      ↓"
)

print(
    "Saved patch embeddings"
)


print(
    "\nTrain:"
)

print(
    TRAIN_PATCH_EMBEDDING_PATH
)


print(
    "\nValidation:"
)

print(
    VAL_PATCH_EMBEDDING_PATH
)


print(
    "\nTest:"
)

print(
    TEST_PATCH_EMBEDDING_PATH
)


print(
    "\nPatch embedding model:"
)

print(
    PATCH_MODEL_PATH
)


print(
    "\nConfiguration:"
)

print(
    PATCH_CONFIG_PATH
)


print(
    "\n============================================"
)

In [ ]:
# ============================================================
# PART 3.5
# TEMPORAL-AWARE DeViT ARCHITECTURE
#
# DeViT V2
#
# Input:
#   N × 16 × 64
#
# Processing:
#   Patch embeddings
#        ↓
#   Temporal-aware Transformer Encoder
#        ↓
#   Deformable Spatial Attention
#        ↓
#   Global representation
#        ↓
#   96-D DeViT representation
#
# IMPORTANT:
#   - No model training here
#   - No data preprocessing here
#   - No feature ordering here
#   - Uses Part 3.4 embedding dimensions
#   - Model is saved for Part 3.6
# ============================================================

import os
import gc
import json
import math

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42

NUM_PATCHES = 16

INPUT_EMBED_DIM = 64

TRANSFORMER_DIM = 96

NUM_HEADS = 8

NUM_LAYERS = 3

FF_DIM = 192

DROPOUT = 0.10

DEFORMABLE_KERNEL_SIZE = 3

OUTPUT_DIM = 96

NUM_CLASSES = 15

DEVICE = torch.device(

    "cuda"
    if torch.cuda.is_available()
    else "cpu"

)


# ============================================================
# REPRODUCIBILITY
# ============================================================

torch.manual_seed(
    SEED
)

np.random.seed(
    SEED
)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        SEED
    )


# ============================================================
# HEADER
# ============================================================

print(
    "\n============================================"
)

print(
    "PART 3.5"
)

print(
    "TEMPORAL-AWARE DeViT ARCHITECTURE"
)

print(
    "============================================"
)


print(
    "Device:",
    DEVICE
)

print(
    "Input patches:",
    NUM_PATCHES
)

print(
    "Input embedding:",
    INPUT_EMBED_DIM
)

print(
    "Transformer dimension:",
    TRANSFORMER_DIM
)

print(
    "Attention heads:",
    NUM_HEADS
)

print(
    "Transformer layers:",
    NUM_LAYERS
)

print(
    "Feed-forward dimension:",
    FF_DIM
)

print(
    "Output representation:",
    OUTPUT_DIM
)


# ============================================================
# CHECK PART 3.4
# ============================================================

required_variables = [

    "DEVIT_V2_EMBEDDING_DIR",

    "TRAIN_PATCH_EMBEDDING_PATH",

    "VAL_PATCH_EMBEDDING_PATH",

    "TEST_PATCH_EMBEDDING_PATH",

    "PATCH_MODEL_PATH"

]


missing_variables = [

    variable

    for variable in required_variables

    if variable not in globals()

]


if len(missing_variables) > 0:

    raise RuntimeError(

        "\nMissing required Part 3.4 variables:\n"
        + str(missing_variables)
        + "\n\n"
        "Please run Part 3.4 first."

    )


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

DEVIT_V2_MODEL_DIR = os.path.join(

    os.path.dirname(
        DEVIT_V2_EMBEDDING_DIR
    ),

    "model"

)


os.makedirs(

    DEVIT_V2_MODEL_DIR,

    exist_ok=True

)


# ============================================================
# DEFORMABLE SPATIAL ATTENTION
# ============================================================

class DeformableSpatialAttention(
    nn.Module
):

    """
    Lightweight deformable spatial attention.

    Input:
        B × N × D

    where:
        N = 16 spatial patches
        D = transformer dimension

    The 16 patches correspond to the 4×4 patch grid
    generated from the original 8×8 feature topology.

    Instead of treating all patch positions equally,
    the module learns attention weights conditioned on
    local spatial context.

    The learned attention offsets are represented by
    an offset predictor over the patch sequence.

    Output:
        B × N × D
    """

    def __init__(

        self,

        embed_dim,

        num_heads=8,

        dropout=0.1

    ):

        super().__init__()


        self.embed_dim = embed_dim

        self.num_heads = num_heads


        # ----------------------------------------------------
        # Standard multi-head attention
        # ----------------------------------------------------

        self.attention = nn.MultiheadAttention(

            embed_dim=embed_dim,

            num_heads=num_heads,

            dropout=dropout,

            batch_first=True

        )


        # ----------------------------------------------------
        # Deformable offset predictor
        #
        # Predicts a scalar spatial offset/gating signal
        # for every patch.
        # ----------------------------------------------------

        self.offset_predictor = nn.Sequential(

            nn.Linear(

                embed_dim,

                embed_dim // 2

            ),

            nn.GELU(),

            nn.Linear(

                embed_dim // 2,

                1

            )

        )


        # ----------------------------------------------------
        # Attention modulation
        # ----------------------------------------------------

        self.attention_gate = nn.Sequential(

            nn.Linear(

                embed_dim,

                embed_dim

            ),

            nn.Sigmoid()

        )


        # ----------------------------------------------------
        # Output projection
        # ----------------------------------------------------

        self.output_projection = nn.Linear(

            embed_dim,

            embed_dim

        )


        self.dropout = nn.Dropout(

            dropout

        )


        self.norm = nn.LayerNorm(

            embed_dim

        )


    def forward(

        self,

        x,

        spatial_mask=None

    ):

        """
        Input:
            x:
                B × N × D

        Output:
            B × N × D
        """

        # ----------------------------------------------------
        # Local deformable offset estimation
        # ----------------------------------------------------

        offsets = self.offset_predictor(

            x

        )


        # ----------------------------------------------------
        # Convert offset to modulation weights
        # ----------------------------------------------------

        offset_weights = torch.sigmoid(

            offsets

        )


        # ----------------------------------------------------
        # Multi-head spatial attention
        # ----------------------------------------------------

        attended, attention_weights = (

            self.attention(

                x,

                x,

                x,

                key_padding_mask=spatial_mask,

                need_weights=True

            )

        )


        # ----------------------------------------------------
        # Deformable modulation
        # ----------------------------------------------------

        attended = (

            attended
            * (
                1.0
                + offset_weights
            )

        )


        # ----------------------------------------------------
        # Feature gate
        # ----------------------------------------------------

        gate = self.attention_gate(

            x

        )


        attended = (

            attended
            * gate

        )


        # ----------------------------------------------------
        # Projection
        # ----------------------------------------------------

        attended = self.output_projection(

            attended

        )


        attended = self.dropout(

            attended

        )


        # ----------------------------------------------------
        # Residual connection
        # ----------------------------------------------------

        output = self.norm(

            x + attended

        )


        return output, attention_weights


# ============================================================
# TRANSFORMER ENCODER BLOCK
# ============================================================

class TemporalAwareTransformerBlock(
    nn.Module
):

    """
    Transformer encoder block with:

        Multi-head self-attention
             ↓
        residual + normalization
             ↓
        feed-forward network
             ↓
        residual + normalization
    """

    def __init__(

        self,

        embed_dim,

        num_heads,

        ff_dim,

        dropout

    ):

        super().__init__()


        self.attention = nn.MultiheadAttention(

            embed_dim=embed_dim,

            num_heads=num_heads,

            dropout=dropout,

            batch_first=True

        )


        self.norm1 = nn.LayerNorm(

            embed_dim

        )


        self.norm2 = nn.LayerNorm(

            embed_dim

        )


        self.ffn = nn.Sequential(

            nn.Linear(

                embed_dim,

                ff_dim

            ),

            nn.GELU(),

            nn.Dropout(

                dropout

            ),

            nn.Linear(

                ff_dim,

                embed_dim

            )

        )


        self.dropout = nn.Dropout(

            dropout

        )


    def forward(

        self,

        x,

        temporal_mask=None

    ):

        # ----------------------------------------------------
        # Self-attention
        # ----------------------------------------------------

        attended, attention_weights = (

            self.attention(

                x,

                x,

                x,

                key_padding_mask=temporal_mask,

                need_weights=True

            )

        )


        # ----------------------------------------------------
        # Residual
        # ----------------------------------------------------

        x = self.norm1(

            x
            + self.dropout(
                attended
            )

        )


        # ----------------------------------------------------
        # Feed-forward
        # ----------------------------------------------------

        ff = self.ffn(

            x

        )


        # ----------------------------------------------------
        # Residual
        # ----------------------------------------------------

        x = self.norm2(

            x
            + self.dropout(
                ff
            )

        )


        return x, attention_weights


# ============================================================
# COMPLETE DeViT MODEL
# ============================================================

class TemporalAwareDeViT(
    nn.Module
):

    """
    Temporal-Aware DeViT.

    Input:
        B × 16 × 64

    Processing:

        64-D patch embeddings
              ↓
        projection to 96-D
              ↓
        learnable temporal positional encoding
              ↓
        Transformer encoder × 3
              ↓
        Deformable spatial attention
              ↓
        global aggregation
              ↓
        96-D representation
              ↓
        classifier
    """

    def __init__(

        self,

        num_patches=16,

        input_dim=64,

        transformer_dim=96,

        num_heads=8,

        num_layers=3,

        ff_dim=192,

        dropout=0.1,

        num_classes=15

    ):

        super().__init__()


        self.num_patches = num_patches

        self.input_dim = input_dim

        self.transformer_dim = transformer_dim


        # ====================================================
        # INPUT PROJECTION
        # ====================================================

        self.input_projection = nn.Sequential(

            nn.Linear(

                input_dim,

                transformer_dim

            ),

            nn.LayerNorm(

                transformer_dim

            )

        )


        # ====================================================
        # TEMPORAL POSITION EMBEDDING
        # ====================================================

        self.temporal_embedding = nn.Parameter(

            torch.zeros(

                1,

                num_patches,

                transformer_dim

            )

        )


        nn.init.trunc_normal_(

            self.temporal_embedding,

            std=0.02

        )


        # ====================================================
        # TRANSFORMER ENCODERS
        # ====================================================

        self.transformer_layers = nn.ModuleList(

            [

                TemporalAwareTransformerBlock(

                    embed_dim=transformer_dim,

                    num_heads=num_heads,

                    ff_dim=ff_dim,

                    dropout=dropout

                )

                for _ in range(

                    num_layers

                )

            ]

        )


        # ====================================================
        # DEFORMABLE SPATIAL ATTENTION
        # ====================================================

        self.deformable_attention = (

            DeformableSpatialAttention(

                embed_dim=transformer_dim,

                num_heads=num_heads,

                dropout=dropout

            )

        )


        # ====================================================
        # REPRESENTATION NORMALIZATION
        # ====================================================

        self.representation_norm = nn.LayerNorm(

            transformer_dim

        )


        # ====================================================
        # REPRESENTATION PROJECTION
        #
        # Keep 96-D output.
        # ====================================================

        self.representation_projection = nn.Sequential(

            nn.Linear(

                transformer_dim,

                OUTPUT_DIM

            ),

            nn.GELU(),

            nn.LayerNorm(

                OUTPUT_DIM

            )

        )


        # ====================================================
        # CLASSIFIER
        # ====================================================

        self.classifier = nn.Linear(

            OUTPUT_DIM,

            num_classes

        )


    def forward(

        self,

        x,

        temporal_mask=None,

        return_representation=False

    ):

        """
        Input:
            x:
                B × 16 × 64

        Output:

            logits:
                B × num_classes

            OR

            representation:
                B × 96
        """

        # ====================================================
        # INPUT PROJECTION
        # ====================================================

        x = self.input_projection(

            x

        )


        # ====================================================
        # TEMPORAL POSITIONAL EMBEDDING
        # ====================================================

        x = (

            x
            + self.temporal_embedding

        )


        # ====================================================
        # TRANSFORMER ENCODER
        # ====================================================

        transformer_attention = []


        for layer in self.transformer_layers:

            x, attention = layer(

                x,

                temporal_mask

            )


            transformer_attention.append(

                attention

            )


        # ====================================================
        # DEFORMABLE SPATIAL ATTENTION
        # ====================================================

        x, deformable_attention = (

            self.deformable_attention(

                x,

                spatial_mask=temporal_mask

            )

        )


        # ====================================================
        # GLOBAL REPRESENTATION
        #
        # Mean pooling over 16 patches.
        # ====================================================

        if temporal_mask is not None:

            valid_mask = (

                ~temporal_mask

            ).float().unsqueeze(

                -1

            )


            pooled = (

                x
                * valid_mask

            ).sum(

                dim=1

            ) / (

                valid_mask.sum(

                    dim=1

                ).clamp(

                    min=1.0

                )

            )

        else:

            pooled = x.mean(

                dim=1

            )


        # ====================================================
        # NORMALIZATION
        # ====================================================

        representation = (

            self.representation_norm(

                pooled

            )

        )


        # ====================================================
        # 96-D REPRESENTATION
        # ====================================================

        representation = (

            self.representation_projection(

                representation

            )

        )


        # ====================================================
        # CLASSIFIER
        # ====================================================

        logits = self.classifier(

            representation

        )


        if return_representation:

            return {

                "logits":
                    logits,

                "representation":
                    representation,

                "transformer_attention":
                    transformer_attention,

                "deformable_attention":
                    deformable_attention

            }


        return logits


# ============================================================
# CREATE MODEL
# ============================================================

model = TemporalAwareDeViT(

    num_patches=NUM_PATCHES,

    input_dim=INPUT_EMBED_DIM,

    transformer_dim=TRANSFORMER_DIM,

    num_heads=NUM_HEADS,

    num_layers=NUM_LAYERS,

    ff_dim=FF_DIM,

    dropout=DROPOUT,

    num_classes=NUM_CLASSES

).to(

    DEVICE

)


# ============================================================
# MODEL SUMMARY
# ============================================================

print(
    "\n============================================"
)

print(
    "DeViT MODEL"
)

print(
    "============================================"
)

print(
    model
)


# ============================================================
# PARAMETER COUNT
# ============================================================

total_parameters = sum(

    parameter.numel()

    for parameter in model.parameters()

)


trainable_parameters = sum(

    parameter.numel()

    for parameter in model.parameters()

    if parameter.requires_grad

)


print(
    "\n============================================"
)

print(
    "MODEL PARAMETERS"
)

print(
    "============================================"
)

print(
    "Total parameters:",
    f"{total_parameters:,}"
)

print(
    "Trainable parameters:",
    f"{trainable_parameters:,}"
)


# ============================================================
# SANITY CHECK
# ============================================================

print(
    "\n============================================"
)

print(
    "SANITY CHECK"
)

print(
    "============================================"
)


dummy_input = torch.randn(

    4,

    NUM_PATCHES,

    INPUT_EMBED_DIM,

    device=DEVICE

)


with torch.no_grad():

    dummy_output = model(

        dummy_input,

        temporal_mask=None,

        return_representation=True

    )


print(
    "Input:",
    dummy_input.shape
)

print(
    "Logits:",
    dummy_output[
        "logits"
    ].shape
)

print(
    "Representation:",
    dummy_output[
        "representation"
    ].shape
)


# ============================================================
# VALIDATE OUTPUT DIMENSION
# ============================================================

if dummy_output[

    "representation"

].shape != (

    4,

    OUTPUT_DIM

):

    raise ValueError(

        "Unexpected DeViT representation shape: "

        + str(

            dummy_output[
                "representation"
            ].shape

        )

    )


if dummy_output[

    "logits"

].shape != (

    4,

    NUM_CLASSES

):

    raise ValueError(

        "Unexpected classifier output shape: "

        + str(

            dummy_output[
                "logits"
            ].shape

        )

    )


print(
    "\nSanity check PASSED."
)


# ============================================================
# SAVE ARCHITECTURE CHECKPOINT
# ============================================================

ARCHITECTURE_PATH = os.path.join(

    DEVIT_V2_MODEL_DIR,

    "devit_v2_architecture.pt"

)


torch.save(

    {

        "model_state_dict":
            model.state_dict(),

        "num_patches":
            NUM_PATCHES,

        "input_embed_dim":
            INPUT_EMBED_DIM,

        "transformer_dim":
            TRANSFORMER_DIM,

        "num_heads":
            NUM_HEADS,

        "num_layers":
            NUM_LAYERS,

        "ff_dim":
            FF_DIM,

        "dropout":
            DROPOUT,

        "output_dim":
            OUTPUT_DIM,

        "num_classes":
            NUM_CLASSES

    },

    ARCHITECTURE_PATH

)


# ============================================================
# SAVE CONFIGURATION
# ============================================================

MODEL_CONFIG_PATH = os.path.join(

    DEVIT_V2_MODEL_DIR,

    "devit_v2_model_config.json"

)


model_config = {

    "architecture":
        "Temporal-Aware DeViT",

    "num_patches":
        NUM_PATCHES,

    "input_embedding_dimension":
        INPUT_EMBED_DIM,

    "transformer_dimension":
        TRANSFORMER_DIM,

    "attention_heads":
        NUM_HEADS,

    "transformer_layers":
        NUM_LAYERS,

    "feed_forward_dimension":
        FF_DIM,

    "dropout":
        DROPOUT,

    "deformable_spatial_attention":
        True,

    "output_representation_dimension":
        OUTPUT_DIM,

    "num_classes":
        NUM_CLASSES,

    "input_shape":
        [
            "N",
            NUM_PATCHES,
            INPUT_EMBED_DIM
        ],

    "representation_shape":
        [
            "N",
            OUTPUT_DIM
        ],

    "classifier_output_shape":
        [
            "N",
            NUM_CLASSES
        ]

}


with open(

    MODEL_CONFIG_PATH,

    "w"

) as f:

    json.dump(

        model_config,

        f,

        indent=4

    )


# ============================================================
# FINAL SUMMARY
# ============================================================

print(
    "\n============================================"
)

print(
    "PART 3.5 COMPLETED"
)

print(
    "============================================"
)

print(
    "Input:"
)

print(
    "N × 16 × 64"
)

print(
    "      ↓"
)

print(
    "Input Projection"
)

print(
    "      ↓"
)

print(
    "Temporal Positional Embedding"
)

print(
    "      ↓"
)

print(
    f"{NUM_LAYERS}-Layer Transformer Encoder"
)

print(
    "      ↓"
)

print(
    "Deformable Spatial Attention"
)

print(
    "      ↓"
)

print(
    "Global Patch Aggregation"
)

print(
    "      ↓"
)

print(
    "96-D DeViT Representation"
)

print(
    "      ↓"
)

print(
    f"{NUM_CLASSES}-Class Classifier"
)


print(
    "\nArchitecture checkpoint:"
)

print(
    ARCHITECTURE_PATH
)


print(
    "\nModel configuration:"
)

print(
    MODEL_CONFIG_PATH
)


print(
    "\n============================================"
)


# ============================================================
# CLEANUP
# ============================================================

del dummy_input
del dummy_output

gc.collect()

if DEVICE.type == "cuda":

    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# PART 3.6
# TRAIN TEMPORAL-AWARE DeViT V2
#
# Main improvements:
#   1. Class-balanced focal loss
#   2. Macro-F1 + Macro Recall model selection
#   3. Checkpoint-based resume
#   4. Early stopping
#   5. Epoch-level checkpointing
#
# Input:
#   Part 3.4 patch embeddings
#
# Model:
#   Part 3.5 TemporalAwareDeViT
#
# Output:
#   Best DeViT checkpoint
# ============================================================

import os
import gc
import json
import time
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)


# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42

NUM_CLASSES = 15

NUM_PATCHES = 16

INPUT_EMBED_DIM = 64

MAX_EPOCHS = 20

EARLY_STOPPING_PATIENCE = 4

BATCH_SIZE = 256

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-4

FOCAL_GAMMA = 2.0

CLASS_BALANCE_BETA = 0.9999

NUM_WORKERS = 2

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# ============================================================
# REPRODUCIBILITY
# ============================================================

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        SEED
    )


# ============================================================
# HEADER
# ============================================================

print(
    "\n============================================"
)

print(
    "PART 3.6"
)

print(
    "TRAIN TEMPORAL-AWARE DeViT V2"
)

print(
    "============================================"
)

print(
    "Device:",
    DEVICE
)

print(
    "Max epochs:",
    MAX_EPOCHS
)

print(
    "Early stopping:",
    EARLY_STOPPING_PATIENCE
)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Learning rate:",
    LEARNING_RATE
)

print(
    "Focal gamma:",
    FOCAL_GAMMA
)

print(
    "Class-balanced beta:",
    CLASS_BALANCE_BETA
)


# ============================================================
# CHECK REQUIRED VARIABLES
# ============================================================

required_variables = [

    "DEVIT_V2_EMBEDDING_DIR",

    "DEVIT_V2_MODEL_DIR",

    "TRAIN_PATCH_EMBEDDING_PATH",

    "VAL_PATCH_EMBEDDING_PATH",

    "TEST_PATCH_EMBEDDING_PATH",

    "PATCH_MODEL_PATH",

    "model"

]


missing_variables = [

    variable

    for variable in required_variables

    if variable not in globals()

]


if len(missing_variables) > 0:

    raise RuntimeError(

        "\nMissing variables:\n"
        + str(missing_variables)
        + "\n\n"
        "Please run Parts 3.4 and 3.5 first."

    )


# ============================================================
# LABEL PATHS
# ============================================================

TRAIN_LABEL_PATH = os.path.join(

    os.path.dirname(
        DEVIT_V2_EMBEDDING_DIR
    ),

    "representations",

    "devit_v2_train_labels.npy"

)

VAL_LABEL_PATH = os.path.join(

    os.path.dirname(
        DEVIT_V2_EMBEDDING_DIR
    ),

    "representations",

    "devit_v2_val_labels.npy"

)

TEST_LABEL_PATH = os.path.join(

    os.path.dirname(
        DEVIT_V2_EMBEDDING_DIR
    ),

    "representations",

    "devit_v2_test_labels.npy"

)


# ============================================================
# CHECK LABEL FILES
# ============================================================

for path in [

    TRAIN_LABEL_PATH,

    VAL_LABEL_PATH,

    TEST_LABEL_PATH

]:

    if not os.path.exists(path):

        raise FileNotFoundError(

            "Label file not found:\n"
            + path

        )


# ============================================================
# CHECKPOINT DIRECTORY
# ============================================================

DEVIT_CHECKPOINT_DIR = os.path.join(

    DEVIT_V2_MODEL_DIR,

    "checkpoints"

)

os.makedirs(

    DEVIT_CHECKPOINT_DIR,

    exist_ok=True

)


LATEST_CHECKPOINT_PATH = os.path.join(

    DEVIT_CHECKPOINT_DIR,

    "devit_v2_latest.pt"

)


BEST_CHECKPOINT_PATH = os.path.join(

    DEVIT_CHECKPOINT_DIR,

    "devit_v2_best.pt"

)


TRAINING_HISTORY_PATH = os.path.join(

    DEVIT_V2_MODEL_DIR,

    "devit_v2_training_history.csv"

)


# ============================================================
# DATASET
# ============================================================

class PatchEmbeddingDataset(
    Dataset
):

    """
    Memory-efficient dataset.

    Input:
        N × 16 × 64

    The .npy file is accessed using memory mapping,
    avoiding loading the complete training set into RAM.
    """

    def __init__(

        self,

        feature_path,

        label_path

    ):

        self.features = np.load(

            feature_path,

            mmap_mode="r"

        )

        self.labels = np.load(

            label_path,

            mmap_mode="r"

        )


        if len(self.features) != len(self.labels):

            raise ValueError(

                "Feature/label size mismatch."

            )


    def __len__(self):

        return len(
            self.labels
        )


    def __getitem__(

        self,

        index

    ):

        x = np.asarray(

            self.features[
                index
            ],

            dtype=np.float32

        )


        y = int(

            self.labels[
                index
            ]

        )


        return (

            torch.from_numpy(x),

            torch.tensor(

                y,

                dtype=torch.long

            )

        )


# ============================================================
# CREATE DATASETS
# ============================================================

print(
    "\n============================================"
)

print(
    "LOADING PATCH EMBEDDINGS"
)

print(
    "============================================"
)


train_dataset = PatchEmbeddingDataset(

    TRAIN_PATCH_EMBEDDING_PATH,

    TRAIN_LABEL_PATH

)


val_dataset = PatchEmbeddingDataset(

    VAL_PATCH_EMBEDDING_PATH,

    VAL_LABEL_PATH

)


test_dataset = PatchEmbeddingDataset(

    TEST_PATCH_EMBEDDING_PATH,

    TEST_LABEL_PATH

)


print(
    "Train samples:",
    f"{len(train_dataset):,}"
)

print(
    "Validation samples:",
    f"{len(val_dataset):,}"
)

print(
    "Test samples:",
    f"{len(test_dataset):,}"
)


# ============================================================
# DATALOADERS
# ============================================================

train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=NUM_WORKERS,

    pin_memory=(
        DEVICE.type == "cuda"
    ),

    drop_last=False

)


val_loader = DataLoader(

    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=(
        DEVICE.type == "cuda"
    ),

    drop_last=False

)


test_loader = DataLoader(

    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=(
        DEVICE.type == "cuda"
    ),

    drop_last=False

)


# ============================================================
# CLASS DISTRIBUTION
# ============================================================

print(
    "\n============================================"
)

print(
    "COMPUTING CLASS-BALANCED WEIGHTS"
)

print(
    "============================================"
)


y_train_for_weights = np.asarray(

    train_dataset.labels

)


class_counts = np.bincount(

    y_train_for_weights,

    minlength=NUM_CLASSES

).astype(
    np.float64
)


print(
    "\nTraining class counts:"
)


for class_id in range(

    NUM_CLASSES

):

    print(

        f"Class {class_id:02d}: "
        f"{int(class_counts[class_id]):,}"

    )


# ============================================================
# CLASS-BALANCED EFFECTIVE NUMBER
#
# weight_i =
#       (1-beta)
#       ----------------
#       1-beta^n_i
#
# ============================================================

effective_weights = np.zeros(

    NUM_CLASSES,

    dtype=np.float64

)


for class_id in range(

    NUM_CLASSES

):

    n_i = class_counts[
        class_id
    ]


    if n_i <= 0:

        effective_weights[
            class_id
        ] = 0.0

    else:

        effective_weights[
            class_id
        ] = (

            (
                1.0
                - CLASS_BALANCE_BETA
            )

            /

            (
                1.0
                -

                CLASS_BALANCE_BETA
                ** n_i

            )

        )


# ============================================================
# NORMALIZE WEIGHTS
# ============================================================

positive_weights = effective_weights[

    effective_weights > 0

]


if len(positive_weights) > 0:

    effective_weights = (

        effective_weights
        /
        positive_weights.mean()

    )


class_weights = torch.tensor(

    effective_weights,

    dtype=torch.float32,

    device=DEVICE

)


print(
    "\nClass-balanced weights:"
)


for class_id in range(

    NUM_CLASSES

):

    print(

        f"Class {class_id:02d}: "
        f"{class_weights[class_id].item():.6f}"

    )


# ============================================================
# CLASS-BALANCED FOCAL LOSS
# ============================================================

class ClassBalancedFocalLoss(
    nn.Module
):

    """
    Class-balanced focal loss.

    CE contribution:
        -log(p_t)

    Focal modulation:
        (1 - p_t)^gamma

    Class-balanced weighting:
        effective class weight

    This is intended to reduce domination by
    the majority BENIGN class while preserving
    attention to difficult/minority classes.
    """

    def __init__(

        self,

        class_weights,

        gamma=2.0

    ):

        super().__init__()


        self.class_weights = class_weights

        self.gamma = gamma


    def forward(

        self,

        logits,

        targets

    ):

        # ----------------------------------------------------
        # Log probabilities
        # ----------------------------------------------------

        log_probs = F.log_softmax(

            logits,

            dim=1

        )


        probs = torch.exp(

            log_probs

        )


        # ----------------------------------------------------
        # Select target class
        # ----------------------------------------------------

        target_log_probs = (

            log_probs.gather(

                1,

                targets.unsqueeze(1)

            ).squeeze(1)

        )


        target_probs = (

            probs.gather(

                1,

                targets.unsqueeze(1)

            ).squeeze(1)

        )


        # ----------------------------------------------------
        # Focal factor
        # ----------------------------------------------------

        focal_factor = (

            1.0
            - target_probs

        ) ** self.gamma


        # ----------------------------------------------------
        # Class-balanced weights
        # ----------------------------------------------------

        sample_weights = (

            self.class_weights[
                targets
            ]

        )


        # ----------------------------------------------------
        # Loss
        # ----------------------------------------------------

        loss = (

            -sample_weights

            * focal_factor

            * target_log_probs

        )


        return loss.mean()


# ============================================================
# IMPORT F
# ============================================================

import torch.nn.functional as F


# ============================================================
# LOSS
# ============================================================

criterion = ClassBalancedFocalLoss(

    class_weights,

    gamma=FOCAL_GAMMA

)


# ============================================================
# OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY

)


# ============================================================
# LR SCHEDULER
# ============================================================

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(

    optimizer,

    mode="max",

    factor=0.5,

    patience=1,

    min_lr=1e-6

)


# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate_model(

    model,

    loader,

    criterion,

    device

):

    model.eval()


    total_loss = 0.0

    total_samples = 0


    all_targets = []

    all_predictions = []


    with torch.no_grad():

        for x, y in loader:

            x = x.to(

                device,

                non_blocking=True

            )

            y = y.to(

                device,

                non_blocking=True

            )


            logits = model(

                x

            )


            loss = criterion(

                logits,

                y

            )


            batch_size = y.size(0)


            total_loss += (

                loss.item()
                * batch_size

            )


            total_samples += batch_size


            predictions = (

                torch.argmax(

                    logits,

                    dim=1

                )

            )


            all_targets.append(

                y.cpu().numpy()

            )


            all_predictions.append(

                predictions.cpu().numpy()

            )


    targets = np.concatenate(

        all_targets

    )


    predictions = np.concatenate(

        all_predictions

    )


    accuracy = accuracy_score(

        targets,

        predictions

    )


    precision, recall, f1, _ = (

        precision_recall_fscore_support(

            targets,

            predictions,

            labels=np.arange(
                NUM_CLASSES
            ),

            average="macro",

            zero_division=0

        )

    )


    return {

        "loss":
            total_loss / total_samples,

        "accuracy":
            accuracy,

        "macro_precision":
            precision,

        "macro_recall":
            recall,

        "macro_f1":
            f1

    }


# ============================================================
# MODEL SELECTION
# ============================================================

def is_better_model(

    current_metrics,

    best_metrics

):

    """
    Primary criterion:
        Macro F1

    Secondary criterion:
        Macro Recall

    This prevents selecting a model solely because
    it aggressively predicts minority classes.
    """

    if best_metrics is None:

        return True


    current_f1 = (

        current_metrics[
            "macro_f1"
        ]

    )

    best_f1 = (

        best_metrics[
            "macro_f1"
        ]

    )


    current_recall = (

        current_metrics[
            "macro_recall"
        ]

    )

    best_recall = (

        best_metrics[
            "macro_recall"
        ]

    )


    # --------------------------------------------------------
    # Primary: Macro F1
    # --------------------------------------------------------

    if current_f1 > (

        best_f1 + 1e-5

    ):

        return True


    # --------------------------------------------------------
    # Secondary: Macro Recall
    # --------------------------------------------------------

    if abs(

        current_f1 - best_f1

    ) <= 1e-5:

        if current_recall > (

            best_recall + 1e-5

        ):

            return True


    return False


# ============================================================
# CHECKPOINT LOADING
# ============================================================

start_epoch = 1

best_metrics = None

best_epoch = 0

no_improvement = 0


if os.path.exists(

    LATEST_CHECKPOINT_PATH

):

    print(
        "\n============================================"
    )

    print(
        "CHECKPOINT FOUND"
    )

    print(
        "============================================"
    )


    checkpoint = torch.load(

        LATEST_CHECKPOINT_PATH,

        map_location=DEVICE

    )


    model.load_state_dict(

        checkpoint[
            "model_state_dict"
        ]

    )


    optimizer.load_state_dict(

        checkpoint[
            "optimizer_state_dict"
        ]

    )


    if (

        "scheduler_state_dict"

        in checkpoint

    ):

        scheduler.load_state_dict(

            checkpoint[
                "scheduler_state_dict"
            ]

        )


    start_epoch = (

        checkpoint[
            "epoch"
        ]
        + 1

    )


    best_metrics = checkpoint.get(

        "best_metrics",

        None

    )


    best_epoch = checkpoint.get(

        "best_epoch",

        0

    )


    no_improvement = checkpoint.get(

        "no_improvement",

        0

    )


    print(
        "Resuming from epoch:",
        start_epoch
    )


    print(
        "Best epoch:",
        best_epoch
    )


    if best_metrics is not None:

        print(

            "Best Macro F1:",

            best_metrics[
                "macro_f1"
            ]

        )


        print(

            "Best Macro Recall:",

            best_metrics[
                "macro_recall"
            ]

        )


# ============================================================
# TRAINING HISTORY
# ============================================================

history = []


if os.path.exists(

    TRAINING_HISTORY_PATH

):

    try:

        history_df = pd.read_csv(

            TRAINING_HISTORY_PATH

        )

        history = history_df.to_dict(

            "records"

        )

        print(

            "\nLoaded previous "
            "training history:",

            len(history),

            "epochs"

        )

    except Exception:

        history = []


# ============================================================
# TRAINING LOOP
# ============================================================

for epoch in range(

    start_epoch,

    MAX_EPOCHS + 1

):

    print(
        "\n"
        + "=" * 44
    )

    print(
        f"EPOCH {epoch} / {MAX_EPOCHS}"
    )

    print(
        "=" * 44
    )


    epoch_start = time.time()


    # ========================================================
    # TRAIN
    # ========================================================

    model.train()


    running_loss = 0.0

    running_samples = 0


    for batch_idx, (

        x,

        y

    ) in enumerate(

        train_loader,

        start=1

    ):

        x = x.to(

            DEVICE,

            non_blocking=True

        )

        y = y.to(

            DEVICE,

            non_blocking=True

        )


        optimizer.zero_grad(

            set_to_none=True

        )


        logits = model(

            x

        )


        loss = criterion(

            logits,

            y

        )


        loss.backward()


        # ----------------------------------------------------
        # Gradient clipping
        # ----------------------------------------------------

        torch.nn.utils.clip_grad_norm_(

            model.parameters(),

            max_norm=1.0

        )


        optimizer.step()


        batch_size = y.size(0)


        running_loss += (

            loss.item()
            * batch_size

        )


        running_samples += batch_size


        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if (

            batch_idx % 250 == 0

            or

            batch_idx == len(train_loader)

        ):

            print(

                f"Batch "
                f"{batch_idx:04d}/"
                f"{len(train_loader):04d} "
                f"| Loss: "
                f"{loss.item():.6f}"

            )


        del x
        del y
        del logits
        del loss


    train_loss = (

        running_loss
        / running_samples

    )


    # ========================================================
    # VALIDATION
    # ========================================================

    val_metrics = evaluate_model(

        model,

        val_loader,

        criterion,

        DEVICE

    )


    # ========================================================
    # LR SCHEDULER
    # ========================================================

    scheduler.step(

        val_metrics[
            "macro_f1"
        ]

    )


    current_lr = (

        optimizer.param_groups[
            0
        ][
            "lr"
        ]

    )


    # ========================================================
    # MODEL SELECTION
    # ========================================================

    improved = is_better_model(

        val_metrics,

        best_metrics

    )


    if improved:

        best_metrics = dict(

            val_metrics

        )

        best_epoch = epoch

        no_improvement = 0


    else:

        no_improvement += 1


    # ========================================================
    # EPOCH SUMMARY
    # ========================================================

    epoch_time = (

        time.time()
        - epoch_start

    )


    print(
        "\n*** Epoch summary ***"
    )


    print(
        "Train loss:",
        train_loss
    )

    print(
        "Val loss:",
        val_metrics[
            "loss"
        ]
    )

    print(
        "Val accuracy:",
        val_metrics[
            "accuracy"
        ]
    )

    print(
        "Val Macro Precision:",
        val_metrics[
            "macro_precision"
        ]
    )

    print(
        "Val Macro Recall:",
        val_metrics[
            "macro_recall"
        ]
    )

    print(
        "Val Macro F1:",
        val_metrics[
            "macro_f1"
        ]
    )


    print(
        "Best Macro F1:",
        best_metrics[
            "macro_f1"
        ]
    )

    print(
        "Best Macro Recall:",
        best_metrics[
            "macro_recall"
        ]
    )

    print(
        "Best epoch:",
        best_epoch
    )

    print(
        "No improvement:",
        f"{no_improvement} / "
        f"{EARLY_STOPPING_PATIENCE}"
    )

    print(
        "Learning rate:",
        current_lr
    )

    print(
        "Epoch time:",
        f"{epoch_time:.2f} seconds"
    )


    # ========================================================
    # SAVE HISTORY
    # ========================================================

    history.append({

        "epoch":
            epoch,

        "train_loss":
            train_loss,

        "val_loss":
            val_metrics[
                "loss"
            ],

        "val_accuracy":
            val_metrics[
                "accuracy"
            ],

        "val_macro_precision":
            val_metrics[
                "macro_precision"
            ],

        "val_macro_recall":
            val_metrics[
                "macro_recall"
            ],

        "val_macro_f1":
            val_metrics[
                "macro_f1"
            ],

        "best_macro_f1":
            best_metrics[
                "macro_f1"
            ],

        "best_macro_recall":
            best_metrics[
                "macro_recall"
            ],

        "best_epoch":
            best_epoch,

        "no_improvement":
            no_improvement,

        "learning_rate":
            current_lr,

        "epoch_time":
            epoch_time

    })


    pd.DataFrame(

        history

    ).to_csv(

        TRAINING_HISTORY_PATH,

        index=False

    )


    # ========================================================
    # SAVE LATEST CHECKPOINT
    # ========================================================

    latest_checkpoint = {

        "epoch":
            epoch,

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scheduler_state_dict":
            scheduler.state_dict(),

        "best_metrics":
            best_metrics,

        "best_epoch":
            best_epoch,

        "no_improvement":
            no_improvement,

        "config": {

            "seed":
                SEED,

            "batch_size":
                BATCH_SIZE,

            "learning_rate":
                LEARNING_RATE,

            "weight_decay":
                WEIGHT_DECAY,

            "focal_gamma":
                FOCAL_GAMMA,

            "class_balance_beta":
                CLASS_BALANCE_BETA,

            "max_epochs":
                MAX_EPOCHS,

            "early_stopping_patience":
                EARLY_STOPPING_PATIENCE

        }

    }


    torch.save(

        latest_checkpoint,

        LATEST_CHECKPOINT_PATH

    )


    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

    if improved:

        torch.save(

            latest_checkpoint,

            BEST_CHECKPOINT_PATH

        )

        print(
            "\n*** BEST MODEL SAVED ***"
        )


    # ========================================================
    # GPU CLEANUP
    # ========================================================

    gc.collect()

    if DEVICE.type == "cuda":

        torch.cuda.empty_cache()


    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if (

        no_improvement
        >= EARLY_STOPPING_PATIENCE

    ):

        print(
            "\n============================================"
        )

        print(
            "EARLY STOPPING"
        )

        print(
            "============================================"
        )

        print(
            "Best epoch:",
            best_epoch
        )

        print(
            "Best Macro F1:",
            best_metrics[
                "macro_f1"
            ]
        )

        print(
            "Best Macro Recall:",
            best_metrics[
                "macro_recall"
            ]
        )

        break


# ============================================================
# LOAD BEST MODEL
# ============================================================

if not os.path.exists(

    BEST_CHECKPOINT_PATH

):

    raise FileNotFoundError(

        "Best DeViT checkpoint was not created."

    )


best_checkpoint = torch.load(

    BEST_CHECKPOINT_PATH,

    map_location=DEVICE

)


model.load_state_dict(

    best_checkpoint[
        "model_state_dict"
    ]

)


model.eval()


# ============================================================
# FINAL SUMMARY
# ============================================================

print(
    "\n============================================"
)

print(
    "PART 3.6 COMPLETED"
)

print(
    "============================================"
)

print(
    "Best epoch:",
    best_checkpoint[
        "best_epoch"
    ]
)

print(
    "Best Macro F1:",
    best_checkpoint[
        "best_metrics"
    ][
        "macro_f1"
    ]
)

print(
    "Best Macro Recall:",
    best_checkpoint[
        "best_metrics"
    ][
        "macro_recall"
    ]
)

print(
    "Best Macro Precision:",
    best_checkpoint[
        "best_metrics"
    ][
        "macro_precision"
    ]
)

print(
    "\nLatest checkpoint:"
)

print(
    LATEST_CHECKPOINT_PATH
)

print(
    "\nBest checkpoint:"
)

print(
    BEST_CHECKPOINT_PATH
)

print(
    "\nTraining history:"
)

print(
    TRAINING_HISTORY_PATH
)

print(
    "\n============================================"
)

## PART 3 DeViT

In [ ]:
# ============================================================
# PART 3.1
# SETUP + LOAD DeViT DATA
#
# No Timestamp required.
# No temporal sequence between flows.
# ============================================================

import os
import gc
import pickle
import random
import time
import copy

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)


# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("Device:", device)


# ============================================================
# PATHS
# ============================================================

OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "My Research/USULAN PENELITIAN DIKTI/2026/"
    "IDS-KN/luaran/outputs"
)


TRAIN_PATH = os.path.join(
    OUTPUT_DIR,
    "train_final.pkl"
)

VAL_PATH = os.path.join(
    OUTPUT_DIR,
    "val_final.pkl"
)

TEST_PATH = os.path.join(
    OUTPUT_DIR,
    "test_final.pkl"
)

FEATURE_INFO_PATH = os.path.join(
    OUTPUT_DIR,
    "feature_information.pkl"
)

IMAGE_MINMAX_PATH = os.path.join(
    OUTPUT_DIR,
    "image_normalization.pkl"
)

LABEL_ENCODER_PATH = os.path.join(
    OUTPUT_DIR,
    "label_encoder.pkl"
)


# ============================================================
# CHECKPOINT DIRECTORY
# ============================================================

CHECKPOINT_DIR = os.path.join(
    OUTPUT_DIR,
    "devit_checkpoints"
)

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)


BEST_MODEL_PATH = os.path.join(
    CHECKPOINT_DIR,
    "best_spatiotemporal_devit.pt"
)

LAST_CHECKPOINT_PATH = os.path.join(
    CHECKPOINT_DIR,
    "last_spatiotemporal_devit.pt"
)

HISTORY_PATH = os.path.join(
    CHECKPOINT_DIR,
    "devit_training_history.csv"
)


# ============================================================
# FINAL FEATURE OUTPUTS
# ============================================================

DEVIT_TRAIN_FEATURE_PATH = os.path.join(
    OUTPUT_DIR,
    "devit_train_features.npy"
)

DEVIT_VAL_FEATURE_PATH = os.path.join(
    OUTPUT_DIR,
    "devit_val_features.npy"
)

DEVIT_TEST_FEATURE_PATH = os.path.join(
    OUTPUT_DIR,
    "devit_test_features.npy"
)

DEVIT_TRAIN_LABEL_PATH = os.path.join(
    OUTPUT_DIR,
    "devit_train_labels.npy"
)

DEVIT_VAL_LABEL_PATH = os.path.join(
    OUTPUT_DIR,
    "devit_val_labels.npy"
)

DEVIT_TEST_LABEL_PATH = os.path.join(
    OUTPUT_DIR,
    "devit_test_labels.npy"
)


# ============================================================
# HYPERPARAMETERS
# ============================================================

IMAGE_SIZE = 8

PATCH_SIZE = 2

EMBED_DIM = 96

NUM_HEADS = 4

NUM_LAYERS = 2

DEFORM_HEADS = 4

DEFORM_POINTS = 4

NUM_CLASSES = 15

DROPOUT = 0.1

BATCH_SIZE = 256

MAX_EPOCHS = 20

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-4

EARLY_STOP_PATIENCE = 4

MAX_CLASS_WEIGHT = 10.0

NUM_WORKERS = 0


print("\n============================================")
print("DeViT CONFIGURATION")
print("============================================")

print("Image size:", IMAGE_SIZE, "x", IMAGE_SIZE)
print("Patch size:", PATCH_SIZE, "x", PATCH_SIZE)
print("Embedding dimension:", EMBED_DIM)
print("Transformer heads:", NUM_HEADS)
print("Transformer layers:", NUM_LAYERS)
print("Deformable heads:", DEFORM_HEADS)
print("Deformable points:", DEFORM_POINTS)
print("Batch size:", BATCH_SIZE)
print("Maximum epochs:", MAX_EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Early stopping patience:", EARLY_STOP_PATIENCE)


# ============================================================
# LOAD FEATURE INFORMATION
# ============================================================

with open(
    FEATURE_INFO_PATH,
    "rb"
) as f:

    feature_information = pickle.load(f)


selected_64 = feature_information[
    "selected_64"
]

class_names = feature_information[
    "class_names"
]

NUM_CLASSES = feature_information[
    "num_classes"
]


print("\n============================================")
print("FEATURE INFORMATION")
print("============================================")

print(
    "Selected features:",
    len(selected_64)
)

print(
    "Number of classes:",
    NUM_CLASSES
)


if len(selected_64) != 64:

    raise ValueError(
        "Expected exactly 64 selected features."
    )


# ============================================================
# LOAD NORMALIZATION
# ============================================================

with open(
    IMAGE_MINMAX_PATH,
    "rb"
) as f:

    image_normalization = pickle.load(f)


train_min = image_normalization[
    "train_min"
]

train_max = image_normalization[
    "train_max"
]


print(
    "Normalization parameters:",
    train_min.shape,
    train_max.shape
)


# ============================================================
# LOAD TRAIN / VAL / TEST
#
# Important:
# Load sequentially enough to avoid unnecessary duplication.
# ============================================================

print("\n============================================")
print("LOADING TRAIN DATA")
print("============================================")


with open(
    TRAIN_PATH,
    "rb"
) as f:

    train_artifact = pickle.load(f)


train_X = train_artifact[
    "X"
]

train_y = np.asarray(
    train_artifact[
        "y"
    ],
    dtype=np.int64
)


del train_artifact

gc.collect()


print(
    "Train:",
    train_X.shape,
    train_y.shape
)


print("\n============================================")
print("LOADING VALIDATION DATA")
print("============================================")


with open(
    VAL_PATH,
    "rb"
) as f:

    val_artifact = pickle.load(f)


val_X = val_artifact[
    "X"
]

val_y = np.asarray(
    val_artifact[
        "y"
    ],
    dtype=np.int64
)


del val_artifact

gc.collect()


print(
    "Validation:",
    val_X.shape,
    val_y.shape
)


print("\n============================================")
print("LOADING TEST DATA")
print("============================================")


with open(
    TEST_PATH,
    "rb"
) as f:

    test_artifact = pickle.load(f)


test_X = test_artifact[
    "X"
]

test_y = np.asarray(
    test_artifact[
        "y"
    ],
    dtype=np.int64
)


del test_artifact

gc.collect()


print(
    "Test:",
    test_X.shape,
    test_y.shape
)


# ============================================================
# SELECT 64 FEATURES
# ============================================================

train_values = (
    train_X[
        selected_64
    ]
    .values
    .astype(np.float32)
)

val_values = (
    val_X[
        selected_64
    ]
    .values
    .astype(np.float32)
)

test_values = (
    test_X[
        selected_64
    ]
    .values
    .astype(np.float32)
)


# Release 77-feature DataFrames
del train_X
del val_X
del test_X

gc.collect()


# ============================================================
# NORMALIZATION
# ============================================================

def normalize_features(
    values,
    train_min,
    train_max
):

    denominator = (
        train_max -
        train_min
    ).copy()

    denominator[
        denominator == 0
    ] = 1.0

    values = (
        values -
        train_min
    ) / denominator

    values = np.clip(
        values,
        0.0,
        1.0
    )

    return values.astype(
        np.float32
    )


train_values = normalize_features(
    train_values,
    train_min,
    train_max
)

val_values = normalize_features(
    val_values,
    train_min,
    train_max
)

test_values = normalize_features(
    test_values,
    train_min,
    train_max
)


# ============================================================
# VERIFY 8x8 REPRESENTATION
# ============================================================

assert (
    train_values.shape[1]
    == 64
)

assert (
    val_values.shape[1]
    == 64
)

assert (
    test_values.shape[1]
    == 64
)


print("\n============================================")
print("IMAGE REPRESENTATION READY")
print("============================================")

print(
    "Train:",
    train_values.shape,
    "->",
    "8 x 8"
)

print(
    "Validation:",
    val_values.shape,
    "->",
    "8 x 8"
)

print(
    "Test:",
    test_values.shape,
    "->",
    "8 x 8"
)

print(
    "Range:",
    train_values.min(),
    "to",
    train_values.max()
)

In [ ]:
# ============================================================
# PART 3.2
# TEMPORAL-AWARE FEATURE MASK
# ============================================================


# ============================================================
# TEMPORAL FEATURE IDENTIFICATION
# ============================================================

TEMPORAL_KEYWORDS = [

    "duration",

    "iat",

    "active",

    "idle",

    "bytes/s",

    "packets/s",

    "rate"

]


def is_temporal_feature(
    feature_name
):

    name = feature_name.lower()

    return any(
        keyword in name
        for keyword in TEMPORAL_KEYWORDS
    )


temporal_flags = np.array(
    [
        is_temporal_feature(
            feature
        )
        for feature in selected_64
    ],
    dtype=np.float32
)


print("\n============================================")
print("TEMPORAL FEATURE ANALYSIS")
print("============================================")


for i, feature in enumerate(
    selected_64
):

    category = (
        "TEMPORAL/DYNAMIC"
        if temporal_flags[i] == 1
        else "GENERAL/FLOW"
    )

    print(
        "{:02d}. {:45s} {}".format(
            i + 1,
            feature,
            category
        )
    )


print(
    "\nTemporal/dynamic features:",
    int(
        temporal_flags.sum()
    ),
    "/ 64"
)


# ============================================================
# 8 x 8 TEMPORAL MASK
# ============================================================

temporal_mask = (
    temporal_flags
    .reshape(
        IMAGE_SIZE,
        IMAGE_SIZE
    )
)


print(
    "\nTemporal mask:"
)

print(
    temporal_mask
)


# ============================================================
# DATASET
# ============================================================

class FlowImageDataset(
    Dataset
):

    def __init__(
        self,
        values,
        labels
    ):

        self.values = values

        self.labels = labels


    def __len__(
        self
    ):

        return len(
            self.values
        )


    def __getitem__(
        self,
        idx
    ):

        image = self.values[
            idx
        ].reshape(
            IMAGE_SIZE,
            IMAGE_SIZE
        )

        label = self.labels[
            idx
        ]

        return (
            torch.from_numpy(
                image
            ).float(),

            torch.tensor(
                label,
                dtype=torch.long
            )
        )


# ============================================================
# CREATE DATASETS
# ============================================================

train_dataset = FlowImageDataset(
    train_values,
    train_y
)

val_dataset = FlowImageDataset(
    val_values,
    val_y
)

test_dataset = FlowImageDataset(
    test_values,
    test_y
)


print("\n============================================")
print("DATASETS")
print("============================================")

print(
    "Train:",
    len(train_dataset)
)

print(
    "Validation:",
    len(val_dataset)
)

print(
    "Test:",
    len(test_dataset)
)


# ============================================================
# DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=NUM_WORKERS,

    pin_memory=torch.cuda.is_available()
)


val_loader = DataLoader(
    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=torch.cuda.is_available()
)


test_loader = DataLoader(
    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=torch.cuda.is_available()
)


# ============================================================
# TEST DATA LOADING
# ============================================================

images, labels = next(
    iter(train_loader)
)


print("\n============================================")
print("DATALOADER TEST")
print("============================================")

print(
    "Images:",
    images.shape
)

print(
    "Labels:",
    labels.shape
)

print(
    "Range:",
    images.min().item(),
    "to",
    images.max().item()
)

In [ ]:
# ============================================================
# PART 3.3
# TEMPORAL-AWARE PATCH EMBEDDING
# ============================================================


class TemporalAwarePatchEmbedding(
    nn.Module
):

    def __init__(
        self,
        image_size=8,
        patch_size=2,
        in_channels=1,
        embed_dim=96
    ):

        super().__init__()


        # ----------------------------------------------------
        # Standard flow-image projection
        # ----------------------------------------------------

        self.feature_proj = nn.Conv2d(

            in_channels,

            embed_dim,

            kernel_size=patch_size,

            stride=patch_size
        )


        # ----------------------------------------------------
        # Temporal mask projection
        #
        # The mask indicates which feature locations
        # correspond to intrinsic temporal/dynamic features.
        # ----------------------------------------------------

        self.temporal_proj = nn.Conv2d(

            1,

            embed_dim,

            kernel_size=patch_size,

            stride=patch_size,

            bias=False
        )


        self.grid_size = (
            image_size //
            patch_size
        )


        self.num_patches = (
            self.grid_size *
            self.grid_size
        )


    def forward(
        self,
        x,
        temporal_mask
    ):

        # Feature representation

        feature_tokens = (
            self.feature_proj(
                x
            )
        )


        # Temporal feature embedding

        temporal_tokens = (
            self.temporal_proj(
                temporal_mask
            )
        )


        # Combine

        tokens = (
            feature_tokens +
            temporal_tokens
        )


        # B,C,H,W
        # ->
        # B,N,C

        tokens = tokens.flatten(
            2
        )

        tokens = tokens.transpose(
            1,
            2
        )


        return tokens


# ============================================================
# TEST PATCH EMBEDDING
# ============================================================

patch_embedding = TemporalAwarePatchEmbedding(
    image_size=IMAGE_SIZE,
    patch_size=PATCH_SIZE,
    in_channels=1,
    embed_dim=EMBED_DIM
).to(device)


dummy_images = images.to(
    device
).unsqueeze(1)


dummy_temporal_mask = torch.tensor(
    temporal_mask,
    dtype=torch.float32,
    device=device
).unsqueeze(0).unsqueeze(0)


with torch.no_grad():

    tokens = patch_embedding(
        dummy_images,
        dummy_temporal_mask
    )


print("\n============================================")
print("PATCH EMBEDDING TEST")
print("============================================")

print(
    "Input:",
    dummy_images.shape
)

print(
    "Tokens:",
    tokens.shape
)

print(
    "Number of patches:",
    patch_embedding.num_patches
)

In [ ]:
# ============================================================
# PART 3.4
# DEFORMABLE SPATIAL ATTENTION
# ============================================================


class DeformableSpatialAttention(
    nn.Module
):

    def __init__(
        self,
        dim,
        heads=4,
        num_points=4
    ):

        super().__init__()


        assert (
            dim % heads == 0
        )


        self.dim = dim

        self.heads = heads

        self.head_dim = (
            dim //
            heads
        )

        self.num_points = (
            num_points
        )


        # ----------------------------------------------------
        # Query
        # ----------------------------------------------------

        self.q_proj = nn.Linear(
            dim,
            dim
        )


        # ----------------------------------------------------
        # Adaptive offset generator
        # ----------------------------------------------------

        self.offset_generator = nn.Sequential(

            nn.Linear(
                dim,
                dim
            ),

            nn.GELU(),

            nn.Linear(
                dim,
                heads *
                num_points *
                2
            )
        )


        # ----------------------------------------------------
        # Output projection
        # ----------------------------------------------------

        self.out_proj = nn.Linear(
            dim,
            dim
        )


    def forward(
        self,
        x,
        temporal_context=None
    ):

        # x:
        # B,H,W,C

        B, H, W, C = x.shape


        # ----------------------------------------------------
        # Query
        # ----------------------------------------------------

        q = self.q_proj(
            x
        )


        # ----------------------------------------------------
        # Temporal conditioning
        #
        # Temporal context modifies the adaptive offsets.
        # ----------------------------------------------------

        if (
            temporal_context
            is not None
        ):

            context = (
                temporal_context
                .unsqueeze(1)
                .unsqueeze(1)
                .expand(
                    -1,
                    H,
                    W,
                    -1
                )
            )


            offset_input = (
                x +
                context
            )

        else:

            offset_input = x


        offsets = (
            self.offset_generator(
                offset_input
            )
        )


        offsets = offsets.view(

            B,
            H,
            W,
            self.heads,
            self.num_points,
            2
        )


        # ----------------------------------------------------
        # Base grid
        # ----------------------------------------------------

        yy, xx = torch.meshgrid(

            torch.linspace(
                -1,
                1,
                H,
                device=x.device
            ),

            torch.linspace(
                -1,
                1,
                W,
                device=x.device
            ),

            indexing="ij"
        )


        base_grid = torch.stack(
            [
                xx,
                yy
            ],
            dim=-1
        )


        base_grid = (
            base_grid
            .unsqueeze(0)
            .expand(
                B,
                -1,
                -1,
                -1
            )
        )


        x_chw = x.permute(
            0,
            3,
            1,
            2
        )


        q = q.view(

            B,
            H,
            W,
            self.heads,
            self.head_dim
        )


        output_heads = []


        # ====================================================
        # HEAD LOOP
        # ====================================================

        for h in range(
            self.heads
        ):

            head_q = q[
                ...,
                h,
                :
            ]


            head_outputs = []


            for p in range(
                self.num_points
            ):

                offset = offsets[
                    ...,
                    h,
                    p,
                    :
                ]


                grid = (
                    base_grid +
                    torch.tanh(
                        offset
                    )
                )


                sampled = F.grid_sample(

                    x_chw,

                    grid,

                    mode="bilinear",

                    padding_mode="border",

                    align_corners=True
                )


                sampled = sampled.permute(
                    0,
                    2,
                    3,
                    1
                )


                sampled = sampled.view(

                    B,
                    H,
                    W,
                    self.heads,
                    self.head_dim

                )[
                    ...,
                    h,
                    :
                ]


                # ------------------------------------------------
                # Query-sampled feature similarity
                # ------------------------------------------------

                score = (
                    head_q *
                    sampled
                ).sum(
                    dim=-1
                )


                score = (
                    score /
                    np.sqrt(
                        self.head_dim
                    )
                )


                head_outputs.append(
                    (
                        score,
                        sampled
                    )
                )


            scores = torch.stack(
                [
                    item[0]
                    for item in head_outputs
                ],
                dim=-1
            )


            values = torch.stack(
                [
                    item[1]
                    for item in head_outputs
                ],
                dim=-2
            )


            attention = F.softmax(
                scores,
                dim=-1
            )


            head_output = (
                attention.unsqueeze(-1)
                *
                values
            ).sum(
                dim=-2
            )


            output_heads.append(
                head_output
            )


        output = torch.stack(
            output_heads,
            dim=3
        )


        output = output.reshape(

            B,
            H,
            W,
            C
        )


        return self.out_proj(
            output
        )


# ============================================================
# TEST
# ============================================================

deform_attn = DeformableSpatialAttention(

    dim=EMBED_DIM,

    heads=DEFORM_HEADS,

    num_points=DEFORM_POINTS

).to(device)


dummy_spatial = torch.randn(

    images.size(0),

    4,

    4,

    EMBED_DIM,

    device=device
)


dummy_context = torch.randn(

    images.size(0),

    EMBED_DIM,

    device=device
)


with torch.no_grad():

    deform_output = deform_attn(

        dummy_spatial,

        temporal_context=
            dummy_context
    )


print("\n============================================")
print("DEFORMABLE ATTENTION TEST")
print("============================================")

print(
    "Input:",
    dummy_spatial.shape
)

print(
    "Output:",
    deform_output.shape
)

print(
    "Output mean:",
    deform_output.mean().item()
)

print(
    "Output std:",
    deform_output.std().item()
)

In [ ]:
# ============================================================
# PART 3.4
# DEFORMABLE SPATIAL ATTENTION
# ============================================================


class DeformableSpatialAttention(
    nn.Module
):

    def __init__(
        self,
        dim,
        heads=4,
        num_points=4
    ):

        super().__init__()


        assert (
            dim % heads == 0
        )


        self.dim = dim

        self.heads = heads

        self.head_dim = (
            dim //
            heads
        )

        self.num_points = (
            num_points
        )


        # ----------------------------------------------------
        # Query
        # ----------------------------------------------------

        self.q_proj = nn.Linear(
            dim,
            dim
        )


        # ----------------------------------------------------
        # Adaptive offset generator
        # ----------------------------------------------------

        self.offset_generator = nn.Sequential(

            nn.Linear(
                dim,
                dim
            ),

            nn.GELU(),

            nn.Linear(
                dim,
                heads *
                num_points *
                2
            )
        )


        # ----------------------------------------------------
        # Output projection
        # ----------------------------------------------------

        self.out_proj = nn.Linear(
            dim,
            dim
        )


    def forward(
        self,
        x,
        temporal_context=None
    ):

        # x:
        # B,H,W,C

        B, H, W, C = x.shape


        # ----------------------------------------------------
        # Query
        # ----------------------------------------------------

        q = self.q_proj(
            x
        )


        # ----------------------------------------------------
        # Temporal conditioning
        #
        # Temporal context modifies the adaptive offsets.
        # ----------------------------------------------------

        if (
            temporal_context
            is not None
        ):

            context = (
                temporal_context
                .unsqueeze(1)
                .unsqueeze(1)
                .expand(
                    -1,
                    H,
                    W,
                    -1
                )
            )


            offset_input = (
                x +
                context
            )

        else:

            offset_input = x


        offsets = (
            self.offset_generator(
                offset_input
            )
        )


        offsets = offsets.view(

            B,
            H,
            W,
            self.heads,
            self.num_points,
            2
        )


        # ----------------------------------------------------
        # Base grid
        # ----------------------------------------------------

        yy, xx = torch.meshgrid(

            torch.linspace(
                -1,
                1,
                H,
                device=x.device
            ),

            torch.linspace(
                -1,
                1,
                W,
                device=x.device
            ),

            indexing="ij"
        )


        base_grid = torch.stack(
            [
                xx,
                yy
            ],
            dim=-1
        )


        base_grid = (
            base_grid
            .unsqueeze(0)
            .expand(
                B,
                -1,
                -1,
                -1
            )
        )


        x_chw = x.permute(
            0,
            3,
            1,
            2
        )


        q = q.view(

            B,
            H,
            W,
            self.heads,
            self.head_dim
        )


        output_heads = []


        # ====================================================
        # HEAD LOOP
        # ====================================================

        for h in range(
            self.heads
        ):

            head_q = q[
                ...,
                h,
                :
            ]


            head_outputs = []


            for p in range(
                self.num_points
            ):

                offset = offsets[
                    ...,
                    h,
                    p,
                    :
                ]


                grid = (
                    base_grid +
                    torch.tanh(
                        offset
                    )
                )


                sampled = F.grid_sample(

                    x_chw,

                    grid,

                    mode="bilinear",

                    padding_mode="border",

                    align_corners=True
                )


                sampled = sampled.permute(
                    0,
                    2,
                    3,
                    1
                )


                sampled = sampled.view(

                    B,
                    H,
                    W,
                    self.heads,
                    self.head_dim

                )[
                    ...,
                    h,
                    :
                ]


                # ------------------------------------------------
                # Query-sampled feature similarity
                # ------------------------------------------------

                score = (
                    head_q *
                    sampled
                ).sum(
                    dim=-1
                )


                score = (
                    score /
                    np.sqrt(
                        self.head_dim
                    )
                )


                head_outputs.append(
                    (
                        score,
                        sampled
                    )
                )


            scores = torch.stack(
                [
                    item[0]
                    for item in head_outputs
                ],
                dim=-1
            )


            values = torch.stack(
                [
                    item[1]
                    for item in head_outputs
                ],
                dim=-2
            )


            attention = F.softmax(
                scores,
                dim=-1
            )


            head_output = (
                attention.unsqueeze(-1)
                *
                values
            ).sum(
                dim=-2
            )


            output_heads.append(
                head_output
            )


        output = torch.stack(
            output_heads,
            dim=3
        )


        output = output.reshape(

            B,
            H,
            W,
            C
        )


        return self.out_proj(
            output
        )


# ============================================================
# TEST
# ============================================================

deform_attn = DeformableSpatialAttention(

    dim=EMBED_DIM,

    heads=DEFORM_HEADS,

    num_points=DEFORM_POINTS

).to(device)


dummy_spatial = torch.randn(

    images.size(0),

    4,

    4,

    EMBED_DIM,

    device=device
)


dummy_context = torch.randn(

    images.size(0),

    EMBED_DIM,

    device=device
)


with torch.no_grad():

    deform_output = deform_attn(

        dummy_spatial,

        temporal_context=
            dummy_context
    )


print("\n============================================")
print("DEFORMABLE ATTENTION TEST")
print("============================================")

print(
    "Input:",
    dummy_spatial.shape
)

print(
    "Output:",
    deform_output.shape
)

print(
    "Output mean:",
    deform_output.mean().item()
)

print(
    "Output std:",
    deform_output.std().item()
)

In [ ]:
# ============================================================
# PART 3.5
# TEMPORAL-AWARE DEFORMABLE DeViT
# ============================================================


class TemporalAwareDeViT(
    nn.Module
):

    def __init__(
        self,
        image_size=8,
        patch_size=2,
        in_channels=1,
        embed_dim=96,
        num_heads=4,
        num_layers=2,
        deform_heads=4,
        deform_points=4,
        num_classes=15,
        dropout=0.1
    ):

        super().__init__()


        self.embed_dim = embed_dim


        # ----------------------------------------------------
        # Temporal-aware patch embedding
        # ----------------------------------------------------

        self.patch_embedding = (
            TemporalAwarePatchEmbedding(

                image_size=image_size,

                patch_size=patch_size,

                in_channels=in_channels,

                embed_dim=embed_dim
            )
        )


        self.grid_size = (
            image_size //
            patch_size
        )


        self.num_patches = (
            self.grid_size *
            self.grid_size
        )


        # ----------------------------------------------------
        # Positional embedding
        # ----------------------------------------------------

        self.pos_embedding = nn.Parameter(

            torch.zeros(

                1,

                self.num_patches,

                embed_dim
            )
        )


        # ----------------------------------------------------
        # Transformer
        # ----------------------------------------------------

        encoder_layer = (
            nn.TransformerEncoderLayer(

                d_model=embed_dim,

                nhead=num_heads,

                dim_feedforward=
                    embed_dim * 4,

                dropout=dropout,

                activation="gelu",

                batch_first=True,

                norm_first=True
            )
        )


        self.encoder = (
            nn.TransformerEncoder(

                encoder_layer,

                num_layers=num_layers
            )
        )


        self.norm = nn.LayerNorm(
            embed_dim
        )


        # ----------------------------------------------------
        # Temporal context projection
        #
        # Aggregates temporal-aware token information.
        # ----------------------------------------------------

        self.temporal_context_proj = nn.Sequential(

            nn.Linear(
                embed_dim,
                embed_dim
            ),

            nn.GELU(),

            nn.Linear(
                embed_dim,
                embed_dim
            )
        )


        # ----------------------------------------------------
        # Deformable spatial attention
        # ----------------------------------------------------

        self.deformable_attention = (

            DeformableSpatialAttention(

                dim=embed_dim,

                heads=deform_heads,

                num_points=deform_points
            )
        )


        # ----------------------------------------------------
        # Classification
        # ----------------------------------------------------

        self.classifier = nn.Linear(

            embed_dim,

            num_classes
        )


        # ----------------------------------------------------
        # Initialization
        # ----------------------------------------------------

        nn.init.normal_(
            self.pos_embedding,
            std=0.02
        )


    def forward(
        self,
        x,
        temporal_mask,
        return_features=False
    ):

        # ----------------------------------------------------
        # x:
        #
        # B, 8, 8
        # ----------------------------------------------------

        x = x.unsqueeze(1)


        # ----------------------------------------------------
        # Temporal mask
        # ----------------------------------------------------

        B = x.size(0)


        mask = temporal_mask.expand(

            B,

            -1,

            -1,

            -1
        )


        # ----------------------------------------------------
        # Patch embedding
        # ----------------------------------------------------

        tokens = self.patch_embedding(

            x,

            mask
        )


        # ----------------------------------------------------
        # Positional embedding
        # ----------------------------------------------------

        tokens = (
            tokens +
            self.pos_embedding
        )


        # ----------------------------------------------------
        # Transformer
        # ----------------------------------------------------

        tokens = self.encoder(
            tokens
        )


        tokens = self.norm(
            tokens
        )


        # ----------------------------------------------------
        # Temporal context
        #
        # Temporal patches receive greater contribution
        # based on the temporal feature mask.
        # ----------------------------------------------------

        temporal_scores = (
            mask
            .mean(
                dim=1
            )
        )


        temporal_scores = F.avg_pool2d(

            temporal_scores,

            kernel_size=PATCH_SIZE,

            stride=PATCH_SIZE
        )


        temporal_scores = (
            temporal_scores
            .flatten(
                1
            )
        )


        temporal_scores = (
            temporal_scores /
            (
                temporal_scores.sum(
                    dim=1,
                    keepdim=True
                ) + 1e-6
            )
        )


        temporal_context = (
            tokens *
            temporal_scores.unsqueeze(-1)
        ).sum(
            dim=1
        )


        temporal_context = (
            self.temporal_context_proj(
                temporal_context
            )
        )


        # ----------------------------------------------------
        # Convert tokens to spatial map
        # ----------------------------------------------------

        spatial = tokens.reshape(

            B,

            self.grid_size,

            self.grid_size,

            self.embed_dim
        )


        # ----------------------------------------------------
        # Deformable spatial attention
        #
        # conditioned by temporal/dynamic characteristics.
        # ----------------------------------------------------

        enhanced_spatial = (
            self.deformable_attention(

                spatial,

                temporal_context=
                    temporal_context
            )
        )


        # ----------------------------------------------------
        # Global representation
        # ----------------------------------------------------

        features = self.norm(

            enhanced_spatial.mean(
                dim=(1, 2)
            )
        )


        # ----------------------------------------------------
        # Classification
        # ----------------------------------------------------

        logits = self.classifier(
            features
        )


        if return_features:

            return (
                logits,
                features
            )


        return logits


# ============================================================
# CREATE MODEL
# ============================================================

model = TemporalAwareDeViT(

    image_size=IMAGE_SIZE,

    patch_size=PATCH_SIZE,

    in_channels=1,

    embed_dim=EMBED_DIM,

    num_heads=NUM_HEADS,

    num_layers=NUM_LAYERS,

    deform_heads=DEFORM_HEADS,

    deform_points=DEFORM_POINTS,

    num_classes=NUM_CLASSES,

    dropout=DROPOUT

).to(device)


num_parameters = sum(

    p.numel()

    for p in model.parameters()

    if p.requires_grad
)


print("\n============================================")
print("DeViT MODEL")
print("============================================")

print(
    "Trainable parameters:",
    num_parameters
)


# ============================================================
# TEST MODEL
# ============================================================

dummy_mask = torch.tensor(

    temporal_mask,

    dtype=torch.float32,

    device=device

).unsqueeze(0).unsqueeze(0)


with torch.no_grad():

    logits, features = model(

        images.to(device),

        dummy_mask,

        return_features=True
    )


print(
    "Input:",
    images.shape
)

print(
    "Logits:",
    logits.shape
)

print(
    "Features:",
    features.shape
)

In [ ]:
# ============================================================
# PART 3.6
# LOSS + OPTIMIZER + SCHEDULER
# ============================================================


# ============================================================
# CLASS WEIGHTS
# ============================================================

class_counts = np.bincount(

    train_y,

    minlength=NUM_CLASSES
)


class_weights_np = (

    len(train_y)

    /

    (
        NUM_CLASSES *
        class_counts
    )
)


class_weights_np = np.asarray(

    class_weights_np,

    dtype=np.float32
)


class_weights_np = np.clip(

    class_weights_np,

    0.0,

    MAX_CLASS_WEIGHT
)


class_weights = torch.tensor(

    class_weights_np,

    dtype=torch.float32,

    device=device
)


print("\n============================================")
print("CLASS WEIGHTS")
print("============================================")


for i, weight in enumerate(
    class_weights
):

    print(

        "{:02d} {:35s} {:.4f}".format(

            i,

            class_names[i],

            weight.item()
        )
    )


# ============================================================
# LOSS
# ============================================================

criterion = nn.CrossEntropyLoss(

    weight=class_weights
)


# ============================================================
# OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY
)


# ============================================================
# SCHEDULER
# ============================================================

scheduler = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(

        optimizer,

        mode="max",

        factor=0.5,

        patience=2
    )
)


print(
    "\nLoss, optimizer, scheduler: READY"
)

In [ ]:
# ============================================================
# PART 3.7
# TRAINING WITH CHECKPOINT RESUME
#
# Maximum: 20 epochs
# Early stopping: validation Macro Recall
# ============================================================


def train_one_epoch(

    model,

    loader,

    criterion,

    optimizer,

    device,

    temporal_mask
):

    model.train()


    running_loss = 0.0

    total_samples = 0


    for batch_idx, (
        images,
        labels
    ) in enumerate(loader):


        images = images.to(

            device,

            non_blocking=True
        )


        labels = labels.to(

            device,

            non_blocking=True
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        logits = model(

            images,

            temporal_mask,

            return_features=False
        )


        loss = criterion(

            logits,

            labels
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(

            model.parameters(),

            max_norm=1.0
        )


        optimizer.step()


        batch_size = labels.size(0)


        running_loss += (

            loss.item() *
            batch_size
        )


        total_samples += batch_size


        if (
            (batch_idx + 1) % 250 == 0
        ):

            print(

                "Batch {:04d}/{:04d} | "
                "Loss: {:.6f}".format(

                    batch_idx + 1,

                    len(loader),

                    loss.item()
                )
            )


    return (

        running_loss /
        total_samples
    )


# ============================================================
# VALIDATION
# ============================================================

def evaluate_devit(

    model,

    loader,

    criterion,

    device,

    temporal_mask
):

    model.eval()


    total_loss = 0.0

    total_samples = 0


    predictions = []

    targets = []


    with torch.no_grad():

        for images, labels in loader:


            images = images.to(

                device,

                non_blocking=True
            )


            labels_device = labels.to(

                device,

                non_blocking=True
            )


            logits = model(

                images,

                temporal_mask,

                return_features=False
            )


            loss = criterion(

                logits,

                labels_device
            )


            batch_size = labels.size(0)


            total_loss += (

                loss.item() *
                batch_size
            )


            total_samples += batch_size


            pred = torch.argmax(

                logits,

                dim=1
            )


            predictions.extend(

                pred.cpu().numpy()
            )


            targets.extend(

                labels.numpy()
            )


    predictions = np.asarray(
        predictions
    )


    targets = np.asarray(
        targets
    )


    metrics = {

        "loss":
            total_loss /
            total_samples,

        "accuracy":
            accuracy_score(
                targets,
                predictions
            ),

        "macro_precision":
            precision_score(
                targets,
                predictions,
                average="macro",
                zero_division=0
            ),

        "macro_recall":
            recall_score(
                targets,
                predictions,
                average="macro",
                zero_division=0
            ),

        "macro_f1":
            f1_score(
                targets,
                predictions,
                average="macro",
                zero_division=0
            )
    }


    return (

        metrics,

        targets,

        predictions
    )


# ============================================================
# TEMPORAL MASK TENSOR
# ============================================================

temporal_mask_tensor = torch.tensor(

    temporal_mask,

    dtype=torch.float32,

    device=device

).unsqueeze(0).unsqueeze(0)


# ============================================================
# CHECK FOR EXISTING CHECKPOINT
# ============================================================

start_epoch = 1

best_val_recall = -np.inf

best_epoch = 0

epochs_without_improvement = 0

history = []


if os.path.exists(
    LAST_CHECKPOINT_PATH
):

    print("\n============================================")
    print("CHECKPOINT FOUND")
    print("============================================")

    checkpoint = torch.load(

        LAST_CHECKPOINT_PATH,

        map_location=device
    )


    model.load_state_dict(

        checkpoint[
            "model_state_dict"
        ]
    )


    optimizer.load_state_dict(

        checkpoint[
            "optimizer_state_dict"
        ]
    )


    scheduler.load_state_dict(

        checkpoint[
            "scheduler_state_dict"
        ]
    )


    start_epoch = (

        checkpoint[
            "epoch"
        ] + 1
    )


    best_val_recall = checkpoint.get(

        "best_val_macro_recall",

        -np.inf
    )


    best_epoch = checkpoint.get(

        "best_epoch",

        0
    )


    epochs_without_improvement = (

        checkpoint.get(

            "epochs_without_improvement",

            0
        )
    )


    history = checkpoint.get(

        "history",

        []
    )


    print(

        "Resuming from epoch:",

        start_epoch
    )


    print(

        "Best validation Macro Recall:",

        best_val_recall
    )


else:

    print("\n============================================")
    print("NO CHECKPOINT FOUND")
    print("STARTING FROM EPOCH 1")
    print("============================================")


# ============================================================
# TRAINING LOOP
# ============================================================

for epoch in range(

    start_epoch,

    MAX_EPOCHS + 1
):


    start_time = time.time()


    print("\n============================================")

    print(

        "EPOCH",

        epoch,

        "/",

        MAX_EPOCHS
    )

    print("============================================")


    train_loss = train_one_epoch(

        model,

        train_loader,

        criterion,

        optimizer,

        device,

        temporal_mask_tensor
    )


    val_metrics, _, _ = evaluate_devit(

        model,

        val_loader,

        criterion,

        device,

        temporal_mask_tensor
    )


    scheduler.step(

        val_metrics[
            "macro_recall"
        ]
    )


    epoch_time = (

        time.time() -
        start_time
    )


    current_lr = (

        optimizer
        .param_groups[0]["lr"]
    )


    # --------------------------------------------------------
    # Improvement?
    # --------------------------------------------------------

    improved = (

        val_metrics[
            "macro_recall"
        ]

        >

        best_val_recall
    )


    if improved:

        best_val_recall = (

            val_metrics[
                "macro_recall"
            ]
        )

        best_epoch = epoch

        epochs_without_improvement = 0

    else:

        epochs_without_improvement += 1


    # --------------------------------------------------------
    # History
    # --------------------------------------------------------

    history.append({

        "epoch":
            epoch,

        "train_loss":
            train_loss,

        "val_loss":
            val_metrics["loss"],

        "val_accuracy":
            val_metrics["accuracy"],

        "val_macro_precision":
            val_metrics["macro_precision"],

        "val_macro_recall":
            val_metrics["macro_recall"],

        "val_macro_f1":
            val_metrics["macro_f1"],

        "learning_rate":
            current_lr,

        "time_seconds":
            epoch_time
    })


    # --------------------------------------------------------
    # SAVE LAST CHECKPOINT
    #
    # This happens EVERY epoch.
    # --------------------------------------------------------

    last_checkpoint = {

        "epoch":
            epoch,

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scheduler_state_dict":
            scheduler.state_dict(),

        "best_val_macro_recall":
            best_val_recall,

        "best_epoch":
            best_epoch,

        "epochs_without_improvement":
            epochs_without_improvement,

        "history":
            history,

        "config": {

            "image_size":
                IMAGE_SIZE,

            "patch_size":
                PATCH_SIZE,

            "embed_dim":
                EMBED_DIM,

            "num_heads":
                NUM_HEADS,

            "num_layers":
                NUM_LAYERS,

            "deform_heads":
                DEFORM_HEADS,

            "deform_points":
                DEFORM_POINTS,

            "batch_size":
                BATCH_SIZE,

            "max_epochs":
                MAX_EPOCHS,

            "learning_rate":
                LEARNING_RATE
        }
    }


    torch.save(

        last_checkpoint,

        LAST_CHECKPOINT_PATH
    )


    # --------------------------------------------------------
    # SAVE BEST MODEL
    # --------------------------------------------------------

    if improved:

        best_checkpoint = copy.deepcopy(
            last_checkpoint
        )


        torch.save(

            best_checkpoint,

            BEST_MODEL_PATH
        )


        print(
            "\n*** BEST MODEL SAVED ***"
        )


    # --------------------------------------------------------
    # SAVE HISTORY
    # --------------------------------------------------------

    pd.DataFrame(
        history
    ).to_csv(

        HISTORY_PATH,

        index=False
    )


    # --------------------------------------------------------
    # SUMMARY
    # --------------------------------------------------------

    print("\nEpoch summary")

    print(
        "Train loss:",
        train_loss
    )

    print(
        "Val loss:",
        val_metrics["loss"]
    )

    print(
        "Val accuracy:",
        val_metrics["accuracy"]
    )

    print(
        "Val Macro Precision:",
        val_metrics[
            "macro_precision"
        ]
    )

    print(
        "Val Macro Recall:",
        val_metrics[
            "macro_recall"
        ]
    )

    print(
        "Val Macro F1:",
        val_metrics[
            "macro_f1"
        ]
    )

    print(
        "Best Macro Recall:",
        best_val_recall
    )

    print(
        "Best epoch:",
        best_epoch
    )

    print(
        "No improvement:",
        epochs_without_improvement,
        "/",
        EARLY_STOP_PATIENCE
    )

    print(
        "Learning rate:",
        current_lr
    )

    print(
        "Epoch time:",
        round(
            epoch_time,
            2
        ),
        "seconds"
    )


    # --------------------------------------------------------
    # EARLY STOPPING
    # --------------------------------------------------------

    if (

        epochs_without_improvement

        >=

        EARLY_STOP_PATIENCE

    ):

        print(
            "\n*** EARLY STOPPING TRIGGERED ***"
        )

        break


print("\n============================================")
print("DEVIT TRAINING FINISHED")
print("============================================")

print(
    "Best epoch:",
    best_epoch
)

print(
    "Best validation Macro Recall:",
    best_val_recall
)

print(
    "Last checkpoint:",
    LAST_CHECKPOINT_PATH
)

print(
    "Best checkpoint:",
    BEST_MODEL_PATH
)

In [ ]:
# ============================================================
# PART 3.8
# LOAD BEST MODEL + FINAL EVALUATION
# + DEVIT PROBABILITY EXTRACTION
#
# Purpose:
#   1. Load the best DeViT checkpoint
#   2. Evaluate Validation and Test
#   3. Save evaluation results
#   4. Extract 15-class probability from the best model
#   5. Save train/validation/test probabilities
#
# No retraining is performed in this part.
# ============================================================


# ============================================================
# CHECK BEST MODEL
# ============================================================

if not os.path.exists(
    BEST_MODEL_PATH
):

    raise FileNotFoundError(
        "Best DeViT checkpoint not found."
    )


# ============================================================
# LOAD BEST CHECKPOINT
# ============================================================

best_checkpoint = torch.load(

    BEST_MODEL_PATH,

    map_location=device
)


model.load_state_dict(

    best_checkpoint[
        "model_state_dict"
    ]
)


model.eval()


print("\n============================================")
print("BEST DeViT CHECKPOINT")
print("============================================")

print(
    "Best epoch:",
    best_checkpoint["epoch"]
)

print(
    "Best validation Macro Recall:",
    best_checkpoint[
        "best_val_macro_recall"
    ]
)


# ============================================================
# VALIDATION
# ============================================================

val_metrics, val_targets, val_predictions = (

    evaluate_devit(

        model,

        val_loader,

        criterion,

        device,

        temporal_mask_tensor
    )
)


print("\n============================================")
print("BEST MODEL — VALIDATION")
print("============================================")


for key, value in val_metrics.items():

    print(
        key + ":",
        value
    )


print("\nClassification report:")

print(

    classification_report(

        val_targets,

        val_predictions,

        labels=np.arange(
            NUM_CLASSES
        ),

        target_names=class_names,

        digits=4,

        zero_division=0
    )
)


# ============================================================
# TEST
# ============================================================

test_metrics, test_targets, test_predictions = (

    evaluate_devit(

        model,

        test_loader,

        criterion,

        device,

        temporal_mask_tensor
    )
)


print("\n============================================")
print("BEST MODEL — TEST")
print("============================================")


for key, value in test_metrics.items():

    print(
        key + ":",
        value
    )


print("\nClassification report:")

print(

    classification_report(

        test_targets,

        test_predictions,

        labels=np.arange(
            NUM_CLASSES
        ),

        target_names=class_names,

        digits=4,

        zero_division=0
    )
)


# ============================================================
# SAVE EVALUATION RESULTS
# ============================================================

devit_results = pd.DataFrame([

    {
        "dataset":
            "validation",

        **val_metrics
    },

    {
        "dataset":
            "test",

        **test_metrics
    }

])


DEVIT_RESULTS_PATH = os.path.join(

    OUTPUT_DIR,

    "devit_results.csv"
)


devit_results.to_csv(

    DEVIT_RESULTS_PATH,

    index=False
)


print(
    "\nSaved:",
    DEVIT_RESULTS_PATH
)


# ============================================================
# DEVIT PROBABILITY EXTRACTION
# ============================================================
#
# IMPORTANT:
# We use the SAME BEST MODEL loaded above.
#
# No retraining.
#
# Probability:
#
#       logits
#          ↓
#       softmax
#          ↓
#   15-class probability
#
# ============================================================


DEVIT_TRAIN_PROB_PATH = os.path.join(

    OUTPUT_DIR,

    "devit_train_prob.npy"
)


DEVIT_VAL_PROB_PATH = os.path.join(

    OUTPUT_DIR,

    "devit_val_prob.npy"
)


DEVIT_TEST_PROB_PATH = os.path.join(

    OUTPUT_DIR,

    "devit_test_prob.npy"
)


# ============================================================
# PROBABILITY EXTRACTION FUNCTION
# ============================================================

def extract_devit_probability(
    model,
    loader,
    output_path,
    split_name,
    temporal_mask_tensor
):

    # --------------------------------------------------------
    # Skip if already exists
    # --------------------------------------------------------

    if os.path.exists(output_path):

        existing_prob = np.load(
            output_path,
            mmap_mode="r"
        )

        print(
            "\n{} probability already exists.".format(
                split_name
            )
        )

        print(
            "Path:",
            output_path
        )

        print(
            "Shape:",
            existing_prob.shape
        )

        return

    print(
        "\n============================================"
    )

    print(
        "EXTRACTING DEVIT PROBABILITY —",
        split_name
    )

    print(
        "============================================"
    )

    model.eval()

    probability_batches = []

    # --------------------------------------------------------
    # INFERENCE
    # --------------------------------------------------------

    with torch.no_grad():

        for batch_idx, batch in enumerate(loader):

            # ------------------------------------------------
            # GET INPUTS
            # ------------------------------------------------

            if isinstance(
                batch,
                (tuple, list)
            ):

                inputs = batch[0]

            else:

                inputs = batch

            inputs = inputs.to(
                device,
                non_blocking=True
            )

            # ------------------------------------------------
            # TEMPORAL MASK
            #
            # TemporalAwareDeViT.forward() requires:
            #
            #     model(inputs, temporal_mask)
            # ------------------------------------------------

            current_mask = temporal_mask_tensor

            if current_mask is not None:

                current_mask = current_mask.to(
                    device
                )

            # ------------------------------------------------
            # FORWARD PASS
            # ------------------------------------------------

            outputs = model(
                inputs,
                current_mask
            )

            # ------------------------------------------------
            # GET LOGITS
            # ------------------------------------------------

            if isinstance(
                outputs,
                (tuple, list)
            ):

                logits = outputs[0]

            else:

                logits = outputs

            # ------------------------------------------------
            # SOFTMAX PROBABILITY
            # ------------------------------------------------

            probabilities = torch.softmax(
                logits,
                dim=1
            )

            probability_batches.append(
                probabilities
                .detach()
                .cpu()
                .numpy()
                .astype(np.float32)
            )

            # ------------------------------------------------
            # PROGRESS
            # ------------------------------------------------

            if (
                (batch_idx + 1) % 100 == 0
                or
                (batch_idx + 1) == len(loader)
            ):

                print(
                    "Batch {}/{}".format(
                        batch_idx + 1,
                        len(loader)
                    )
                )

            del inputs
            del outputs
            del logits
            del probabilities

    # --------------------------------------------------------
    # CONCATENATE
    # --------------------------------------------------------

    probabilities = np.concatenate(
        probability_batches,
        axis=0
    )

    # --------------------------------------------------------
    # SAVE
    # --------------------------------------------------------

    np.save(
        output_path,
        probabilities
    )

    print(
        "\nSaved:",
        output_path
    )

    print(
        "Shape:",
        probabilities.shape
    )

    # --------------------------------------------------------
    # SANITY CHECK
    # --------------------------------------------------------

    print(
        "Probability range:",
        probabilities.min(),
        "to",
        probabilities.max()
    )

    print(
        "Mean probability sum:",
        np.mean(
            np.sum(
                probabilities,
                axis=1
            )
        )
    )

    # --------------------------------------------------------
    # CLEAN MEMORY
    # --------------------------------------------------------

    del probability_batches
    del probabilities

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# EXTRACT TRAIN PROBABILITY
# ============================================================

extract_devit_probability(
    model,
    train_loader,
    DEVIT_TRAIN_PROB_PATH,
    "TRAIN",
    temporal_mask_tensor
)


# ============================================================
# EXTRACT VALIDATION PROBABILITY
# ============================================================

extract_devit_probability(
    model,
    val_loader,
    DEVIT_VAL_PROB_PATH,
    "VALIDATION",
    temporal_mask_tensor
)


# ============================================================
# EXTRACT TEST PROBABILITY
# ============================================================

extract_devit_probability(
    model,
    test_loader,
    DEVIT_TEST_PROB_PATH,
    "TEST",
    temporal_mask_tensor
)


# ============================================================
# FINAL VERIFICATION
# ============================================================

print(
    "\n============================================"
)

print(
    "VERIFYING DEVIT PROBABILITY ARTIFACTS"
)

print(
    "============================================"
)


devit_probability_files = {

    "Train":
        DEVIT_TRAIN_PROB_PATH,

    "Validation":
        DEVIT_VAL_PROB_PATH,

    "Test":
        DEVIT_TEST_PROB_PATH

}


for split_name, path in (

    devit_probability_files.items()

):

    if os.path.exists(
        path
    ):

        prob = np.load(

            path,

            mmap_mode="r"
        )

        print(
            "{}: FOUND".format(
                split_name
            )
        )

        print(
            "  File:",
            os.path.basename(path)
        )

        print(
            "  Shape:",
            prob.shape
        )

        print(
            "  Classes:",
            prob.shape[1]
        )

    else:

        print(
            "{}: MISSING".format(
                split_name
            )
        )


# ============================================================
# FINAL STATUS
# ============================================================

all_probability_files_exist = all(

    os.path.exists(path)

    for path in
    devit_probability_files.values()

)


print(
    "\n============================================"
)

if all_probability_files_exist:

    print(
        "DEVIT PROBABILITY EXTRACTION COMPLETED"
    )

    print(
        "All probability artifacts are ready."
    )

else:

    print(
        "WARNING: SOME DEVIT PROBABILITY FILES "
        "ARE MISSING."
    )

print(
    "============================================"
)


print(
    "\nPart 3.8 completed."
)

print(
    "Ready for Part 4.1."
)

In [ ]:
# ============================================================
# PART 3.9
# EXTRACT FINAL 96-D DeViT FEATURES
# ============================================================


def extract_features(

    model,

    loader,

    device,

    temporal_mask

):

    model.eval()


    features = []

    labels = []


    with torch.no_grad():

        for images, batch_labels in loader:


            images = images.to(

                device,

                non_blocking=True
            )


            _, batch_features = model(

                images,

                temporal_mask,

                return_features=True
            )


            features.append(

                batch_features
                .cpu()
                .numpy()
                .astype(
                    np.float32
                )
            )


            labels.append(

                batch_labels.numpy()
            )


    features = np.concatenate(

        features,

        axis=0
    )


    labels = np.concatenate(

        labels,

        axis=0
    )


    return (

        features,

        labels
    )


# ============================================================
# TRAIN
# ============================================================

if os.path.exists(
    DEVIT_TRAIN_FEATURE_PATH
):

    print(
        "Train features already exist."
    )

    devit_train_features = np.load(

        DEVIT_TRAIN_FEATURE_PATH
    )

    devit_train_labels = np.load(

        DEVIT_TRAIN_LABEL_PATH
    )

else:

    print("\n============================================")
    print("EXTRACTING TRAIN FEATURES")
    print("============================================")


    devit_train_features, devit_train_labels = (

        extract_features(

            model,

            train_loader,

            device,

            temporal_mask_tensor
        )
    )


    np.save(

        DEVIT_TRAIN_FEATURE_PATH,

        devit_train_features
    )


    np.save(

        DEVIT_TRAIN_LABEL_PATH,

        devit_train_labels
    )


print(
    "Train features:",
    devit_train_features.shape
)


# ============================================================
# VALIDATION
# ============================================================

if os.path.exists(
    DEVIT_VAL_FEATURE_PATH
):

    print(
        "Validation features already exist."
    )

    devit_val_features = np.load(

        DEVIT_VAL_FEATURE_PATH
    )

    devit_val_labels = np.load(

        DEVIT_VAL_LABEL_PATH
    )

else:

    print("\n============================================")
    print("EXTRACTING VALIDATION FEATURES")
    print("============================================")


    devit_val_features, devit_val_labels = (

        extract_features(

            model,

            val_loader,

            device,

            temporal_mask_tensor
        )
    )


    np.save(

        DEVIT_VAL_FEATURE_PATH,

        devit_val_features
    )


    np.save(

        DEVIT_VAL_LABEL_PATH,

        devit_val_labels
    )


print(
    "Validation features:",
    devit_val_features.shape
)


# ============================================================
# TEST
# ============================================================

if os.path.exists(
    DEVIT_TEST_FEATURE_PATH
):

    print(
        "Test features already exist."
    )

    devit_test_features = np.load(

        DEVIT_TEST_FEATURE_PATH
    )

    devit_test_labels = np.load(

        DEVIT_TEST_LABEL_PATH
    )

else:

    print("\n============================================")
    print("EXTRACTING TEST FEATURES")
    print("============================================")


    devit_test_features, devit_test_labels = (

        extract_features(

            model,

            test_loader,

            device,

            temporal_mask_tensor
        )
    )


    np.save(

        DEVIT_TEST_FEATURE_PATH,

        devit_test_features
    )


    np.save(

        DEVIT_TEST_LABEL_PATH,

        devit_test_labels
    )


print(
    "Test features:",
    devit_test_features.shape
)


# ============================================================
# FINAL VERIFICATION
# ============================================================

print("\n============================================")
print("DeViT FEATURES READY FOR PART 4")
print("============================================")


print(
    "Train:",
    devit_train_features.shape
)

print(
    "Validation:",
    devit_val_features.shape
)

print(
    "Test:",
    devit_test_features.shape
)

print(
    "\nExpected feature dimension:",
    EMBED_DIM
)


assert (
    devit_train_features.shape[1]
    == 96
)

assert (
    devit_val_features.shape[1]
    == 96
)

assert (
    devit_test_features.shape[1]
    == 96
)


print(
    "\nAll DeViT artifacts are ready."
)

## Part 4 XGB Fusion

In [ ]:
# ============================================================
# PART 4.1
# MEMORY-EFFICIENT LOAD ALL XGBOOST + DEVIT ARTIFACTS
#
# Purpose:
#   Load and verify saved artifacts from Part 2 and Part 3
#   without loading all large arrays into RAM.
#
# IMPORTANT:
#   - No XGBoost retraining
#   - No DeViT retraining
#   - No fusion yet
#   - Uses NumPy memory mapping for large .npy files
# ============================================================

import os
import gc
import pickle
import numpy as np

from google.colab import drive


# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================

drive.mount(
    "/content/drive"
)


# ============================================================
# PATHS
# ============================================================

OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "My Research/USULAN PENELITIAN DIKTI/2026/"
    "IDS-KN/luaran/outputs"
)


# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42

EXPECTED_NUM_CLASSES = 15
EXPECTED_DEVIT_DIM = 96


# ============================================================
# ARTIFACT PATHS
# ============================================================

# ------------------------------------------------------------
# XGBoost probabilities
# ------------------------------------------------------------

XGB_TRAIN_PROB_PATH = os.path.join(
    OUTPUT_DIR,
    "xgb_train_prob.npy"
)

XGB_VAL_PROB_PATH = os.path.join(
    OUTPUT_DIR,
    "xgb_val_prob.npy"
)

XGB_TEST_PROB_PATH = os.path.join(
    OUTPUT_DIR,
    "xgb_test_prob.npy"
)


# ------------------------------------------------------------
# DeViT representations
# ------------------------------------------------------------

DEVIT_TRAIN_FEATURE_PATH = os.path.join(
    OUTPUT_DIR,
    "devit_train_features.npy"
)

DEVIT_VAL_FEATURE_PATH = os.path.join(
    OUTPUT_DIR,
    "devit_val_features.npy"
)

DEVIT_TEST_FEATURE_PATH = os.path.join(
    OUTPUT_DIR,
    "devit_test_features.npy"
)


# ------------------------------------------------------------
# DeViT probabilities
# ------------------------------------------------------------

DEVIT_TRAIN_PROB_PATH = os.path.join(
    OUTPUT_DIR,
    "devit_train_prob.npy"
)

DEVIT_VAL_PROB_PATH = os.path.join(
    OUTPUT_DIR,
    "devit_val_prob.npy"
)

DEVIT_TEST_PROB_PATH = os.path.join(
    OUTPUT_DIR,
    "devit_test_prob.npy"
)


# ------------------------------------------------------------
# Labels
# ------------------------------------------------------------

DEVIT_TRAIN_LABEL_PATH = os.path.join(
    OUTPUT_DIR,
    "devit_train_labels.npy"
)

DEVIT_VAL_LABEL_PATH = os.path.join(
    OUTPUT_DIR,
    "devit_val_labels.npy"
)

DEVIT_TEST_LABEL_PATH = os.path.join(
    OUTPUT_DIR,
    "devit_test_labels.npy"
)


# ------------------------------------------------------------
# Label encoder
# ------------------------------------------------------------

LABEL_ENCODER_PATH = os.path.join(
    OUTPUT_DIR,
    "label_encoder.pkl"
)


# ============================================================
# CHECK REQUIRED FILES
# ============================================================

required_files = [

    # XGBoost
    XGB_TRAIN_PROB_PATH,
    XGB_VAL_PROB_PATH,
    XGB_TEST_PROB_PATH,

    # DeViT representations
    DEVIT_TRAIN_FEATURE_PATH,
    DEVIT_VAL_FEATURE_PATH,
    DEVIT_TEST_FEATURE_PATH,

    # DeViT probabilities
    DEVIT_TRAIN_PROB_PATH,
    DEVIT_VAL_PROB_PATH,
    DEVIT_TEST_PROB_PATH,

    # Labels
    DEVIT_TRAIN_LABEL_PATH,
    DEVIT_VAL_LABEL_PATH,
    DEVIT_TEST_LABEL_PATH,

    # Encoder
    LABEL_ENCODER_PATH
]


print(
    "\n============================================"
)

print(
    "CHECKING REQUIRED ARTIFACTS"
)

print(
    "============================================"
)


missing_files = []


for path in required_files:

    if os.path.exists(path):

        file_size_mb = (
            os.path.getsize(path)
            / (1024 ** 2)
        )

        print(
            "FOUND:",
            os.path.basename(path),
            "|",
            f"{file_size_mb:.2f} MB"
        )

    else:

        print(
            "MISSING:",
            os.path.basename(path)
        )

        missing_files.append(path)


# ============================================================
# STOP IF ARTIFACT IS MISSING
# ============================================================

if len(missing_files) > 0:

    print(
        "\n============================================"
    )

    print(
        "MISSING ARTIFACTS"
    )

    print(
        "============================================"
    )

    for path in missing_files:

        print(
            path
        )

    raise FileNotFoundError(
        "\nRequired artifact(s) are missing. "
        "Do not continue to Part 4.2."
    )


# ============================================================
# LOAD LABEL ENCODER
# ============================================================

print(
    "\n============================================"
)

print(
    "LOADING LABEL ENCODER"
)

print(
    "============================================"
)


with open(
    LABEL_ENCODER_PATH,
    "rb"
) as f:

    label_encoder = pickle.load(f)


class_names = (
    label_encoder.classes_.tolist()
)


NUM_CLASSES = len(
    class_names
)


print(
    "Number of classes:",
    NUM_CLASSES
)

print(
    "Classes:"
)

for idx, name in enumerate(
    class_names
):

    print(
        f"{idx:02d}: {name}"
    )


# ============================================================
# CHECK NUMBER OF CLASSES
# ============================================================

if NUM_CLASSES != EXPECTED_NUM_CLASSES:

    raise ValueError(
        "Unexpected number of classes: "
        f"{NUM_CLASSES}. "
        f"Expected {EXPECTED_NUM_CLASSES}."
    )


# ============================================================
# MEMORY-MAPPED LOADER
# ============================================================

def load_memmap(
    path,
    name
):

    print(
        f"\nLoading {name}..."
    )

    array = np.load(
        path,
        mmap_mode="r"
    )

    print(
        "  Shape:",
        array.shape
    )

    print(
        "  Dtype:",
        array.dtype
    )

    print(
        "  Memory mapped: YES"
    )

    return array


# ============================================================
# LOAD XGBOOST PROBABILITIES
# ============================================================

print(
    "\n============================================"
)

print(
    "LOADING XGBOOST PROBABILITIES"
)

print(
    "============================================"
)


xgb_train_prob = load_memmap(
    XGB_TRAIN_PROB_PATH,
    "XGB train probability"
)

xgb_val_prob = load_memmap(
    XGB_VAL_PROB_PATH,
    "XGB validation probability"
)

xgb_test_prob = load_memmap(
    XGB_TEST_PROB_PATH,
    "XGB test probability"
)


# ============================================================
# LOAD DEVIT REPRESENTATIONS
# ============================================================

print(
    "\n============================================"
)

print(
    "LOADING DEVIT 96-D REPRESENTATIONS"
)

print(
    "============================================"
)


devit_train = load_memmap(
    DEVIT_TRAIN_FEATURE_PATH,
    "DeViT train features"
)

devit_val = load_memmap(
    DEVIT_VAL_FEATURE_PATH,
    "DeViT validation features"
)

devit_test = load_memmap(
    DEVIT_TEST_FEATURE_PATH,
    "DeViT test features"
)


# ============================================================
# LOAD DEVIT PROBABILITIES
# ============================================================

print(
    "\n============================================"
)

print(
    "LOADING DEVIT PROBABILITIES"
)

print(
    "============================================"
)


devit_train_prob = load_memmap(
    DEVIT_TRAIN_PROB_PATH,
    "DeViT train probability"
)

devit_val_prob = load_memmap(
    DEVIT_VAL_PROB_PATH,
    "DeViT validation probability"
)

devit_test_prob = load_memmap(
    DEVIT_TEST_PROB_PATH,
    "DeViT test probability"
)


# ============================================================
# LOAD LABELS
#
# Labels are relatively small compared with feature matrices,
# so normal loading is acceptable.
# ============================================================

print(
    "\n============================================"
)

print(
    "LOADING LABELS"
)

print(
    "============================================"
)


y_train = np.load(
    DEVIT_TRAIN_LABEL_PATH
)

y_val = np.load(
    DEVIT_VAL_LABEL_PATH
)

y_test = np.load(
    DEVIT_TEST_LABEL_PATH
)


print(
    "Train labels:",
    y_train.shape,
    y_train.dtype
)

print(
    "Validation labels:",
    y_val.shape,
    y_val.dtype
)

print(
    "Test labels:",
    y_test.shape,
    y_test.dtype
)


# ============================================================
# ARTIFACT SHAPES
# ============================================================

print(
    "\n============================================"
)

print(
    "ARTIFACT SHAPES"
)

print(
    "============================================"
)


print(
    "XGB train probability :",
    xgb_train_prob.shape
)

print(
    "XGB validation        :",
    xgb_val_prob.shape
)

print(
    "XGB test              :",
    xgb_test_prob.shape
)

print()

print(
    "DeViT train features  :",
    devit_train.shape
)

print(
    "DeViT validation      :",
    devit_val.shape
)

print(
    "DeViT test            :",
    devit_test.shape
)

print()

print(
    "DeViT train probability:",
    devit_train_prob.shape
)

print(
    "DeViT validation       :",
    devit_val_prob.shape
)

print(
    "DeViT test             :",
    devit_test_prob.shape
)

print()

print(
    "Train labels:",
    y_train.shape
)

print(
    "Val labels:",
    y_val.shape
)

print(
    "Test labels:",
    y_test.shape
)


# ============================================================
# CONSISTENCY CHECK
# ============================================================

print(
    "\n============================================"
)

print(
    "CHECKING DIMENSION CONSISTENCY"
)

print(
    "============================================"
)


# ------------------------------------------------------------
# XGBoost probability
# ------------------------------------------------------------

assert (
    xgb_train_prob.shape[0]
    ==
    y_train.shape[0]
)

assert (
    xgb_val_prob.shape[0]
    ==
    y_val.shape[0]
)

assert (
    xgb_test_prob.shape[0]
    ==
    y_test.shape[0]
)


# ------------------------------------------------------------
# DeViT representation
# ------------------------------------------------------------

assert (
    devit_train.shape[0]
    ==
    y_train.shape[0]
)

assert (
    devit_val.shape[0]
    ==
    y_val.shape[0]
)

assert (
    devit_test.shape[0]
    ==
    y_test.shape[0]
)


# ------------------------------------------------------------
# DeViT probability
# ------------------------------------------------------------

assert (
    devit_train_prob.shape[0]
    ==
    y_train.shape[0]
)

assert (
    devit_val_prob.shape[0]
    ==
    y_val.shape[0]
)

assert (
    devit_test_prob.shape[0]
    ==
    y_test.shape[0]
)


# ------------------------------------------------------------
# Number of classes
# ------------------------------------------------------------

assert (
    xgb_train_prob.shape[1]
    ==
    NUM_CLASSES
)

assert (
    xgb_val_prob.shape[1]
    ==
    NUM_CLASSES
)

assert (
    xgb_test_prob.shape[1]
    ==
    NUM_CLASSES
)


assert (
    devit_train_prob.shape[1]
    ==
    NUM_CLASSES
)

assert (
    devit_val_prob.shape[1]
    ==
    NUM_CLASSES
)

assert (
    devit_test_prob.shape[1]
    ==
    NUM_CLASSES
)


# ------------------------------------------------------------
# DeViT representation dimension
# ------------------------------------------------------------

assert (
    devit_train.shape[1]
    ==
    EXPECTED_DEVIT_DIM
)

assert (
    devit_val.shape[1]
    ==
    EXPECTED_DEVIT_DIM
)

assert (
    devit_test.shape[1]
    ==
    EXPECTED_DEVIT_DIM
)


print(
    "All dimensions are consistent."
)


# ============================================================
# CHECK PROBABILITY VALUES
# ============================================================

print(
    "\n============================================"
)

print(
    "CHECKING PROBABILITY ARTIFACTS"
)

print(
    "============================================"
)


def check_probability(
    probability_array,
    name
):

    # Read only a small sample.
    #
    # This avoids scanning the entire memory-mapped
    # array and keeps RAM usage low.

    sample_size = min(
        10000,
        probability_array.shape[0]
    )

    sample = probability_array[
        :sample_size
    ]


    # --------------------------------------------------------
    # Range
    # --------------------------------------------------------

    min_value = np.min(
        sample
    )

    max_value = np.max(
        sample
    )


    # --------------------------------------------------------
    # Probability sum
    # --------------------------------------------------------

    probability_sum = np.sum(
        sample,
        axis=1
    )

    mean_sum = np.mean(
        probability_sum
    )


    print(
        f"{name}:"
    )

    print(
        "  Min:",
        float(min_value)
    )

    print(
        "  Max:",
        float(max_value)
    )

    print(
        "  Mean probability sum:",
        float(mean_sum)
    )


    # --------------------------------------------------------
    # Sanity conditions
    # --------------------------------------------------------

    if min_value < -1e-5:

        raise ValueError(
            f"{name} contains negative probabilities."
        )

    if max_value > 1.00001:

        raise ValueError(
            f"{name} contains values greater than 1."
        )

    if not np.allclose(
        probability_sum,
        1.0,
        atol=1e-4
    ):

        raise ValueError(
            f"{name} probabilities do not sum to 1."
        )


# ------------------------------------------------------------
# XGBoost
# ------------------------------------------------------------

check_probability(
    xgb_train_prob,
    "XGB train"
)

check_probability(
    xgb_val_prob,
    "XGB validation"
)

check_probability(
    xgb_test_prob,
    "XGB test"
)


# ------------------------------------------------------------
# DeViT
# ------------------------------------------------------------

check_probability(
    devit_train_prob,
    "DeViT train"
)

check_probability(
    devit_val_prob,
    "DeViT validation"
)

check_probability(
    devit_test_prob,
    "DeViT test"
)


# ============================================================
# FINAL MEMORY-MAPPED STATUS
# ============================================================

print(
    "\n============================================"
)

print(
    "MEMORY-EFFICIENT ARTIFACT STATUS"
)

print(
    "============================================"
)

print(
    "XGB probabilities       : MEMORY-MAPPED"
)

print(
    "DeViT representations   : MEMORY-MAPPED"
)

print(
    "DeViT probabilities     : MEMORY-MAPPED"
)

print(
    "Labels                  : LOADED"
)

print(
    "Label encoder           : LOADED"
)


# ============================================================
# PART 4.1 COMPLETED
# ============================================================

print(
    "\n============================================"
)

print(
    "PART 4.1 COMPLETED"
)

print(
    "============================================"
)

print(
    "All artifacts are available."
)

print(
    "No model training was performed."
)

print(
    "Large arrays remain memory-mapped."
)

print(
    "Ready for Part 4.2."
)

In [ ]:
# ============================================================
# PART 4.2
# FEATURE-LEVEL FUSION PREPARATION
#
# XGBoost + DeViT
#
# Input:
#   - DeViT 96-D representation
#   - XGBoost 15-D probability
#   - DeViT 15-D probability
#
# Output:
#   - 126-D fused feature representation
#
# IMPORTANT:
#   - No model retraining
#   - No RF
#   - No second-stage XGBoost yet
#   - Memory-conscious
#   - Existing artifacts are reused
#   - If output already exists, it is NOT regenerated
# ============================================================

import os
import gc
import numpy as np
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

FUSION_BATCH_SIZE = 32768

FUSION_FEATURE_DIM = 96 + 15 + 15


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

FUSION_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR,
    "feature_fusion"
)

os.makedirs(
    FUSION_OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# OUTPUT PATHS
# ============================================================

FUSION_TRAIN_PATH = os.path.join(
    FUSION_OUTPUT_DIR,
    "fusion_train_features.npy"
)

FUSION_VAL_PATH = os.path.join(
    FUSION_OUTPUT_DIR,
    "fusion_val_features.npy"
)

FUSION_TEST_PATH = os.path.join(
    FUSION_OUTPUT_DIR,
    "fusion_test_features.npy"
)

FUSION_LABEL_TRAIN_PATH = os.path.join(
    FUSION_OUTPUT_DIR,
    "fusion_train_labels.npy"
)

FUSION_LABEL_VAL_PATH = os.path.join(
    FUSION_OUTPUT_DIR,
    "fusion_val_labels.npy"
)

FUSION_LABEL_TEST_PATH = os.path.join(
    FUSION_OUTPUT_DIR,
    "fusion_test_labels.npy"
)


# ============================================================
# HEADER
# ============================================================

print(
    "\n============================================"
)

print(
    "PART 4.2"
)

print(
    "FEATURE-LEVEL XGBOOST + DEVIT FUSION"
)

print(
    "============================================"
)


print(
    "DeViT representation : 96"
)

print(
    "XGBoost probability   : 15"
)

print(
    "DeViT probability     : 15"
)

print(
    "Total fused features :",
    FUSION_FEATURE_DIM
)

print(
    "Batch size            :",
    FUSION_BATCH_SIZE
)


# ============================================================
# CHECK INPUT ARTIFACTS
# ============================================================

required_inputs = {

    "devit_train":
        devit_train,

    "devit_val":
        devit_val,

    "devit_test":
        devit_test,

    "xgb_train_prob":
        xgb_train_prob,

    "xgb_val_prob":
        xgb_val_prob,

    "xgb_test_prob":
        xgb_test_prob,

    "devit_train_prob":
        devit_train_prob,

    "devit_val_prob":
        devit_val_prob,

    "devit_test_prob":
        devit_test_prob,

    "y_train":
        y_train,

    "y_val":
        y_val,

    "y_test":
        y_test
}


print(
    "\n============================================"
)

print(
    "CHECKING INPUT ARTIFACTS"
)

print(
    "============================================"
)


for name, value in required_inputs.items():

    if value is None:

        raise RuntimeError(
            "Input artifact is None: "
            + name
        )

    print(
        "FOUND:",
        name
    )


# ============================================================
# SHAPE VALIDATION
# ============================================================

assert devit_train.shape[1] == 96
assert devit_val.shape[1] == 96
assert devit_test.shape[1] == 96

assert xgb_train_prob.shape[1] == 15
assert xgb_val_prob.shape[1] == 15
assert xgb_test_prob.shape[1] == 15

assert devit_train_prob.shape[1] == 15
assert devit_val_prob.shape[1] == 15
assert devit_test_prob.shape[1] == 15


assert devit_train.shape[0] == xgb_train_prob.shape[0]
assert devit_train.shape[0] == devit_train_prob.shape[0]
assert devit_train.shape[0] == y_train.shape[0]

assert devit_val.shape[0] == xgb_val_prob.shape[0]
assert devit_val.shape[0] == devit_val_prob.shape[0]
assert devit_val.shape[0] == y_val.shape[0]

assert devit_test.shape[0] == xgb_test_prob.shape[0]
assert devit_test.shape[0] == devit_test_prob.shape[0]
assert devit_test.shape[0] == y_test.shape[0]


print(
    "\nInput dimensions are consistent."
)


# ============================================================
# BUILD FUSION FEATURES
# ============================================================

def build_fusion_features(
    devit_features,
    xgb_probability,
    devit_probability,
    output_path,
    split_name
):

    # --------------------------------------------------------
    # CHECK EXISTING FILE
    # --------------------------------------------------------

    if os.path.exists(
        output_path
    ):

        existing = np.load(
            output_path,
            mmap_mode="r"
        )

        print(
            "\n============================================"
        )

        print(
            split_name,
            "FUSION ARTIFACT ALREADY EXISTS"
        )

        print(
            "============================================"
        )

        print(
            "Path:",
            output_path
        )

        print(
            "Shape:",
            existing.shape
        )

        print(
            "No regeneration required."
        )

        return existing


    # --------------------------------------------------------
    # Number of samples
    # --------------------------------------------------------

    n_samples = (
        devit_features.shape[0]
    )


    print(
        "\n============================================"
    )

    print(
        "BUILDING FUSION FEATURES —",
        split_name
    )

    print(
        "============================================"
    )

    print(
        "Samples:",
        f"{n_samples:,}"
    )

    print(
        "Features:",
        FUSION_FEATURE_DIM
    )


    # --------------------------------------------------------
    # Disk-backed NumPy array
    #
    # This avoids holding the entire output temporarily
    # in memory.
    # --------------------------------------------------------

    fusion_features = np.lib.format.open_memmap(

        output_path,

        mode="w+",

        dtype=np.float32,

        shape=(
            n_samples,
            FUSION_FEATURE_DIM
        )
    )


    # --------------------------------------------------------
    # Batch processing
    # --------------------------------------------------------

    for start in range(
        0,
        n_samples,
        FUSION_BATCH_SIZE
    ):

        end = min(
            start + FUSION_BATCH_SIZE,
            n_samples
        )


        # ----------------------------------------------------
        # Current DeViT representation
        # ----------------------------------------------------

        devit_batch = np.asarray(

            devit_features[
                start:end
            ],

            dtype=np.float32
        )


        # ----------------------------------------------------
        # Current XGBoost probability
        # ----------------------------------------------------

        xgb_batch = np.asarray(

            xgb_probability[
                start:end
            ],

            dtype=np.float32
        )


        # ----------------------------------------------------
        # Current DeViT probability
        # ----------------------------------------------------

        devit_prob_batch = np.asarray(

            devit_probability[
                start:end
            ],

            dtype=np.float32
        )


        # ----------------------------------------------------
        # Construct 126-D representation
        #
        # [ DeViT-96 |
        #   XGB-15   |
        #   DeViT-15 ]
        # ----------------------------------------------------

        fusion_batch = np.concatenate(

            [

                devit_batch,

                xgb_batch,

                devit_prob_batch

            ],

            axis=1
        )


        # ----------------------------------------------------
        # Write directly to disk
        # ----------------------------------------------------

        fusion_features[
            start:end
        ] = fusion_batch


        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if (

            start == 0

            or

            end == n_samples

            or

            (
                (start // FUSION_BATCH_SIZE)
                % 20
                == 0
            )

        ):

            print(

                "Processed:",

                f"{end:,}",

                "/",

                f"{n_samples:,}"

            )


        # ----------------------------------------------------
        # Cleanup
        # ----------------------------------------------------

        del devit_batch
        del xgb_batch
        del devit_prob_batch
        del fusion_batch


    # --------------------------------------------------------
    # Flush
    # --------------------------------------------------------

    fusion_features.flush()

    del fusion_features

    gc.collect()


    # --------------------------------------------------------
    # Reload as memory map
    # --------------------------------------------------------

    fusion_features = np.load(

        output_path,

        mmap_mode="r"
    )


    print(
        "\nFusion completed."
    )

    print(
        "Output:",
        output_path
    )

    print(
        "Shape:",
        fusion_features.shape
    )

    print(
        "Dtype:",
        fusion_features.dtype
    )


    return fusion_features


# ============================================================
# TRAIN FUSION
# ============================================================

fusion_train = build_fusion_features(

    devit_train,

    xgb_train_prob,

    devit_train_prob,

    FUSION_TRAIN_PATH,

    "TRAIN"
)


# ============================================================
# VALIDATION FUSION
# ============================================================

fusion_val = build_fusion_features(

    devit_val,

    xgb_val_prob,

    devit_val_prob,

    FUSION_VAL_PATH,

    "VALIDATION"
)


# ============================================================
# TEST FUSION
# ============================================================

fusion_test = build_fusion_features(

    devit_test,

    xgb_test_prob,

    devit_test_prob,

    FUSION_TEST_PATH,

    "TEST"
)


# ============================================================
# SAVE LABELS
# ============================================================

print(
    "\n============================================"
)

print(
    "SAVING FUSION LABELS"
)

print(
    "============================================"
)


if not os.path.exists(
    FUSION_LABEL_TRAIN_PATH
):

    np.save(
        FUSION_LABEL_TRAIN_PATH,
        y_train
    )

else:

    print(
        "Train labels already exist."
    )


if not os.path.exists(
    FUSION_LABEL_VAL_PATH
):

    np.save(
        FUSION_LABEL_VAL_PATH,
        y_val
    )

else:

    print(
        "Validation labels already exist."
    )


if not os.path.exists(
    FUSION_LABEL_TEST_PATH
):

    np.save(
        FUSION_LABEL_TEST_PATH,
        y_test
    )

else:

    print(
        "Test labels already exist."
    )


# ============================================================
# FINAL SHAPE CHECK
# ============================================================

print(
    "\n============================================"
)

print(
    "FINAL FUSION ARTIFACT CHECK"
)

print(
    "============================================"
)


print(
    "Train:",
    fusion_train.shape
)

print(
    "Validation:",
    fusion_val.shape
)

print(
    "Test:",
    fusion_test.shape
)


assert fusion_train.shape == (
    y_train.shape[0],
    126
)

assert fusion_val.shape == (
    y_val.shape[0],
    126
)

assert fusion_test.shape == (
    y_test.shape[0],
    126
)


# ============================================================
# FINAL STATUS
# ============================================================

print(
    "\n============================================"
)

print(
    "PART 4.2 COMPLETED"
)

print(
    "============================================"
)

print(
    "Train fusion:",
    FUSION_TRAIN_PATH
)

print(
    "Validation fusion:",
    FUSION_VAL_PATH
)

print(
    "Test fusion:",
    FUSION_TEST_PATH
)

print(
    "\n126-D fusion representation is ready."
)

print(
    "Ready for Part 4.3 — Second-Stage XGBoost."
)

In [ ]:
# ============================================================
# PART 4.3
# SECOND-STAGE XGBOOST FUSION CLASSIFIER
#
# Input:
#   96-D DeViT representation
#   15-D XGBoost probability
#   15-D DeViT probability
#
# Total:
#   126-D fusion representation
#
# Classifier:
#   Second-Stage XGBoost
#
# No RF
# No DeViT retraining
# No first-stage XGBoost retraining
# ============================================================

import os
import gc
import pickle
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

from xgboost import XGBClassifier


# ============================================================
# CONFIGURATION
# ============================================================

SECOND_XGB_N_ESTIMATORS = 300

SECOND_XGB_MAX_DEPTH = 8

SECOND_XGB_LEARNING_RATE = 0.05

SECOND_XGB_SUBSAMPLE = 0.8

SECOND_XGB_COLSAMPLE = 0.8

SECOND_XGB_MIN_CHILD_WEIGHT = 1

SECOND_XGB_GAMMA = 0

SECOND_XGB_REG_ALPHA = 0

SECOND_XGB_REG_LAMBDA = 1

SECOND_XGB_TREE_METHOD = "hist"

SECOND_XGB_N_JOBS = 2

RANDOM_STATE = 42

NUM_CLASSES = 15

FUSION_FEATURE_DIM = 126


# ============================================================
# HEADER
# ============================================================

print(
    "\n============================================"
)

print(
    "PART 4.3"
)

print(
    "SECOND-STAGE XGBOOST FUSION CLASSIFIER"
)

print(
    "============================================"
)

print(
    "Input features:",
    FUSION_FEATURE_DIM
)

print(
    "Classes:",
    NUM_CLASSES
)

print(
    "Estimators:",
    SECOND_XGB_N_ESTIMATORS
)

print(
    "Max depth:",
    SECOND_XGB_MAX_DEPTH
)

print(
    "Learning rate:",
    SECOND_XGB_LEARNING_RATE
)


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

SECOND_XGB_DIR = os.path.join(
    OUTPUT_DIR,
    "second_stage_xgboost"
)

os.makedirs(
    SECOND_XGB_DIR,
    exist_ok=True
)


# ============================================================
# PART 4.2 ARTIFACT PATHS
#
# IMPORTANT:
# Define paths here independently.
# This prevents NameError after runtime restart.
# ============================================================

FUSION_DIR = os.path.join(
    OUTPUT_DIR,
    "feature_fusion"
)


FUSION_TRAIN_PATH = os.path.join(
    FUSION_DIR,
    "fusion_train_features.npy"
)

FUSION_VAL_PATH = os.path.join(
    FUSION_DIR,
    "fusion_val_features.npy"
)

FUSION_TEST_PATH = os.path.join(
    FUSION_DIR,
    "fusion_test_features.npy"
)


FUSION_LABEL_TRAIN_PATH = os.path.join(
    FUSION_DIR,
    "fusion_train_labels.npy"
)

FUSION_LABEL_VAL_PATH = os.path.join(
    FUSION_DIR,
    "fusion_val_labels.npy"
)

FUSION_LABEL_TEST_PATH = os.path.join(
    FUSION_DIR,
    "fusion_test_labels.npy"
)


# ============================================================
# OUTPUT PATHS
# ============================================================

SECOND_XGB_MODEL_PATH = os.path.join(
    SECOND_XGB_DIR,
    "second_stage_xgboost_model.pkl"
)

SECOND_XGB_TRAIN_PROB_PATH = os.path.join(
    SECOND_XGB_DIR,
    "second_stage_xgb_train_prob.npy"
)

SECOND_XGB_VAL_PROB_PATH = os.path.join(
    SECOND_XGB_DIR,
    "second_stage_xgb_val_prob.npy"
)

SECOND_XGB_TEST_PROB_PATH = os.path.join(
    SECOND_XGB_DIR,
    "second_stage_xgb_test_prob.npy"
)

SECOND_XGB_TRAIN_PRED_PATH = os.path.join(
    SECOND_XGB_DIR,
    "second_stage_xgb_train_pred.npy"
)

SECOND_XGB_VAL_PRED_PATH = os.path.join(
    SECOND_XGB_DIR,
    "second_stage_xgb_val_pred.npy"
)

SECOND_XGB_TEST_PRED_PATH = os.path.join(
    SECOND_XGB_DIR,
    "second_stage_xgb_test_pred.npy"
)

SECOND_XGB_RESULTS_PATH = os.path.join(
    SECOND_XGB_DIR,
    "second_stage_xgboost_results.csv"
)


# ============================================================
# CHECK PART 4.2 ARTIFACTS
# ============================================================

print(
    "\n============================================"
)

print(
    "CHECKING PART 4.2 ARTIFACTS"
)

print(
    "============================================"
)


required_paths = {

    "fusion_train":
        FUSION_TRAIN_PATH,

    "fusion_val":
        FUSION_VAL_PATH,

    "fusion_test":
        FUSION_TEST_PATH,

    "fusion_train_labels":
        FUSION_LABEL_TRAIN_PATH,

    "fusion_val_labels":
        FUSION_LABEL_VAL_PATH,

    "fusion_test_labels":
        FUSION_LABEL_TEST_PATH
}


missing = []


for name, path in required_paths.items():

    if os.path.exists(path):

        print(
            "FOUND:",
            name
        )

    else:

        print(
            "MISSING:",
            name
        )

        missing.append(path)


if len(missing) > 0:

    raise FileNotFoundError(
        "\nPart 4.2 artifacts are incomplete.\n"
        "Run Part 4.2 first."
    )


# ============================================================
# LOAD FUSION FEATURES AS MEMORY MAP
# ============================================================

print(
    "\n============================================"
)

print(
    "LOADING FUSION FEATURES"
)

print(
    "============================================"
)


fusion_train = np.load(
    FUSION_TRAIN_PATH,
    mmap_mode="r"
)

fusion_val = np.load(
    FUSION_VAL_PATH,
    mmap_mode="r"
)

fusion_test = np.load(
    FUSION_TEST_PATH,
    mmap_mode="r"
)


# ============================================================
# LOAD LABELS
# ============================================================

y_train = np.load(
    FUSION_LABEL_TRAIN_PATH
)

y_val = np.load(
    FUSION_LABEL_VAL_PATH
)

y_test = np.load(
    FUSION_LABEL_TEST_PATH
)


# ============================================================
# SHAPE CHECK
# ============================================================

print(
    "Train fusion:",
    fusion_train.shape
)

print(
    "Validation fusion:",
    fusion_val.shape
)

print(
    "Test fusion:",
    fusion_test.shape
)

print(
    "Train labels:",
    y_train.shape
)

print(
    "Validation labels:",
    y_val.shape
)

print(
    "Test labels:",
    y_test.shape
)


assert fusion_train.shape[1] == 126
assert fusion_val.shape[1] == 126
assert fusion_test.shape[1] == 126

assert fusion_train.shape[0] == y_train.shape[0]
assert fusion_val.shape[0] == y_val.shape[0]
assert fusion_test.shape[0] == y_test.shape[0]


print(
    "\nFusion dimensions are consistent."
)


# ============================================================
# CREATE SECOND-STAGE XGBOOST
# ============================================================

print(
    "\n============================================"
)

print(
    "CREATING SECOND-STAGE XGBOOST"
)

print(
    "============================================"
)


second_xgb = XGBClassifier(

    n_estimators=
        SECOND_XGB_N_ESTIMATORS,

    max_depth=
        SECOND_XGB_MAX_DEPTH,

    learning_rate=
        SECOND_XGB_LEARNING_RATE,

    subsample=
        SECOND_XGB_SUBSAMPLE,

    colsample_bytree=
        SECOND_XGB_COLSAMPLE,

    min_child_weight=
        SECOND_XGB_MIN_CHILD_WEIGHT,

    gamma=
        SECOND_XGB_GAMMA,

    reg_alpha=
        SECOND_XGB_REG_ALPHA,

    reg_lambda=
        SECOND_XGB_REG_LAMBDA,

    objective=
        "multi:softprob",

    num_class=
        NUM_CLASSES,

    eval_metric=
        "mlogloss",

    tree_method=
        SECOND_XGB_TREE_METHOD,

    n_jobs=
        SECOND_XGB_N_JOBS,

    random_state=
        RANDOM_STATE,

    verbosity=
        1
)


# ============================================================
# TRAIN
# ============================================================

print(
    "\n============================================"
)

print(
    "TRAINING SECOND-STAGE XGBOOST"
)

print(
    "============================================"
)

print(
    "Training samples:",
    f"{fusion_train.shape[0]:,}"
)

print(
    "Training features:",
    fusion_train.shape[1]
)

print(
    "Validation samples:",
    f"{fusion_val.shape[0]:,}"
)

print(
    "Number of estimators:",
    SECOND_XGB_N_ESTIMATORS
)


second_xgb.fit(

    fusion_train,

    y_train,

    eval_set=[
        (
            fusion_val,
            y_val
        )
    ],

    verbose=True
)


# ============================================================
# SAVE MODEL
# ============================================================

print(
    "\n============================================"
)

print(
    "SAVING SECOND-STAGE XGBOOST"
)

print(
    "============================================"
)


with open(
    SECOND_XGB_MODEL_PATH,
    "wb"
) as f:

    pickle.dump(
        second_xgb,
        f
    )


print(
    "Saved:",
    SECOND_XGB_MODEL_PATH
)


# ============================================================
# GENERATE PROBABILITIES
# ============================================================

print(
    "\n============================================"
)

print(
    "GENERATING FINAL PROBABILITIES"
)

print(
    "============================================"
)


print(
    "TRAIN..."
)

second_xgb_train_prob = (
    second_xgb.predict_proba(
        fusion_train
    )
)


print(
    "VALIDATION..."
)

second_xgb_val_prob = (
    second_xgb.predict_proba(
        fusion_val
    )
)


print(
    "TEST..."
)

second_xgb_test_prob = (
    second_xgb.predict_proba(
        fusion_test
    )
)


# ============================================================
# SAVE PROBABILITIES
# ============================================================

np.save(
    SECOND_XGB_TRAIN_PROB_PATH,
    second_xgb_train_prob.astype(
        np.float32
    )
)

np.save(
    SECOND_XGB_VAL_PROB_PATH,
    second_xgb_val_prob.astype(
        np.float32
    )
)

np.save(
    SECOND_XGB_TEST_PROB_PATH,
    second_xgb_test_prob.astype(
        np.float32
    )
)


# ============================================================
# PREDICTIONS
# ============================================================

second_xgb_train_pred = np.argmax(
    second_xgb_train_prob,
    axis=1
)

second_xgb_val_pred = np.argmax(
    second_xgb_val_prob,
    axis=1
)

second_xgb_test_pred = np.argmax(
    second_xgb_test_prob,
    axis=1
)


np.save(
    SECOND_XGB_TRAIN_PRED_PATH,
    second_xgb_train_pred.astype(
        np.int16
    )
)

np.save(
    SECOND_XGB_VAL_PRED_PATH,
    second_xgb_val_pred.astype(
        np.int16
    )
)

np.save(
    SECOND_XGB_TEST_PRED_PATH,
    second_xgb_test_pred.astype(
        np.int16
    )
)


print(
    "\nProbabilities and predictions saved."
)


# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate_second_xgb(
    y_true,
    y_pred
):

    return {

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "macro_precision":
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "macro_recall":
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            )
    }


# ============================================================
# TRAIN METRICS
# ============================================================

train_metrics = evaluate_second_xgb(

    y_train,

    second_xgb_train_pred
)


print(
    "\n============================================"
)

print(
    "SECOND-STAGE XGBOOST — TRAIN"
)

print(
    "============================================"
)


for key, value in train_metrics.items():

    print(
        key + ":",
        value
    )


# ============================================================
# VALIDATION METRICS
# ============================================================

val_metrics = evaluate_second_xgb(

    y_val,

    second_xgb_val_pred
)


print(
    "\n============================================"
)

print(
    "SECOND-STAGE XGBOOST — VALIDATION"
)

print(
    "============================================"
)


for key, value in val_metrics.items():

    print(
        key + ":",
        value
    )


print(
    "\nClassification report:"
)


print(

    classification_report(

        y_val,

        second_xgb_val_pred,

        labels=np.arange(
            NUM_CLASSES
        ),

        target_names=class_names,

        digits=4,

        zero_division=0
    )
)


# ============================================================
# TEST METRICS
# ============================================================

test_metrics = evaluate_second_xgb(

    y_test,

    second_xgb_test_pred
)


print(
    "\n============================================"
)

print(
    "SECOND-STAGE XGBOOST — TEST"
)

print(
    "============================================"
)


for key, value in test_metrics.items():

    print(
        key + ":",
        value
    )


print(
    "\nClassification report:"
)


print(

    classification_report(

        y_test,

        second_xgb_test_pred,

        labels=np.arange(
            NUM_CLASSES
        ),

        target_names=class_names,

        digits=4,

        zero_division=0
    )
)


# ============================================================
# SAVE RESULTS
# ============================================================

results = pd.DataFrame([

    {
        "dataset":
            "train",

        **train_metrics
    },

    {
        "dataset":
            "validation",

        **val_metrics
    },

    {
        "dataset":
            "test",

        **test_metrics
    }

])


results.to_csv(

    SECOND_XGB_RESULTS_PATH,

    index=False
)


# ============================================================
# CLEANUP
# ============================================================

gc.collect()


# ============================================================
# FINAL STATUS
# ============================================================

print(
    "\n============================================"
)

print(
    "PART 4.3 COMPLETED"
)

print(
    "============================================"
)

print(
    "Model:",
    SECOND_XGB_MODEL_PATH
)

print(
    "Validation probability:",
    SECOND_XGB_VAL_PROB_PATH
)

print(
    "Test probability:",
    SECOND_XGB_TEST_PROB_PATH
)

print(
    "Results:",
    SECOND_XGB_RESULTS_PATH
)

print(
    "\nSecond-stage XGBoost fusion completed."
)